# 📓 دفتر تقييم النموذج — نسخة مُحسّنة (v4.1)

إضافات هذه النسخة (v4.1) فوق النسخة السابقة (v4):

1. **إدارة المخرجات** (`save_or_print`): أي جدول/نص يتجاوز حجماً معيّناً يُحفظ تلقائياً في مجلد `analysis_outputs/` بدل طباعته بالكامل، مع عرض معاينة مختصرة فقط — لتقليل التشتت في النتائج.
2. **رسوم بيانية** لكل تقرير تقريباً: منحنى المعايرة (Calibration)، الثقة×الخطأ، توزيع الأخطاء، Actual-vs-Predicted، فحص الإزاحة الزمنية (Lag Scan)، خريطة مصفوفة الارتباك، أهمية الخصائص، العناقيد السلوكية (PCA)، منحنى رأس المال والـ Drawdown، وأداء الدفعات — كلها تُعرض inline **وتُحفظ تلقائياً** كملفات PNG في `analysis_outputs/figures/`.
3. **تشخيص نزاهة النموذج** (`run_integrity_diagnostics`): مُدمَج من سكربتي التشخيص المرفقين — يقارن النموذج بـ Naive Baseline (هل يتفوق فعلاً أم يعيد إنتاج آخر سعر؟)، يفحص الإزاحة الزمنية (Lag Scan) لكشف "النسخ المتأخر"، ويحسب الانحياز الموجّه (Directional Bias).
4. **مقاييس أداء تداول إضافية** (Tearsheet-style): Sharpe, Sortino, Omega, Profit Factor, SQN, Max Drawdown + منحنى رأس المال — مبنية على ممارسات تقارير الـ backtest المالية القياسية.
5. **`run_full_analysis` محدَّثة** لتشغّل كل ما سبق تلقائياً وتحفظ كل شيء كبير في ملف واحد منظّم.

> ✅ **التوافق الخلفي محفوظ بالكامل** مع النسخة السابقة وأسماء الدوال القديمة.
### ✨ الجديد في v4.2

1. **`TargetSpec` الموحّد** (بحقول `reference / magnitude_weight / clip_percentile`) + **`resolve_targets`**: تُمرَّر الأهداف كنصوص (`'close'` أو `'high,low'` أو `['high','low']`) أو `TargetSpec` أو `None/'all'`. إن طلبتَ هدفاً أو هدفين والنموذج يُخرج ثلاثة، تُتجاهل البقية بالكامل.
2. **`test_all_assets_v4` بعرض مضغوط**: ملخص التحقق من فك التشفير **مرة واحدة** بدل تكراره لكل أصل، وجدول واحد لكل أصل، و`n_display` يعرض **آخر** العينات (الأحدث في الأسفل)، مع جدول ملخص وإجمالي في النهاية.
3. **تقرير آخر العينات للتداول الحي** (`predict_latest_v4` / `predict_latest_all_assets`): التاريخ · الإشارة · الثقة · عدم اليقين % · التحرك المتوقع % · السعر المتوقع — ولا يشترط وجود هدف لآخر عينة (تظهر ⏳ ولا تدخل في أي مقياس).
4. **إصلاح** `TypeError: unhashable type: 'numpy.ndarray'` في `run_integrity_diagnostics` (كان سببه `drop_duplicates()` على إطار يحوي أعمدة مصفوفات).


## 1️⃣ الإعداد الأساسي: TargetSpec (وصف الأهداف المستمرة/الفئوية)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔧 الإصدار v4.2: TargetSpec الموحّد + resolve_targets (أهداف نصية / جزئية بمرونة)
# ═══════════════════════════════════════════════════════════════════════════════
#
# الجديد في هذا الإصدار:
# 1. ✅ TargetSpec بحقول النسخة الجديدة (reference / magnitude_weight / clip_percentile)
#    مع مزامنة entry_col <-> price_index تلقائياً.
# 2. ✅ resolve_targets: تقبل الأهداف كنصوص ('close' أو 'high,low' أو ['high','low'])
#    أو TargetSpec أو dict أو None/'all' (= كل ما يُخرجه النموذج).
# 3. ✅ إن طلبتَ هدفاً أو هدفين والنموذج يُخرج ثلاثة: يُتجاهل الباقي تماماً
#    (لا يُقرأ ولا يُفكّ ولا يُقيَّم ولا يظهر في أي تقرير).
# 4. ✅ توافق كامل مع الدوال القديمة.
# ═══════════════════════════════════════════════════════════════════════════════

import re
import warnings
import numpy as np
import tensorflow as tf
import pandas as pd
from dataclasses import dataclass, field, replace
from typing import Tuple, Dict, Optional, List, Union, Literal, Iterable


# ─────────────────────────────────────────────────────────────────────────────
# 📋 وصف الأهداف (Target Specification)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class TargetSpec:
    """
    وصف موحّد لأي هدف تنبؤ (مستمر أو فئوي)، يجمع بين واجهة النسخة الأصلية
    (kind / class_names / price_index / has_uncertainty) وحقول النسخة الجديدة
    (reference / magnitude_weight / clip_percentile).

    ملاحظة: entry_col و price_index نفس المفهوم، ويتم مزامنتهما تلقائيًا.
    ملاحظة: reference / magnitude_weight / clip_percentile محفوظة هنا للتوافق مع
            كود التدريب؛ دوال التقييم في هذا الدفتر لا تستخدمها.
    """
    name: str
    kind: Literal['continuous', 'categorical'] = 'continuous'
    class_names: Optional[List[str]] = None
    has_uncertainty: bool = True

    # مرجع سعر الدخول (index داخل ['high','low','close'])
    price_index: Optional[int] = None
    entry_col: Optional[int] = None      # اسم بديل لـ price_index

    # القيمة الحقيقية المستقبلية
    future_col: Optional[int] = None

    # إعدادات حساب الاتجاه/المقدار
    reference: str = 'entry'             # 'entry' أو 'median'
    magnitude_weight: float = 0.1
    clip_percentile: float = 99.0

    # True: المخرج الخام عائد نسبة لسعر دخول *هذا الهدف نفسه* (last_candles[:, price_index])،
    # فيُفكّ بـ entry × (1 + raw) — مطابق لـ invert_reg_predictions في خط الأنابيب عند
    # reg_target_mode='return' (high نسبة لآخر high، low لآخر low، close لآخر close).
    # False (الافتراضي، السلوك القديم): (raw × iqr) + median من base_params المشتركة بين الأهداف —
    # base_params واحدة لكل الأهداف لا تستطيع تمثيل مرجع مختلف لكل هدف.
    relative_to_entry: bool = False

    def __post_init__(self):
        # 1) فحص الأهداف الفئوية
        if self.kind == 'categorical' and not self.class_names:
            raise ValueError(
                f"الهدف '{self.name}' فئوي لكنه بدون class_names."
            )

        # 2) مزامنة entry_col <-> price_index
        default_price_index = {'close': 2, 'low': 1, 'high': 0}
        if self.price_index is None and self.entry_col is not None:
            self.price_index = self.entry_col
        if self.entry_col is None and self.price_index is not None:
            self.entry_col = self.price_index
        if self.price_index is None:
            self.price_index = default_price_index.get(self.name)
            self.entry_col = self.price_index

        # 3) future_col الافتراضي حسب الاسم
        if self.future_col is None:
            self.future_col = {'close': 4, 'low': 5, 'high': 6}.get(self.name)

        # 4) فحص reference
        assert self.reference in ('entry', 'median'), \
            f"reference غير صالحة: {self.reference}"


# ─── إعداد افتراضي متوافق مع النسخة الأصلية ───
DEFAULT_PRICE_TARGETS: List[TargetSpec] = [
    TargetSpec(name='high',  kind='continuous', has_uncertainty=True, price_index=0, future_col=6),
    TargetSpec(name='low',   kind='continuous', has_uncertainty=True, price_index=1, future_col=5),
    TargetSpec(name='close', kind='continuous', has_uncertainty=True, price_index=2, future_col=4),
]


def make_categorical_spec(name: str, class_names: List[str], has_uncertainty: bool = False) -> TargetSpec:
    """اختصار لإنشاء TargetSpec لهدف فئوي (مثال: اتجاه السوق: صعود/هبوط/تذبذب)."""
    return TargetSpec(name=name, kind='categorical', class_names=class_names, has_uncertainty=has_uncertainty)


# ─────────────────────────────────────────────────────────────────────────────
# 🧰 أدوات صغيرة مشتركة (آمنة مع NaN وأعمدة last_candles الناقصة)
# ─────────────────────────────────────────────────────────────────────────────

def _safe_col(arr, idx):
    """عمود idx من مصفوفة ثنائية، أو None إن لم يوجد (مثال: last_candles في التداول الحي)."""
    if arr is None or idx is None:
        return None
    a = np.asarray(arr)
    if a.ndim < 2 or a.shape[1] <= idx:
        return None
    return a[:, idx]


def _spec_base(spec, median, iqr, last_candles, limit=None):
    """(offset, scale) لفك تشفير هدف مستمر: pred_real = raw × scale + offset.
    relative_to_entry → (entry, entry) أي entry × (1 + raw)؛ وإلا (median, iqr) من base_params."""
    if getattr(spec, 'relative_to_entry', False):
        col = _safe_col(last_candles[:limit] if (last_candles is not None and limit is not None)
                        else last_candles, spec.price_index)
        if col is None:
            raise ValueError(f"الهدف '{spec.name}' (relative_to_entry) يحتاج last_candles بعمود السعر {spec.price_index}.")
        entry = np.asarray(col, dtype=np.float64)
        return entry, entry
    return median, iqr


def _nanmean(x) -> float:
    x = np.asarray(x, dtype=np.float64)
    m = np.isfinite(x)
    return float(x[m].mean()) if m.any() else float('nan')


def _nanmax(x) -> float:
    x = np.asarray(x, dtype=np.float64)
    m = np.isfinite(x)
    return float(x[m].max()) if m.any() else float('nan')


# ─────────────────────────────────────────────────────────────────────────────
# 🎯 resolve_targets: تحويل أي صيغة أهداف إلى قائمة TargetSpec
# ─────────────────────────────────────────────────────────────────────────────

_UNCERTAINTY_SUFFIXES = ('_epistemic', '_aleatoric', '_confidence')
_ALL_KEYWORDS = {'all', '*', 'الكل', 'كل', 'كلها'}


def _target_name_key(name) -> str:
    """'Close' / ' y_close ' → 'close'."""
    n = str(name).strip().lower()
    return n[2:] if n.startswith('y_') else n


def _default_spec(name: str) -> Optional[TargetSpec]:
    for s in DEFAULT_PRICE_TARGETS:
        if s.name == name:
            return replace(s)          # نسخة مستقلة كي لا نُعدّل الافتراضي بالخطأ
    return None


def _flatten_targets(obj):
    """يُسطّح أي تركيبة (نص، قائمة، مجموعة، مصفوفة، TargetSpec، dict) إلى عناصر مفردة."""
    if obj is None:
        return
    if isinstance(obj, (TargetSpec, dict)):
        yield obj
    elif isinstance(obj, str):
        for part in re.split(r'[,\s;|+/،&]+', obj.strip()):
            if part:
                yield part
    elif isinstance(obj, (set, frozenset)):
        for o in sorted(obj, key=str):
            yield from _flatten_targets(o)
    elif isinstance(obj, (list, tuple)) or hasattr(obj, 'tolist'):
        seq = obj if isinstance(obj, (list, tuple)) else obj.tolist()
        for o in seq:
            yield from _flatten_targets(o)
    else:
        raise TypeError(f"نوع هدف غير مدعوم: {type(obj).__name__} ({obj!r})")


def _is_all_keyword(item) -> bool:
    return isinstance(item, str) and item.strip().lower() in _ALL_KEYWORDS


def resolve_targets(targets=None, available: Optional[Iterable[str]] = None) -> List[TargetSpec]:
    """
    يحوّل أي صيغة أهداف إلى List[TargetSpec] مرتّبة وبلا تكرار.

    الصيغ المقبولة (ويمكن خلطها):
        None أو 'all'         → كل الأهداف (ما يُخرجه النموذج إن مُرِّر available، وإلا high/low/close)
        'close'               → هدف واحد
        'high,low' / 'high low' / ['high', 'low']
        TargetSpec(...) أو dict(name='trend', kind='categorical', class_names=[...])

    Args:
        available: أسماء الأهداف التي يُخرجها النموذج فعلاً (اختياري). إن مُرِّرت:
                   • يُتحقق أن كل هدف مطلوب موجود فيها (وإلا خطأ واضح).
                   • الأهداف غير المطلوبة تُتجاهل تلقائياً.
    """
    avail = [_target_name_key(a) for a in available] if available is not None else None

    items = list(_flatten_targets(targets))
    wants_all = (not items) or any(_is_all_keyword(i) for i in items)
    items = [i for i in items if not _is_all_keyword(i)]
    if wants_all:
        items += list(avail) if avail else [s.name for s in DEFAULT_PRICE_TARGETS]

    specs: List[TargetSpec] = []
    seen = set()
    for it in items:
        if isinstance(it, TargetSpec):
            spec = it
        elif isinstance(it, dict):
            spec = TargetSpec(**it)
        else:
            key = _target_name_key(it)
            spec = _default_spec(key)
            if spec is None:
                if avail is not None and key in avail:
                    warnings.warn(
                        f"الهدف '{key}' خارج (high/low/close): سيُعامَل كهدف مستمر بلا سعر دخول مرجعي. "
                        f"مرّر TargetSpec(name='{key}', price_index=..., future_col=...) لتحديدهما.",
                        stacklevel=3,
                    )
                    spec = TargetSpec(name=key, kind='continuous', has_uncertainty=True)
                else:
                    known = ', '.join(s.name for s in DEFAULT_PRICE_TARGETS)
                    raise ValueError(
                        f"هدف غير معروف: '{it}'. المعروف: {known}. "
                        f"للأهداف الأخرى مرّر TargetSpec(...) أو dict بالحقل name."
                    )
        if spec.name in seen:
            continue
        seen.add(spec.name)
        specs.append(spec)

    if not specs:
        raise ValueError("لم يُحدَّد أي هدف صالح.")

    if avail is not None:
        missing = [s.name for s in specs if s.name not in avail]
        if missing:
            raise ValueError(f"الأهداف {missing} غير موجودة في مخرجات النموذج. المتاح: {avail}")
    return specs


def get_model_target_names(model, X_inputs=None) -> Optional[List[str]]:
    """أسماء الأهداف التي يُخرجها النموذج فعلاً (مفاتيح 'y_<name>' بدون epistemic/aleatoric/confidence)."""
    keys = None
    names = getattr(model, 'output_names', None)
    if names and any(str(n).startswith('y_') for n in names):
        keys = list(names)
    elif X_inputs is not None:
        try:
            probe = tuple(np.asarray(x)[:1] for x in X_inputs)
            out = model(probe, training=False)
            if isinstance(out, dict):
                keys = list(out.keys())
        except Exception:
            keys = None
    if not keys:
        return None
    found = []
    for k in map(str, keys):
        if k.startswith('y_') and not k.endswith(_UNCERTAINTY_SUFFIXES):
            found.append(k[2:])
    return found or None


def _resolve_for_model(target_specs, model=None, X_inputs=None) -> List[TargetSpec]:
    """resolve_targets + استكشاف مخرجات النموذج فقط عند الحاجة (None/'all'/اسم غير معروف)."""
    items = list(_flatten_targets(target_specs))
    needs_probe = (not items) or any(_is_all_keyword(i) for i in items) or any(
        isinstance(i, str) and _default_spec(_target_name_key(i)) is None for i in items
    )
    avail = get_model_target_names(model, X_inputs) if (needs_probe and model is not None) else None
    return resolve_targets(target_specs, available=avail)


## 2️⃣ 💾 إدارة المخرجات: `save_or_print` و `save_figure`

ضع هذه الخلية مبكراً — تعتمد عليها كل التقارير والرسوم اللاحقة لتفادي إغراق الـ notebook بمخرجات ضخمة (كل شيء كبير يُحفظ كاملاً في `analysis_outputs/` مع معاينة مختصرة هنا).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 💾 إدارة المخرجات: طباعة مختصرة + حفظ كامل في ملف عند كِبر الحجم
# ═══════════════════════════════════════════════════════════════════════════
#
# المشكلة: بعض التقارير (مثل تحليل كل عملة × كل دفعة × كل عيّنة شاذة) قد
# تنتج مئات/آلاف الأسطر، مما "يُغرق" الـ notebook ويصعّب القراءة. الحل هنا:
# أي DataFrame/نص يتجاوز حداً معيّناً يُحفظ كاملاً في ملف (CSV أو TXT) داخل
# مجلد analysis_outputs/، وتُطبع فقط معاينة مختصرة + مسار الملف.
# ═══════════════════════════════════════════════════════════════════════════

import os
from datetime import datetime

DEFAULT_OUTPUT_DIR = "analysis_outputs"
DEFAULT_MAX_ROWS = 40
DEFAULT_MAX_CHARS = 4000


def ensure_output_dir(out_dir: str = DEFAULT_OUTPUT_DIR) -> str:
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


def save_or_print(
    data: Union[pd.DataFrame, str],
    name: str,
    out_dir: str = DEFAULT_OUTPUT_DIR,
    max_rows: int = DEFAULT_MAX_ROWS,
    max_chars: int = DEFAULT_MAX_CHARS,
    verbose: bool = True,
    timestamp: bool = False,
) -> Optional[str]:
    """
    يطبع البيانات مباشرة إن كانت صغيرة، أو يحفظها كاملة في ملف ويطبع معاينة
    مختصرة + مسار الملف إن كانت كبيرة. يقلّل التشتت في مخرجات الـ notebook
    دون فقدان أي تفصيل (كل شيء محفوظ في analysis_outputs/).

    Args:
        data: DataFrame أو نص طويل
        name: اسم وصفي يُستخدم كاسم ملف (بدون امتداد)
        out_dir: المجلد الذي تُحفظ فيه الملفات الكبيرة
        max_rows: أقصى عدد صفوف يُطبع مباشرة (لـ DataFrame)
        max_chars: أقصى عدد حروف يُطبع مباشرة (لنص)
        timestamp: أضف طابعاً زمنياً لاسم الملف (لتفادي الكتابة فوق ملف سابق)

    Returns:
        مسار الملف المحفوظ، أو None إن طُبعت البيانات مباشرة بدون حفظ
    """
    ensure_output_dir(out_dir)
    suffix = f"_{datetime.now().strftime('%Y%m%d_%H%M%S')}" if timestamp else ""

    if isinstance(data, pd.DataFrame):
        if len(data) > max_rows:
            path = os.path.join(out_dir, f"{name}{suffix}.csv")
            data.to_csv(path, index=False, encoding="utf-8-sig")
            if verbose:
                print(f"📄 النتيجة كبيرة ({len(data):,} صف) → حُفظت كاملة في: {path}")
                print(f"   معاينة أول {min(max_rows, len(data))} صف:")
                print(data.head(max_rows).to_string(index=False))
                if len(data) > max_rows:
                    print(f"   ... ({len(data) - max_rows:,} صف إضافي في الملف)")
            return path
        else:
            if verbose:
                print(data.to_string(index=False))
            return None

    elif isinstance(data, str):
        if len(data) > max_chars:
            path = os.path.join(out_dir, f"{name}{suffix}.txt")
            with open(path, "w", encoding="utf-8") as f:
                f.write(data)
            if verbose:
                print(f"📄 النص طويل ({len(data):,} حرف) → حُفظ كاملاً في: {path}")
                print(data[:max_chars])
                print(f"   ... [تم الاقتصاص هنا — راجع الملف للنص الكامل]")
            return path
        else:
            if verbose:
                print(data)
            return None

    else:
        raise TypeError("save_or_print يدعم فقط pd.DataFrame أو str")


def save_figure(fig, name: str, out_dir: str = DEFAULT_OUTPUT_DIR, dpi: int = 130) -> str:
    """يحفظ أي matplotlib figure في analysis_outputs/figures/ ويُرجع المسار."""
    fig_dir = ensure_output_dir(os.path.join(out_dir, "figures"))
    path = os.path.join(fig_dir, f"{name}.png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path


## 3️⃣ المرحلة 1: التنبؤ على الدفعات (مرن)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# المرحلة 1: التنبؤ على الدفعات (مرن لأي عدد/نوع أهداف)
# ─────────────────────────────────────────────────────────────────────────────

def predict_batch_v4(
    model,
    X_inputs: Tuple[np.ndarray, ...],
    target_specs: List['TargetSpec'],
    batch_size: int = 256,
    verbose: bool = False,
) -> Dict[str, np.ndarray]:
    """
    تنبؤ على الدفعات لأي مجموعة أهداف (مستمرة و/أو فئوية).

    يتوقع أن يُخرج النموذج مفاتيح بصيغة:
        'y_{name}'                  → القيمة (مستمرة) أو logits/probabilities (فئوية)
        'y_{name}_epistemic'        → (اختياري) عدم يقين ابستيمي
        'y_{name}_aleatoric'        → (اختياري) عدم يقين أليتوري
        'y_{name}_confidence'       → (اختياري) درجة ثقة

    Returns:
        dict خام يحوي لكل هدف: raw (+ epistemic/aleatoric/confidence إن وُجدت)
    """
    target_specs = resolve_targets(target_specs)   # نصوص/جزئية مقبولة؛ غير المطلوب يُتجاهل
    n_samples = X_inputs[0].shape[0]
    ds = tf.data.Dataset.from_tensor_slices((X_inputs,)).batch(batch_size)

    keys = []
    for spec in target_specs:
        keys.append(spec.name)
        if spec.has_uncertainty:
            keys += [f'{spec.name}_epistemic', f'{spec.name}_aleatoric', f'{spec.name}_confidence']

    preds = {k: [] for k in keys}
    prog = tf.keras.utils.Progbar(n_samples // batch_size + 1) if verbose else None

    for i, (b_x,) in enumerate(ds):
        out = model(b_x, training=False)

        for spec in target_specs:
            key = f'y_{spec.name}'
            if key not in out:
                _avail = sorted(str(k)[2:] for k in out
                                if str(k).startswith('y_') and not str(k).endswith(_UNCERTAINTY_SUFFIXES))
                raise KeyError(
                    f"مخرجات النموذج لا تحوي '{key}'. الأهداف المتاحة في النموذج: {_avail}. "
                    f"تأكد من أن اسم الهدف في TargetSpec يطابق مخرجات النموذج."
                )
            preds[spec.name].append(np.asarray(out[key]))

            if spec.has_uncertainty:
                for suffix in ['epistemic', 'aleatoric', 'confidence']:
                    k2 = f'y_{spec.name}_{suffix}'
                    if k2 in out:
                        preds[f'{spec.name}_{suffix}'].append(np.asarray(out[k2]))
                    else:
                        # نملأ بأصفار حتى لا ينهار concatenate لاحقاً
                        n_b = b_x[0].shape[0] if isinstance(b_x, (tuple, list)) else b_x.shape[0]
                        preds[f'{spec.name}_{suffix}'].append(np.zeros((n_b, 1), dtype=np.float32))

        if prog:
            prog.update(i + 1)

    return {k: np.concatenate(v, axis=0) for k, v in preds.items()}


# دالة قديمة تبقى للتوافق الخلفي (high/low/close مستمرة فقط)
def predict_batch(model, X_inputs, batch_size: int = 256, verbose: bool = False) -> Dict[str, np.ndarray]:
    """(نسخة سابقة - محفوظة للتوافق) تنبؤ للأهداف الثلاثة high/low/close فقط."""
    return predict_batch_v4(model, X_inputs, DEFAULT_PRICE_TARGETS, batch_size, verbose)


## 4️⃣ المرحلة 2: فك التشفير المرن (مستمر + فئوي)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# المرحلة 2: فك التشفير المرن (مستمر + فئوي)
# ─────────────────────────────────────────────────────────────────────────────

def _softmax_stable(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def decode_predictions_v4(
    raw_preds: Dict[str, np.ndarray],
    target_specs: List['TargetSpec'],
    base_params: Optional[np.ndarray] = None,
    last_candles: Optional[np.ndarray] = None,
    limit: Optional[int] = None,
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    فك تشفير مرن لكل هدف حسب نوعه:

    • مستمر (continuous):
        pred_scaled  → القيمة الخام كما خرجت من النموذج (فضاء مُطبَّع)
        pred_real    → القيمة بعد فك التطبيع: (scaled * iqr) + median

    • فئوي (categorical):
        pred_encoded → القيمة الخام: إما logits/probabilities [N, C] أو مؤشر فئة [N]
        pred_index   → مؤشر الفئة المتوقعة (argmax)
        pred_label   → اسم الفئة الأصلي (str) بعد فك الترميز عبر class_names
        pred_proba   → احتمال أعلى فئة (إن كانت المخرجات probabilities)

    Args:
        raw_preds: مخرجات predict_batch_v4
        target_specs: قائمة TargetSpec تصف كل هدف
        base_params: [N, 2] = (median, iqr) — مطلوب فقط للأهداف المستمرة
        last_candles: مصفوفة الشمعة الحالية/المستقبلية (اختياري، لأهداف الأسعار)
        limit: تحديد عدد العينات المُعالجة

    Returns:
        dict[target_name] -> dict بالحقول أعلاه (+ عدم اليقين إن وُجد)
    """
    target_specs = resolve_targets(target_specs)
    any_target = target_specs[0].name
    if limit is None:
        limit = raw_preds[any_target].shape[0]

    if base_params is not None:
        base_params = base_params[:limit]
        median = base_params[:, 0]
        iqr = base_params[:, 1]
    else:
        median = iqr = None

    if last_candles is not None:
        last_candles = last_candles[:limit]

    results: Dict[str, Dict[str, np.ndarray]] = {}

    for spec in target_specs:
        raw = raw_preds[spec.name][:limit]
        entry = {}
        s_median, s_iqr = (_spec_base(spec, median, iqr, last_candles)
                           if spec.kind == 'continuous' else (median, iqr))

        # ── عدم اليقين (مشترك بين النوعين إن وُجد) ──────────────────────────
        if spec.has_uncertainty and f'{spec.name}_epistemic' in raw_preds:
            epi = raw_preds[f'{spec.name}_epistemic'][:limit].flatten()
            ale = raw_preds[f'{spec.name}_aleatoric'][:limit].flatten()
            entry['epistemic'] = epi if s_iqr is None else epi * s_iqr
            entry['aleatoric'] = ale if s_iqr is None else ale * s_iqr
            total_scaled = np.sqrt(epi ** 2 + ale ** 2)
            entry['uncertainty_real'] = total_scaled if s_iqr is None else total_scaled * s_iqr
            entry['uncertainty_scaled'] = total_scaled
            entry['confidence'] = raw_preds[f'{spec.name}_confidence'][:limit].flatten()

        # ── فك التشفير حسب النوع ────────────────────────────────────────────
        if spec.kind == 'continuous':
            if s_median is None or s_iqr is None:
                raise ValueError(f"الهدف المستمر '{spec.name}' يحتاج base_params (median, iqr) لفك التشفير.")
            raw_flat = raw.flatten()
            pred_real = (raw_flat * s_iqr) + s_median

            entry_price = np.zeros_like(pred_real)
            target_real = np.full_like(pred_real, np.nan)
            # أعمدة last_candles قد تكون ناقصة في التداول الحي (بلا أعمدة المستقبل) — نتعامل معها بأمان
            _entry_col = _safe_col(last_candles, spec.price_index)
            if _entry_col is not None:
                entry_price = np.asarray(_entry_col, dtype=np.float64)
            _future_col = _safe_col(last_candles, spec.future_col)
            if _future_col is not None:
                target_real = np.asarray(_future_col, dtype=np.float64)

            change_from_entry = ((pred_real - entry_price) / (np.abs(entry_price) + 1e-7)) * 100
            # طابع زمني لكل عيّنة (عمود 3 في تخطيط last_candles لخط الأنابيب) — لتجميع الصفقات
            # المتزامنة في محفظة واحدة لكل فترة بدل سلسلة صفقات متتالية وهمية.
            _ts_col = _safe_col(last_candles, 3) if (last_candles is not None and np.ndim(last_candles) == 2
                                                     and last_candles.shape[1] >= 7) else None

            entry.update({
                'kind': 'continuous',
                'pred_scaled': raw_flat,
                'pred_real': pred_real,
                'entry_price': entry_price,
                'change_from_entry': change_from_entry,
                'target_real': target_real,
                'median': s_median,
                'iqr': s_iqr,
                'timestamp': (np.asarray(_ts_col, dtype=np.float64) if _ts_col is not None
                              else np.full_like(pred_real, np.nan)),
            })

        elif spec.kind == 'categorical':
            n_classes = len(spec.class_names)

            if raw.ndim == 1 or (raw.ndim == 2 and raw.shape[1] == 1):
                # النموذج أخرج مؤشر الفئة مباشرة (وليس احتمالات)
                pred_index = raw.flatten().astype(int)
                pred_proba = np.full_like(pred_index, np.nan, dtype=np.float32)
                raw_encoded = pred_index
            else:
                if raw.shape[1] != n_classes:
                    raise ValueError(
                        f"عدد أعمدة مخرجات '{spec.name}' ({raw.shape[1]}) لا يطابق "
                        f"عدد الفئات المُعرَّفة ({n_classes})."
                    )
                # نتحقق إن كانت logits أو probabilities (مجموع الصف ≈ 1)
                row_sums = raw.sum(axis=1)
                looks_like_proba = np.allclose(row_sums, 1.0, atol=1e-3)
                proba = raw if looks_like_proba else _softmax_stable(raw, axis=1)

                pred_index = np.argmax(proba, axis=1)
                pred_proba = proba[np.arange(len(proba)), pred_index]
                raw_encoded = raw

            pred_label = np.array([spec.class_names[i] for i in pred_index], dtype=object)

            entry.update({
                'kind': 'categorical',
                'pred_encoded': raw_encoded,
                'pred_index': pred_index,
                'pred_label': pred_label,
                'pred_proba': pred_proba,
                'class_names': spec.class_names,
            })
        else:
            raise ValueError(f"نوع هدف غير مدعوم: {spec.kind}")

        results[spec.name] = entry

    return results


# دالة قديمة تبقى للتوافق الخلفي
def decode_predictions(raw_preds, base_params, last_candles=None, limit=None):
    """(نسخة سابقة - محفوظة للتوافق) فك تشفير high/low/close المستمرة فقط."""
    decoded = decode_predictions_v4(raw_preds, DEFAULT_PRICE_TARGETS, base_params, last_candles, limit)
    # إعادة تسمية الحقول لتطابق الواجهة القديمة تماماً
    for t, d in decoded.items():
        d['pred_scaled'] = d.pop('pred_scaled')
    return decoded


## 5️⃣ المرحلة 2.5: التحقق من صحة فك التشفير

تقارن `verify_decoding` النتائج في مسارين مستقلين: **الفضاء المطبّع** (قبل أي تحويل) و**الفضاء الأصلي** (بعد فك التطبيع)، مع اختبار Round-trip يكتشف أي خطأ حسابي في معادلة فك التشفير نفسها.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# المرحلة 2.5: التحقق من صحة فك التشفير (مطبّع مقابل أصلي)
# ─────────────────────────────────────────────────────────────────────────────

def verify_decoding(
    raw_preds: Dict[str, np.ndarray],
    decoded: Dict[str, Dict[str, np.ndarray]],
    target_specs: List['TargetSpec'],
    base_params: Optional[np.ndarray] = None,
    y_true: Optional[Dict[str, np.ndarray]] = None,
    last_candles: Optional[np.ndarray] = None,
    limit: Optional[int] = None,
    atol: float = 1e-4,
    verbose: bool = True,
) -> Dict[str, Dict]:
    """
    ✅ يتحقق من أن فك التشفير صحيح عبر مقارنتين مستقلتين لكل هدف:

    1) فضاء القيم المطبّعة (Normalized / Scaled space):
       - للمستمر: مقارنة raw_pred (scaled) مع y_true scaled مباشرة (قبل فك التطبيع)
       - للفئوي:  مقارنة pred_index المُستخرج مع argmax(y_true) في نفس الفضاء المُرمّز

    2) فضاء القيم الأصلية (Real / Denormalized space):
       - للمستمر: مقارنة pred_real المُعاد حسابه يدوياً ((scaled*iqr)+median) مع
         القيمة المخزّنة فعلاً في decoded[target]['pred_real'] (اختبار Round-trip)،
         ثم مقارنته بالقيمة الحقيقية الفعلية (target_real من last_candles أو
         y_true بعد فك تطبيعه) لحساب اتساق الخطأ بين الفضاءين:
             نسبة الاتساق = |خطأ الفضاء الحقيقي| / (|خطأ الفضاء المطبّع| * iqr)
         يجب أن تكون قريبة من 1.0 إن كان فك التشفير صحيحاً حسابياً.
       - للفئوي: مقارنة pred_label مع اسم الفئة الحقيقي (إن أمكن اشتقاقه من y_true)

    Returns:
        dict[target_name] -> تقرير تحقق (مطابقة round-trip، اتساق الفضاءين، أخطاء)
    """
    target_specs = resolve_targets(target_specs)
    report: Dict[str, Dict] = {}

    if limit is None:
        any_name = target_specs[0].name
        limit = raw_preds[any_name].shape[0]

    if base_params is not None:
        bp = base_params[:limit]
        median, iqr = bp[:, 0], bp[:, 1]
    else:
        median = iqr = None

    if verbose:
        print("=" * 100)
        print("🔍 التحقق من صحة فك التشفير (Decoding Verification)")
        print("=" * 100)

    for spec in target_specs:
        d = decoded[spec.name]
        entry_report = {'target': spec.name, 'kind': spec.kind}

        if spec.kind == 'continuous':
            s_median, s_iqr = _spec_base(spec, median, iqr, last_candles, limit)
            if s_median is None:
                entry_report['status'] = 'skipped'
                entry_report['reason'] = 'لا يوجد base_params للتحقق'
            else:
                median_s, iqr_s = s_median, s_iqr
                # 1) round-trip: إعادة حساب pred_real من pred_scaled يدوياً
                recomputed_real = (d['pred_scaled'] * iqr_s) + median_s
                roundtrip_diff = np.abs(recomputed_real - d['pred_real'])
                roundtrip_ok = bool(np.all(roundtrip_diff <= atol))

                entry_report['roundtrip_max_diff'] = float(roundtrip_diff.max())
                entry_report['roundtrip_ok'] = roundtrip_ok

                # 2) مقارنة الفضاء المطبّع مقابل الفضاء الأصلي (إن وُجدت الحقيقة)
                true_scaled = None
                if y_true is not None and f'y_{spec.name}' in y_true:
                    true_scaled = y_true[f'y_{spec.name}'][:limit].flatten()

                true_real = d.get('target_real')
                has_real_truth = true_real is not None and not np.all(np.isnan(true_real))

                if true_scaled is not None:
                    err_scaled = np.abs(d['pred_scaled'] - true_scaled)
                    entry_report['mae_scaled'] = _nanmean(err_scaled)

                    # فك تطبيع الحقيقة يدوياً للمقارنة مع target_real المخزّن (إن وُجد)
                    true_real_from_scaled = (true_scaled * iqr_s) + median_s
                    if has_real_truth:
                        consistency_diff = np.abs(true_real_from_scaled - true_real)
                        entry_report['true_value_consistency_max_diff'] = _nanmax(consistency_diff)
                        # ✅ يدخل الحكم الآن: فكّ الهدف الحقيقي بنفس صيغة فكّ التنبؤ يجب أن يعيد السعر
                        # المستقبلي الفعلي. اختلاف كبير = مرجع فكّ التشفير خاطئ لهذا الهدف (مثلاً high
                        # يُفكّ حول آخر close بينما الهدف عائد نسبة لآخر high) — كان يُطبع رقماً فقط.
                        rel = consistency_diff / (np.abs(true_real) + 1e-12)
                        entry_report['true_value_consistent'] = bool(np.nanmax(rel) <= 1e-3) if np.isfinite(rel).any() else True
                    else:
                        true_real = true_real_from_scaled
                        has_real_truth = True

                if has_real_truth:
                    err_real = np.abs(d['pred_real'] - true_real)
                    entry_report['mae_real'] = _nanmean(err_real)

                    if true_scaled is not None and iqr_s is not None:
                        # اتساق الخطأ بين الفضاءين: يجب أن تكون النسبة ≈ 1
                        denom = err_scaled * iqr_s
                        denom_safe = np.where(denom < 1e-9, np.nan, denom)
                        ratio = err_real / denom_safe
                        entry_report['space_consistency_ratio_mean'] = _nanmean(ratio)

                entry_report['status'] = ('ok' if roundtrip_ok and entry_report.get('true_value_consistent', True)
                                          else 'FAILED')

        elif spec.kind == 'categorical':
            entry_report['n_classes'] = len(spec.class_names)

            # round-trip: التأكد أن كل pred_index يقع ضمن مجال الفئات الصحيح
            valid_range = np.all((d['pred_index'] >= 0) & (d['pred_index'] < len(spec.class_names)))
            entry_report['roundtrip_ok'] = bool(valid_range)

            # التأكد من تطابق pred_label مع class_names[pred_index] (اتساق داخلي)
            expected_labels = np.array([spec.class_names[i] for i in d['pred_index']], dtype=object)
            label_match = bool(np.all(expected_labels == d['pred_label']))
            entry_report['label_mapping_ok'] = label_match

            # مقارنة الفضاء المُرمّز (index/one-hot) بالفضاء الأصلي (label) مقابل y_true
            if y_true is not None and f'y_{spec.name}' in y_true:
                true_raw = y_true[f'y_{spec.name}'][:limit]
                if true_raw.ndim > 1 and true_raw.shape[1] > 1:
                    true_index = np.argmax(true_raw, axis=1)
                else:
                    true_index = true_raw.flatten().astype(int)

                acc_encoded_space = float(np.mean(d['pred_index'] == true_index))
                true_label = np.array([spec.class_names[i] for i in true_index], dtype=object)
                acc_label_space = float(np.mean(d['pred_label'] == true_label))

                entry_report['accuracy_encoded_space'] = acc_encoded_space
                entry_report['accuracy_label_space'] = acc_label_space
                # يجب أن تتطابق الدقتان تماماً إن كان فك التشفير صحيحاً
                entry_report['spaces_agree'] = bool(np.isclose(acc_encoded_space, acc_label_space))

            entry_report['status'] = 'ok' if (valid_range and label_match) else 'FAILED'

        report[spec.name] = entry_report

        if verbose:
            icon = "✅" if entry_report['status'] == 'ok' else ("⚠️" if entry_report['status'] == 'skipped' else "❌")
            print(f"\n{icon} الهدف: {spec.name} ({spec.kind})")
            for k, v in entry_report.items():
                if k in ('target', 'kind', 'status'):
                    continue
                if isinstance(v, float):
                    print(f"     • {k}: {v:.6f}")
                else:
                    print(f"     • {k}: {v}")

    if verbose:
        n_failed = sum(1 for r in report.values() if r['status'] == 'FAILED')
        print("\n" + "-" * 100)
        if n_failed == 0:
            print("✅ جميع الأهداف اجتازت التحقق من فك التشفير بنجاح")
        else:
            print(f"❌ {n_failed} هدف/أهداف فشلت في التحقق — راجع التفاصيل أعلاه")

    return report


## 6️⃣ المرحلة 3: التقييم المرن (Metrics مستمرة + فئوية)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# المرحلة 3: التقييم المرن (مستمر + فئوي) — آمن مع العينات التي بلا هدف
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_predictions_v4(
    decoded_preds: Dict[str, Dict[str, np.ndarray]],
    target_specs: List['TargetSpec'],
    y_true: Optional[Dict[str, np.ndarray]] = None,
    base_params: Optional[np.ndarray] = None,
    direction_tolerance: float = 0.2,
    price_tolerance: Optional[float] = None,
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    تقييم مرن: للأهداف المستمرة يحافظ على منطق v3 (MAE/اتجاه/تسامح)،
    وللأهداف الفئوية يضيف accuracy/precision/recall/F1 لكل فئة.

    ✅ v4.2: العينات التي بلا هدف حقيقي (NaN — مثل آخر شمعة في التداول الحي) تُستثنى
       من كل المقاييس بدل أن تُحسب "خطأً"، ويُسجَّل قناع `has_truth` لكل هدف.
    """
    target_specs = resolve_targets(target_specs)
    if y_true is None:
        return decoded_preds

    limit = None
    for spec in target_specs:
        limit = decoded_preds[spec.name].get('pred_real', decoded_preds[spec.name].get('pred_index'))
        limit = len(limit)
        break

    if base_params is not None:
        bp = base_params[:limit]
        median, iqr = bp[:, 0], bp[:, 1]
    else:
        median = iqr = None

    # المدى الفعلي (لحساب within_tolerance) — يُحسب فقط إن وُجد high/low مستمرين
    actual_range = None
    if 'high' in decoded_preds and 'low' in decoded_preds:
        h, l = decoded_preds['high'].get('target_real'), decoded_preds['low'].get('target_real')
        if h is not None and l is not None and not (np.all(np.isnan(h)) or np.all(np.isnan(l))):
            actual_range = np.maximum(h - l, 1e-7)

    for spec in target_specs:
        r = decoded_preds[spec.name]
        key = f'y_{spec.name}'
        if key not in y_true:
            continue

        if spec.kind == 'continuous':
            true_val = r.get('target_real')
            if true_val is None or np.all(np.isnan(true_val)):
                true_scaled = y_true[key][:limit].flatten()
                if getattr(spec, 'relative_to_entry', False):
                    true_val = np.asarray(r['entry_price'], dtype=np.float64) * (1.0 + true_scaled)
                else:
                    true_val = (true_scaled * iqr) + median
            true_val = np.asarray(true_val, dtype=np.float64)

            entry = np.asarray(r['entry_price'], dtype=np.float64)
            pred_real = np.asarray(r['pred_real'], dtype=np.float64)
            change_from_entry = np.asarray(r['change_from_entry'], dtype=np.float64)

            # عينات بلا هدف (NaN) تُستثنى من كل المقاييس
            valid = np.isfinite(true_val) & np.isfinite(pred_real) & np.isfinite(entry)

            abs_error = np.abs(true_val - pred_real)                      # NaN حيث لا هدف
            pct_error = (abs_error / (np.abs(true_val) + 1e-7)) * 100
            true_change_pct = ((true_val - entry) / (np.abs(entry) + 1e-7)) * 100

            pred_direction = np.sign(change_from_entry)
            true_direction = np.sign(true_change_pct)
            direction_correct = ((pred_direction == true_direction) & valid).astype(int)

            tol = price_tolerance if price_tolerance is not None else (
                direction_tolerance * actual_range if actual_range is not None else direction_tolerance * np.abs(true_val)
            )
            within_tolerance = ((abs_error <= tol) & valid).astype(int)

            if valid.sum() > 2 and np.std(change_from_entry[valid]) > 1e-7 and np.std(true_change_pct[valid]) > 1e-7:
                corr = np.corrcoef(change_from_entry[valid], true_change_pct[valid])[0, 1]
                corr = 0.0 if np.isnan(corr) else corr
            else:
                corr = 0.0

            win_rate = float(np.mean(direction_correct[valid]) * 100) if valid.any() else float('nan')
            uncertainty_pct = None
            within_1_std = within_2_std = None
            if 'uncertainty_real' in r:
                unc = np.asarray(r['uncertainty_real'], dtype=np.float64)
                uncertainty_pct = (unc / (np.abs(pred_real) + 1e-7)) * 100
                if valid.any():
                    within_1_std = float(np.mean(abs_error[valid] <= unc[valid]) * 100)
                    within_2_std = float(np.mean(abs_error[valid] <= 2 * unc[valid]) * 100)
                else:
                    within_1_std = within_2_std = float('nan')

            r.update({
                'true_real': true_val,
                'has_truth': valid,
                'n_valid': int(valid.sum()),
                'true_change_from_entry_pct': true_change_pct,
                'error_abs': abs_error,
                'error_pct': pct_error,
                'same_direction': direction_correct,
                'within_tolerance': within_tolerance,
                'direction_correct': direction_correct,
                'correlation': corr,
                'win_rate': win_rate,
                'uncertainty_pct': uncertainty_pct,
                'within_1_std': within_1_std,
                'within_2_std': within_2_std,
            })

        elif spec.kind == 'categorical':
            true_raw = y_true[key][:limit]
            if true_raw.ndim > 1 and true_raw.shape[1] > 1:
                true_index = np.argmax(true_raw, axis=1)
            else:
                true_index = true_raw.flatten().astype(int)
            true_label = np.array([spec.class_names[i] for i in true_index], dtype=object)

            correct = (r['pred_index'] == true_index).astype(int)
            accuracy = float(np.mean(correct)) * 100

            try:
                from sklearn.metrics import classification_report, confusion_matrix
                report_dict = classification_report(
                    true_index, r['pred_index'],
                    labels=list(range(len(spec.class_names))),
                    target_names=spec.class_names,
                    output_dict=True, zero_division=0,
                )
                cm = confusion_matrix(true_index, r['pred_index'], labels=list(range(len(spec.class_names))))
            except Exception:
                report_dict, cm = None, None

            r.update({
                'true_index': true_index,
                'true_label': true_label,
                'correct': correct,
                'accuracy': accuracy,
                'classification_report': report_dict,
                'confusion_matrix': cm,
            })

    return decoded_preds


# دالة قديمة تبقى للتوافق الخلفي
def evaluate_predictions(decoded_preds, y_true=None, base_params=None, last_candles=None,
                          direction_tolerance=0.2, price_tolerance=None):
    """(نسخة سابقة - محفوظة للتوافق) تقييم high/low/close المستمرة فقط."""
    if y_true is None or base_params is None:
        return decoded_preds
    return evaluate_predictions_v4(
        decoded_preds, DEFAULT_PRICE_TARGETS, y_true, base_params,
        direction_tolerance, price_tolerance,
    )


## 6️⃣.٥ جدول آخر العينات (التاريخ · التوقع · الثقة · عدم اليقين) — يعمل بلا أهداف حقيقية

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# المرحلة 3.5: جدول "آخر العينات" (تقييم/تداول حي) — التاريخ + التوقع + الثقة + عدم اليقين
# ─────────────────────────────────────────────────────────────────────────────
#
# يعمل بلا أهداف حقيقية إطلاقاً: عمود الهدف/النتيجة يظهر فقط إن وُجدت قيمة فعلية
# (وإلا يظهر ⏳ "بلا هدف بعد" — مثل آخر عينة في التداول الحي).
# ─────────────────────────────────────────────────────────────────────────────

_TIME_KEYS = ('timestamps', 'timestamp', 'times', 'time', 'dates', 'date',
              'datetime', 'open_time', 'candle_time')


def _to_datetime_index(values) -> Optional[pd.DatetimeIndex]:
    """
    يحوّل قيم الوقت إلى DatetimeIndex: datetime / نص / epoch (ثوانٍ، مللي، مايكرو، نانو
    — يُكتشف تلقائياً من الحجم). يُرجع None إن تعذّر التحويل أو لم تبدُ القيم كأوقات.
    """
    if values is None:
        return None
    try:
        arr = np.asarray(values)
        if arr.ndim > 1:
            arr = arr.reshape(-1)
        if arr.size == 0:
            return None
        if np.issubdtype(arr.dtype, np.datetime64):
            return pd.DatetimeIndex(arr)
        if arr.dtype.kind in 'iuf':
            f = arr.astype(np.float64)
            finite = f[np.isfinite(f)]
            if finite.size == 0:
                return None
            m = abs(float(np.median(finite)))
            unit = 'ns' if m >= 1e17 else 'us' if m >= 1e14 else 'ms' if m >= 1e11 else 's' if m >= 1e8 else None
            if unit is None:
                return None                    # أرقام صغيرة (فهارس) وليست طوابع زمنية
            return pd.DatetimeIndex(pd.to_datetime(f, unit=unit, errors='coerce'))
        idx = pd.DatetimeIndex(pd.to_datetime(arr, errors='coerce'))
        return idx if idx.notna().any() else None
    except Exception:
        return None


def _extract_timestamps(timestamps=None, last_candles=None, timestamp_col=None) -> Optional[pd.DatetimeIndex]:
    """الأولوية: timestamps الصريحة ثم عمود timestamp_col من last_candles."""
    if timestamps is not None:
        return _to_datetime_index(timestamps)
    if last_candles is not None and timestamp_col is not None:
        return _to_datetime_index(_safe_col(last_candles, timestamp_col))
    return None


def _find_timestamps(test_data: Dict, timestamp_key: Optional[str] = None,
                     timestamp_col: Optional[int] = None) -> Optional[pd.DatetimeIndex]:
    """يبحث عن الأوقات داخل بيانات أصل واحد (مفاتيح شائعة أو timestamp_col في last_candles)."""
    keys = [timestamp_key] if timestamp_key else list(_TIME_KEYS)
    for k in keys:
        if k and k in test_data and test_data[k] is not None:
            ts = _to_datetime_index(test_data[k])
            if ts is not None:
                return ts
    return _extract_timestamps(None, test_data.get('last_candles'), timestamp_col)


def _align_timestamps(ts: Optional[pd.DatetimeIndex], n_total: int, k: int) -> Optional[pd.DatetimeIndex]:
    """يُرجع أوقات آخر k عينة: ts بطول n_total الكامل أو بطول k فقط؛ غير ذلك None."""
    if ts is None:
        return None
    if len(ts) == n_total:
        return ts[n_total - k:]
    if len(ts) == k:
        return ts
    warnings.warn(f"طول الأوقات ({len(ts)}) لا يطابق عدد العينات ({n_total}) — سيُعرض رقم العينة بدل التاريخ.")
    return None


# ─── جدول آخر العينات ─────────────────────────────────────────────────────────

def build_latest_table(
    decoded: Dict[str, Dict[str, np.ndarray]],
    target_specs,
    n_display: int = 5,
    timestamps=None,
    asset: Optional[str] = None,
) -> pd.DataFrame:
    """
    جدول (عينة × هدف) لآخر `n_display` عينة من نتائج decode/evaluate.

    الأعمدة: idx, date, [asset], target, kind, signal, confidence_%, uncertainty_%,
             move_% (التحرك المتوقع عن سعر الدخول)، pred (السعر المتوقع)، pred_lo/pred_hi
             (pred ± عدم اليقين)، entry، true، true_label، has_true، ok.
    `ok`: True/False إن وُجد هدف فعلي، None إن لم يوجد بعد (تداول حي).
    `timestamps` يجب أن تطابق (بالترتيب) العينات المُمرَّرة في decoded.
    """
    specs = [s for s in resolve_targets(target_specs) if isinstance(decoded.get(s.name), dict)]
    if not specs or not n_display or n_display <= 0:
        return pd.DataFrame()

    def _n_of(r):
        return len(r['pred_real']) if 'pred_real' in r else len(r['pred_index'])

    n = min(_n_of(decoded[s.name]) for s in specs)
    k = int(min(n_display, n))

    ts = _to_datetime_index(timestamps) if timestamps is not None else None
    if ts is not None:
        ts = ts[len(ts) - n:] if len(ts) >= n else None

    rows = []
    for i in range(n - k, n):
        date = ts[i] if ts is not None else pd.NaT
        for spec in specs:
            r = decoded[spec.name]
            row = {'idx': i, 'date': date, 'target': spec.name, 'kind': spec.kind}
            if asset is not None:
                row['asset'] = asset

            if spec.kind == 'continuous':
                pred = float(r['pred_real'][i])
                entry = float(r['entry_price'][i]) if 'entry_price' in r else float('nan')
                entry_ok = np.isfinite(entry) and entry != 0.0
                move = (pred - entry) / (abs(entry) + 1e-7) * 100 if entry_ok else float('nan')

                unc = r.get('uncertainty_real')
                unc_i = float(unc[i]) if unc is not None else float('nan')
                conf = r.get('confidence')
                conf_i = float(conf[i]) * 100 if conf is not None else float('nan')

                true = r.get('true_real')
                if true is None:
                    true = r.get('target_real')
                true_i = float(true[i]) if true is not None and np.isfinite(true[i]) else float('nan')
                has_true = bool(np.isfinite(true_i))
                ok = bool(np.sign(pred - entry) == np.sign(true_i - entry)) if (has_true and entry_ok) else None

                row.update({
                    'signal': '—' if not np.isfinite(move) else ('▲' if move > 0 else '▼' if move < 0 else '■'),
                    'confidence_%': conf_i,
                    'uncertainty_%': unc_i / (abs(pred) + 1e-7) * 100 if np.isfinite(unc_i) else float('nan'),
                    'move_%': move,
                    'pred': pred, 'pred_lo': pred - unc_i, 'pred_hi': pred + unc_i,
                    'entry': entry if entry_ok else float('nan'),
                    'true': true_i, 'true_label': None, 'has_true': has_true, 'ok': ok,
                })

            else:  # categorical
                proba = float(r['pred_proba'][i]) if 'pred_proba' in r else float('nan')
                conf = r.get('confidence')
                conf_i = float(conf[i]) * 100 if conf is not None else proba * 100
                has_true = 'true_label' in r
                ok = bool(r['pred_index'][i] == r['true_index'][i]) if 'true_index' in r else None
                row.update({
                    'signal': str(r['pred_label'][i]),
                    'confidence_%': conf_i, 'uncertainty_%': float('nan'), 'move_%': float('nan'),
                    'pred': float('nan'), 'pred_lo': float('nan'), 'pred_hi': float('nan'),
                    'entry': float('nan'), 'true': float('nan'),
                    'true_label': str(r['true_label'][i]) if has_true else None,
                    'has_true': has_true, 'ok': ok,
                })
            rows.append(row)

    return pd.DataFrame(rows)


def _fmt_price(x) -> str:
    if x is None or not np.isfinite(x):
        return '—'
    a = abs(x)
    if a >= 1000:
        return f"{x:,.2f}"
    if a >= 1:
        return f"{x:.4f}"
    if a >= 0.01:
        return f"{x:.5f}"
    return f"{x:.7f}"


def _fmt_num(x, fmt: str) -> str:
    return '—' if (x is None or not np.isfinite(x)) else format(x, fmt)


def format_latest_table(df: pd.DataFrame) -> pd.DataFrame:
    """يحوّل جدول build_latest_table إلى نصوص مضغوطة جاهزة للطباعة."""
    if df is None or df.empty:
        return pd.DataFrame()
    out = pd.DataFrame(index=df.index)
    if df['date'].notna().any():
        out['date'] = df['date'].map(lambda d: d.strftime('%Y-%m-%d %H:%M') if pd.notna(d) else '—')
    else:
        out['#'] = df['idx']
    if 'asset' in df.columns:
        out['asset'] = df['asset']
    out['target'] = df['target']
    out['signal'] = df['signal']
    out['conf'] = df['confidence_%'].map(lambda v: _fmt_num(v, '.1f') + ('%' if np.isfinite(v) else ''))
    out['unc'] = df['uncertainty_%'].map(lambda v: _fmt_num(v, '.2f') + ('%' if np.isfinite(v) else ''))
    out['move'] = df['move_%'].map(lambda v: _fmt_num(v, '+.3f') + ('%' if np.isfinite(v) else ''))
    out['pred'] = [_fmt_price(p) if k == 'continuous' else '—' for p, k in zip(df['pred'], df['kind'])]
    if (df['kind'] == 'continuous').any():
        out['entry'] = df['entry'].map(_fmt_price)
    if df['has_true'].any():
        out['true'] = [
            tl if (k == 'categorical' and isinstance(tl, str)) else _fmt_price(t)
            for t, tl, k in zip(df['true'], df['true_label'], df['kind'])
        ]
    if df['ok'].notna().any() or df['has_true'].any():
        out['ok'] = df['ok'].map(
            lambda v: '⏳' if (v is None or (isinstance(v, float) and np.isnan(v))) else ('✅' if bool(v) else '❌')
        )
    return out


def _latest_legend(df: pd.DataFrame) -> str:
    txt = ("signal: ▲ صعود متوقع | ▼ هبوط  ·  conf: الثقة  ·  unc: عدم اليقين كنسبة من السعر المتوقع"
           "  ·  move: التحرك المتوقع عن سعر الدخول")
    if df is not None and not df.empty and (df['ok'].notna().any() or df['has_true'].any()):
        txt += "  ·  ⏳: لا يوجد هدف فعلي بعد"
    return txt


def print_latest_table(df: pd.DataFrame, title: Optional[str] = None, legend: bool = True, indent: str = "  "):
    if df is None or df.empty:
        print(indent + "(لا توجد عينات للعرض)")
        return
    if title:
        print(title)
    txt = format_latest_table(df).to_string(index=False)
    print("\n".join(indent + ln for ln in txt.split("\n")))
    if legend:
        print(indent + _latest_legend(df))


## 7️⃣ المرحلة 4: الدالة الموحّدة predict_with_evaluation_v4

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# المرحلة 4: الدالة الرئيسية المُوحّدة (v4) — تنبؤ + فك تشفير + تحقق + تقييم
# ─────────────────────────────────────────────────────────────────────────────

def predict_with_evaluation_v4(
    model,
    X_inputs: Tuple[np.ndarray, ...],
    target_specs,
    base_params: Optional[np.ndarray] = None,
    last_candles: Optional[np.ndarray] = None,
    y_true: Optional[Dict[str, np.ndarray]] = None,
    verbose: bool = False,
    n_display: int = 5,
    batch_size: int = 256,
    direction_tolerance: float = 0.2,
    price_tolerance: Optional[float] = None,
    min_samples: int = 1,
    run_verification: bool = True,
    timestamps=None,
    timestamp_col: Optional[int] = None,
    show_stages: Optional[bool] = None,
    show_verification: Optional[bool] = None,
    asset: Optional[str] = None,
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    التنبؤ + فك التشفير المرن + التحقق + التقييم — الإصدار v4.2

    المراحل: predict_batch_v4 → decode_predictions_v4 → verify_decoding
              → evaluate_predictions_v4

    target_specs: List[TargetSpec] أو نصوص ('close' / 'high,low' / ['high','low']) أو None/'all'.
                  يُقرأ ويُقيَّم المطلوب فقط، وباقي مخرجات النموذج تُتجاهل.
    n_display:    عدد **آخر** العينات المعروضة (الأحدث في الأسفل).
    timestamps / timestamp_col: تاريخ كل عينة (مصفوفة أوقات، أو رقم عمود داخل last_candles).
    show_stages / show_verification: افتراضياً = verbose (تُغلق من test_all_assets_v4
                  ليظهر ملخص التحقق مرة واحدة فقط).
    """
    n_samples = X_inputs[0].shape[0]
    if n_samples < min_samples:
        raise ValueError(f"عدد العينات ({n_samples}) أقل من الحد الأدنى ({min_samples})")

    target_specs = _resolve_for_model(target_specs, model, X_inputs)
    show_stages = verbose if show_stages is None else show_stages
    show_verification = verbose if show_verification is None else show_verification

    if show_stages:
        print(f"\n🔮 Predicting & Evaluating v4 (مرن: مستمر + فئوي) — الأهداف: {[s.name for s in target_specs]}")
        print("   Stage 1: Batch Predictions...")
    raw_preds = predict_batch_v4(model, X_inputs, target_specs, batch_size, show_stages)

    if show_stages:
        print("   Stage 2: Decoding (Flexible)...")
    decoded = decode_predictions_v4(raw_preds, target_specs, base_params, last_candles)

    verification_report = None
    if run_verification:
        if show_stages:
            print("   Stage 2.5: Verifying decoding (normalized vs. original)...")
        verification_report = verify_decoding(
            raw_preds, decoded, target_specs, base_params, y_true, last_candles,
            verbose=show_verification,
        )

    if show_stages:
        print("   Stage 3: Evaluation...")
    results = evaluate_predictions_v4(
        decoded, target_specs, y_true, base_params, direction_tolerance, price_tolerance
    )

    if verification_report is not None:
        results['_verification'] = verification_report

    if verbose:
        ts = _align_timestamps(_extract_timestamps(timestamps, last_candles, timestamp_col), n_samples, n_samples)
        _print_v4_report(results, target_specs, n_samples, n_display, y_true is not None,
                         timestamps=ts, asset=asset)

    return results


def _print_v4_report(results, target_specs, n_samples, n_display, has_truth,
                     timestamps=None, asset=None, legend=True):
    """
    تقرير مضغوط: جدول واحد بآخر `n_display` عينة لكل الأهداف (الأحدث في الأسفل)،
    ثم سطر أداء واحد لكل هدف على كامل العينات (إن وُجدت أهداف حقيقية).
    """
    specs = [s for s in resolve_targets(target_specs) if isinstance(results.get(s.name), dict)]
    k = int(min(n_display, n_samples)) if n_display else 0

    if k > 0:
        table = build_latest_table(results, specs, k, timestamps, asset)
        print(f"\n📊 آخر {k} عينة لكل هدف (الأحدث في الأسفل):")
        print_latest_table(table, legend=legend)

    if has_truth:
        lines = []
        for spec in specs:
            r = results[spec.name]
            if spec.kind == 'continuous' and 'error_abs' in r:
                n_valid = int(r.get('n_valid', len(r['error_abs'])))
                lines.append(
                    f"   ▸ {spec.name:<6}│ MAE {_fmt_price(_nanmean(r['error_abs']))}"
                    f" │ MAPE {_nanmean(r['error_pct']):.4f}%"
                    f" │ Win {r.get('win_rate', float('nan')):.2f}%"
                    f" │ corr {r.get('correlation', 0.0):+.3f}"
                    f" │ n={n_valid:,}"
                )
            elif spec.kind == 'categorical' and 'accuracy' in r:
                lines.append(f"   ▸ {spec.name:<6}│ Accuracy {r['accuracy']:.2f}% │ n={len(r['correct']):,}")
        if lines:
            print("\n📈 الأداء على كامل العينات:")
            print("\n".join(lines))


# دالة قديمة تبقى للتوافق الخلفي
def predict_with_evaluation_v3(model, X_inputs, base_params, last_candles=None, y_true=None,
                                verbose=False, n_display=5, batch_size=256,
                                direction_tolerance=0.2, price_tolerance=None, min_samples=1):
    """(نسخة سابقة - محفوظة للتوافق) تستخدم داخلياً محرك v4 بأهداف الأسعار الافتراضية."""
    return predict_with_evaluation_v4(
        model, X_inputs, DEFAULT_PRICE_TARGETS, base_params, last_candles, y_true,
        verbose, n_display, batch_size, direction_tolerance, price_tolerance, min_samples,
        run_verification=False,
    )


## 8️⃣ اختبار كل الأصول (Assets) دفعة واحدة — test_all_assets_v4

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# اختبار كل الأصول (Assets) دفعة واحدة — نسخة v4.2: مخرجات مضغوطة ومرنة الأهداف
# ═══════════════════════════════════════════════════════════════════════════
#
# ما تغيّر في العرض:
#   • ملخص التحقق من فك التشفير يظهر **مرة واحدة** (جدول واحد لكل الأهداف عبر كل الأصول)
#     بدل تكراره لكل أصل.
#   • لكل أصل: جدول واحد بآخر `n_display` عينة لكل هدف + سطر أداء واحد لكل هدف.
#   • في النهاية: جدول ملخص لكل الأصول + جدول إجمالي لكل هدف.
#   • الأهداف تُمرَّر كنصوص أو TargetSpec؛ والأهداف غير المطلوبة تُتجاهل.
# ═══════════════════════════════════════════════════════════════════════════

def summarize_verification(verification_df: pd.DataFrame) -> pd.DataFrame:
    """يجمع تقارير verify_decoding لكل الأصول في صف واحد لكل هدف."""
    if verification_df is None or verification_df.empty:
        return pd.DataFrame()

    def _col(g, name):
        return pd.to_numeric(g[name], errors='coerce') if name in g else pd.Series(np.nan, index=g.index)

    rows = []
    for tgt, g in verification_df.groupby('target', sort=False):
        n = _col(g, 'n_samples') if 'n_samples' in g else pd.Series(1.0, index=g.index)

        def wmean(name):
            v = _col(g, name)
            m = v.notna() & n.notna() & (n > 0)
            return float(np.average(v[m], weights=n[m])) if m.any() else float('nan')

        n_failed = int((g['status'] == 'FAILED').sum()) if 'status' in g else 0
        row = {
            'target': tgt,
            'kind': g['kind'].iloc[0] if 'kind' in g else '',
            'assets': len(g),
            'status': '✅ ok' if n_failed == 0 else f'❌ {n_failed} فشل',
        }
        if row['kind'] == 'continuous':
            row.update({
                'roundtrip_max_diff': _nanmax(_col(g, 'roundtrip_max_diff')),
                'true_consistency_max_diff': _nanmax(_col(g, 'true_value_consistency_max_diff')),
                'space_ratio_mean': _nanmean(_col(g, 'space_consistency_ratio_mean')),
                'mae_scaled': wmean('mae_scaled'),
                'mae_real': wmean('mae_real'),
            })
        else:
            row.update({
                'roundtrip_max_diff': float('nan'), 'true_consistency_max_diff': float('nan'),
                'space_ratio_mean': float('nan'),
                'mae_scaled': float('nan'), 'mae_real': float('nan'),
            })
        rows.append(row)
    return pd.DataFrame(rows)


def print_verification_summary(verification_df: pd.DataFrame):
    """يطبع ملخص التحقق من فك التشفير مرة واحدة."""
    summary = summarize_verification(verification_df)
    if summary.empty:
        return summary
    print(f"\n{'=' * 100}")
    print("🔍 التحقق من فك التشفير — مرة واحدة لكل الأصول (round-trip / اتساق الفضاءين المطبّع والأصلي)")
    print("=" * 100)
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
    n_failed = int((verification_df['status'] == 'FAILED').sum()) if 'status' in verification_df else 0
    print("✅ جميع الأهداف والأصول اجتازت التحقق" if n_failed == 0 else f"❌ {n_failed} حالة فشل — راجع verification_summary")
    print("   (mae_scaled / mae_real: متوسط مرجَّح بعدد العينات؛ mae_real يُقارَن فقط إن كانت أسعار الأصول بمقياس متقارب "
          "— تفصيله لكل أصل في جدول الملخص أدناه)")
    return summary


def _overall_metrics_table(all_results: List[Dict], specs: List['TargetSpec']) -> pd.DataFrame:
    """مقاييس مجمّعة (بالعينات) لكل هدف عبر كل الأصول."""
    rows = []
    for s in specs:
        if s.kind == 'continuous':
            errs, pcts, dirs = [], [], []
            for res in all_results:
                r = res.get(s.name)
                if not isinstance(r, dict) or 'error_abs' not in r:
                    continue
                v = np.asarray(r.get('has_truth', np.ones(len(r['error_abs']), dtype=bool)), dtype=bool)
                errs.append(np.asarray(r['error_abs'])[v])
                pcts.append(np.asarray(r['error_pct'])[v])
                dirs.append(np.asarray(r['direction_correct'])[v])
            if errs:
                e, p, d = np.concatenate(errs), np.concatenate(pcts), np.concatenate(dirs)
                rows.append({'target': s.name, 'n': int(len(e)), 'MAE': _nanmean(e),
                             'MAPE_%': _nanmean(p), 'Win_%': float(d.mean() * 100) if len(d) else float('nan')})
        else:
            accs = [(r['accuracy'], len(r['correct'])) for r in (res.get(s.name) for res in all_results)
                    if isinstance(r, dict) and 'accuracy' in r]
            if accs:
                tot = sum(n for _, n in accs)
                rows.append({'target': s.name, 'n': int(tot),
                             'Acc_%': float(sum(a * n for a, n in accs) / tot)})
    return pd.DataFrame(rows)


def test_all_assets_v4(
    model,
    test_dict: Dict,
    timeframes: List[str],
    target_specs,
    verbose: bool = True,
    n_display: int = 1,
    batch_size: int = 256,
    range_frac: float = 0.2,
    min_samples_per_asset: int = 20,
    run_verification: bool = True,
    timestamp_col: Optional[int] = None,
    timestamp_key: Optional[str] = None,
) -> Dict:
    """
    اختبار النموذج على كل أصل/عملة على حدة، بدعم أهداف مستمرة وفئوية معاً.

    target_specs: TargetSpec أو نصوص ('close' / 'high,low') أو None/'all'. الأهداف التي
                  يُخرجها النموذج ولم تُطلب تُتجاهل بالكامل.
    n_display:    عدد **آخر** العينات المعروضة لكل هدف في كل أصل (0 = بلا جدول عينات).
    timestamp_key / timestamp_col: مصدر التاريخ (مفتاح داخل بيانات الأصل، أو رقم عمود
                  في last_candles). إن لم يُحدَّد يُبحث عن مفاتيح شائعة (timestamps/times/dates...).

    Returns:
        dict: {'per_asset_results', 'aggregated_results', 'verification_summary',
               'asset_summary', 'target_specs'}
    """
    all_results: List[Dict] = []
    verification_rows: List[Dict] = []
    summary_rows: List[Dict] = []
    specs: Optional[List['TargetSpec']] = None
    n_assets = len(test_dict)

    if verbose:
        print(f"\n{'=' * 100}")
        print(f"🧪 اختبار {n_assets} عملة (v4.2) — يُعرض آخر {n_display} عينة لكل هدف")
        print("=" * 100)

    for idx, (asset_name, test_data) in enumerate(test_dict.items(), 1):
        n_samples = len(test_data['base_params'])
        if n_samples < min_samples_per_asset:
            continue

        X_inputs = tuple([test_data[f'X_{tf_name}'] for tf_name in timeframes])
        if specs is None:
            specs = _resolve_for_model(target_specs, model, X_inputs)
            if verbose:
                print(f"🎯 الأهداف المُقيَّمة: {[s.name for s in specs]}")

        base_params = test_data['base_params']
        last_candles = test_data.get('last_candles')
        y_true = {f'y_{s.name}': test_data['y'][s.name] for s in specs if s.name in test_data['y']}

        try:
            ts = _align_timestamps(_find_timestamps(test_data, timestamp_key, timestamp_col),
                                   n_samples, n_samples)

            asset_results = predict_with_evaluation_v4(
                model=model,
                X_inputs=X_inputs,
                target_specs=specs,
                base_params=base_params,
                last_candles=last_candles,
                y_true=y_true,
                verbose=False,
                n_display=0,
                batch_size=batch_size,
                direction_tolerance=range_frac,
                run_verification=run_verification,
            )

            verification = asset_results.pop('_verification', None)
            if verification:
                for target_name, rep in verification.items():
                    verification_rows.append({'asset': asset_name, 'n_samples': n_samples, **rep})

            if verbose:
                print(f"\n{'─' * 100}")
                print(f"[{idx}/{n_assets}] 🪙 {asset_name}  ({n_samples:,} عينة)")
                _print_v4_report(asset_results, specs, n_samples, n_display, bool(y_true),
                                 timestamps=ts, asset=None, legend=False)

            row = {'asset': asset_name, 'n': n_samples}
            for s in specs:
                r = asset_results.get(s.name)
                if not isinstance(r, dict):
                    continue
                if s.kind == 'continuous' and 'error_abs' in r:
                    row[f'{s.name}_MAE'] = _nanmean(r['error_abs'])
                    row[f'{s.name}_win%'] = r.get('win_rate', float('nan'))
                elif s.kind == 'categorical' and 'accuracy' in r:
                    row[f'{s.name}_acc%'] = r['accuracy']
            summary_rows.append(row)

            asset_results['asset'] = asset_name
            asset_results['n_samples'] = n_samples
            all_results.append(asset_results)

        except Exception as e:
            print(f"   ❌ خطأ في {asset_name}: {e}")
            import traceback
            if verbose:
                traceback.print_exc()
            continue

    if specs is None:
        specs = resolve_targets(target_specs)

    if not all_results:
        print("\n⚠️ لم يتم اختبار أي عملة بنجاح")
        return {'per_asset_results': pd.DataFrame(), 'aggregated_results': {},
                'verification_summary': pd.DataFrame(), 'asset_summary': pd.DataFrame(),
                'target_specs': specs}

    results_df = pd.DataFrame(all_results)
    aggregated = calculate_aggregated_metrics_v4(results_df, specs)
    verification_df = pd.DataFrame(verification_rows) if verification_rows else pd.DataFrame()
    asset_summary = pd.DataFrame(summary_rows)

    if verbose:
        if run_verification and not verification_df.empty:
            print_verification_summary(verification_df)

        if len(asset_summary) > 1:
            print(f"\n{'=' * 100}")
            print("📋 ملخص الأداء لكل أصل (MAE بوحدة سعر الأصل، win% = صحة الاتجاه)")
            print("=" * 100)
            save_or_print(asset_summary.round(4), 'per_asset_summary')

        overall = _overall_metrics_table(all_results, specs)
        if not overall.empty:
            print(f"\n{'=' * 100}")
            print("🧮 الإجمالي لكل هدف عبر كل الأصول (مجمّع بالعينات)")
            print("=" * 100)
            print(overall.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    return {
        'per_asset_results': results_df,
        'aggregated_results': aggregated,
        'verification_summary': verification_df,
        'asset_summary': asset_summary,
        'target_specs': specs,
    }


def calculate_aggregated_metrics_v4(results_df: pd.DataFrame, target_specs) -> Dict:
    """حساب metrics مجمّعة عبر كل الأصول، لكل هدف بحسب نوعه (آمنة مع NaN)."""
    target_specs = resolve_targets(target_specs)
    aggregated = {
        'total_assets': len(results_df),
        'total_samples': int(results_df['n_samples'].sum()),
        'avg_samples_per_asset': float(results_df['n_samples'].mean()),
    }

    weights = (results_df['n_samples'] / results_df['n_samples'].sum()).to_numpy()

    for spec in target_specs:
        maes, accs, w_used = [], [], []
        for pos, (_, row) in enumerate(results_df.iterrows()):
            r = row[spec.name]
            if spec.kind == 'continuous' and 'error_abs' in r:
                maes.append(_nanmean(r['error_abs']))
                w_used.append(weights[pos])
            elif spec.kind == 'categorical' and 'accuracy' in r:
                accs.append(r['accuracy'])

        if maes:
            m = np.asarray(maes, dtype=np.float64)
            w = np.asarray(w_used, dtype=np.float64)
            ok = np.isfinite(m)
            if ok.any():
                aggregated[f'{spec.name}_weighted_mae'] = float(np.average(m[ok], weights=w[ok]))
                aggregated[f'{spec.name}_mean_mae'] = float(m[ok].mean())
        if accs:
            aggregated[f'{spec.name}_mean_accuracy'] = float(np.mean(accs))

    return aggregated


# دالة قديمة تبقى للتوافق الخلفي (تقبل الآن نصوصاً أو TargetSpec)
def test_all_assets(model, test_dict, timeframes, targets, verbose=True, n_display=1,
                     batch_size=256, range_frac=0.2):
    """(نسخة سابقة - محفوظة للتوافق)"""
    specs = resolve_targets(targets)
    result = test_all_assets_v4(model, test_dict, timeframes, specs, verbose, n_display,
                                 batch_size, range_frac, run_verification=False)
    result['detailed_predictions'] = None
    return result


## 🔴 التداول الحي: تقرير آخر N عينات — `predict_latest_v4` / `predict_latest_all_assets`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🔴 التداول الحي: تقرير آخر N عينات (التاريخ · التوقع · الثقة · عدم اليقين · التحرك · السعر)
# ═══════════════════════════════════════════════════════════════════════════
#
# • تشغّل النموذج على آخر N عينة فقط (سريعة — مناسبة للاستدعاء المتكرر لايف).
# • لا تحتاج أي هدف حقيقي: آخر عينة بلا هدف تظهر بـ ⏳ ولا تُحسب في أي مقياس.
# • تعمل على بيانات الاختبار أيضاً (فإن وُجدت أهداف فعلية تظهر true وعلامة ✅/❌).
# • تعتمد ترتيب العينات زمنياً (الأقدم أولاً، والأحدث آخراً).
# ═══════════════════════════════════════════════════════════════════════════

def predict_latest_v4(
    model,
    X_inputs: Tuple[np.ndarray, ...],
    base_params: np.ndarray,
    last_candles: Optional[np.ndarray] = None,
    target_specs=None,
    n_display: int = 5,
    timestamps=None,
    timestamp_col: Optional[int] = None,
    y_true: Optional[Dict[str, np.ndarray]] = None,
    batch_size: int = 256,
    asset: Optional[str] = None,
    verbose: bool = True,
    legend: bool = True,
) -> pd.DataFrame:
    """
    يتنبأ بآخر `n_display` عينة ويُرجع جدولاً (وتطبعه إن verbose=True).

    Args:
        X_inputs:      نفس مدخلات النموذج (tuple لكل إطار زمني)، مرتبة زمنياً.
        base_params:   [N, 2] = (median, iqr) لفك التطبيع.
        last_candles:  الشمعة المرجعية (سعر الدخول) لكل عينة؛ الأعمدة price_index ضرورية لحساب
                       التحرك المتوقع. أعمدة الهدف المستقبلي (future_col) اختيارية.
        target_specs:  نصوص ('close' / 'high,low') أو TargetSpec أو None/'all'.
        timestamps:    تاريخ كل عينة (datetime / نص / epoch بالثواني أو مللي)، بطول N أو n_display.
        timestamp_col: بديلاً عن timestamps: رقم عمود التاريخ داخل last_candles.
        y_true:        اختياري: {'y_close': ...} لإظهار الهدف الفعلي عند توفره.

    Returns:
        DataFrame: idx, date, [asset], target, signal, confidence_%, uncertainty_%, move_%,
                   pred, pred_lo, pred_hi, entry, true, ok ...
    """
    X_inputs = tuple(np.asarray(x) for x in X_inputs)
    n_total = X_inputs[0].shape[0]
    if n_total == 0:
        raise ValueError("لا توجد عينات.")
    k = int(min(max(int(n_display), 1), n_total))

    specs = _resolve_for_model(target_specs, model, X_inputs)
    sl = slice(n_total - k, n_total)

    X_last = tuple(x[sl] for x in X_inputs)
    bp = None if base_params is None else np.asarray(base_params)[sl]
    lc = None if last_candles is None else np.asarray(last_candles)[sl]
    ts_last = _align_timestamps(_extract_timestamps(timestamps, last_candles, timestamp_col), n_total, k)
    yt = None if not y_true else {kk: np.asarray(v)[sl] for kk, v in y_true.items()}

    raw = predict_batch_v4(model, X_last, specs, batch_size)
    decoded = decode_predictions_v4(raw, specs, bp, lc)
    if yt:
        decoded = evaluate_predictions_v4(decoded, specs, yt, bp)

    table = build_latest_table(decoded, specs, n_display=k, timestamps=ts_last, asset=asset)

    if verbose:
        title = f"\n🔴 آخر {k} عينة" + (f" — {asset}" if asset else "") + " (الأحدث في الأسفل):"
        print_latest_table(table, title=title, legend=legend)
    return table


def predict_latest_all_assets(
    model,
    test_dict: Dict,
    timeframes: List[str],
    target_specs=None,
    n_display: int = 5,
    latest_only: bool = False,
    timestamp_col: Optional[int] = None,
    timestamp_key: Optional[str] = None,
    batch_size: int = 256,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    تقرير آخر N عينات لكل الأصول في test_dict (أو آخر عينة فقط لكل أصل إن latest_only=True،
    فيُطبع جدول واحد مضغوط بكل الأصول — مناسب للقرار اللحظي).
    """
    k = 1 if latest_only else int(n_display)
    frames = []
    specs = None

    for asset_name, test_data in test_dict.items():
        try:
            X_inputs = tuple(test_data[f'X_{tf_name}'] for tf_name in timeframes)
            if specs is None:
                specs = _resolve_for_model(target_specs, model, X_inputs)
            ts = _find_timestamps(test_data, timestamp_key, timestamp_col)
            y_true = {f'y_{s.name}': test_data['y'][s.name]
                      for s in specs if 'y' in test_data and s.name in test_data['y']} or None
            frames.append(predict_latest_v4(
                model, X_inputs, test_data['base_params'], test_data.get('last_candles'),
                specs, n_display=k, timestamps=ts, y_true=y_true, batch_size=batch_size,
                asset=asset_name, verbose=(verbose and not latest_only), legend=False,
            ))
        except Exception as e:
            print(f"   ❌ {asset_name}: {e}")

    if not frames:
        return pd.DataFrame()
    table = pd.concat(frames, ignore_index=True)

    if verbose:
        if latest_only:
            print(f"\n🔴 آخر عينة لكل أصل ({table['asset'].nunique()} أصل):")
            print_latest_table(table, legend=True)
        else:
            print("\n  " + _latest_legend(table))
    return table


# ══════════════════════════════════════════════════════════════════════════
# 🚀 أمثلة استخدام
# ══════════════════════════════════════════════════════════════════════════
#
# # أصل واحد — آخر 5 عينات، هدف close فقط (والنموذج يُخرج الثلاثة؛ الباقي يُتجاهل):
# table = predict_latest_v4(model, X_inputs, base_params, last_candles,
#                           target_specs='close', n_display=5, timestamps=times)
#
# # هدفان بنصوص، والتاريخ من عمود داخل last_candles (مثلاً العمود 3):
# table = predict_latest_v4(model, X_inputs, base_params, last_candles,
#                           target_specs='high,low', n_display=5, timestamp_col=3)
#
# # كل الأصول — آخر عينة لكل أصل (لقطة لايف):
# snap = predict_latest_all_assets(model, test_dict, ['1h', '4h', '1D'],
#                                  target_specs=['high', 'low', 'close'], latest_only=True)
#
# # في اختبار كل الأصول: n_display يعرض آخر العينات، وأهداف نصية مقبولة:
# res = test_all_assets_v4(model, test_dict, ['1h', '4h', '1D'], 'close', n_display=5)


## 9️⃣ دالة مساعدة مشتركة: تسطيح النتائج إلى DataFrame واحد

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🧱 دالة مساعدة مشتركة: تحويل نتائج كل الأصول إلى DataFrame مسطّح واحد
# ═══════════════════════════════════════════════════════════════════════════

def build_flat_dataframe(
    per_asset_results: pd.DataFrame,
    target_specs: List['TargetSpec'],
    continuous_only: bool = False,
) -> pd.DataFrame:
    """
    يحوّل نتائج test_all_assets_v4 (DataFrame فيه عمود لكل هدف يحوي dict)
    إلى DataFrame واحد "مسطّح" بصف لكل (عملة × هدف × عينة). تستخدمه كل
    دوال التحليل والتقارير أدناه بدل تكرار كود التجميع في كل مرة.

    الأعمدة المشتركة: asset, target, kind, confidence, uncertainty
    الأعمدة الإضافية للمستمر: pred, true, entry, abs_error, pct_error,
                              direction_correct, predicted_change, price_change
    الأعمدة الإضافية للفئوي: pred_label, true_label, correct, pred_proba
    """
    target_specs = resolve_targets(target_specs)
    rows = []

    for _, row in per_asset_results.iterrows():
        asset = row['asset']

        for spec in target_specs:
            if spec.name not in row or not isinstance(row[spec.name], dict):
                continue
            data = row[spec.name]
            if continuous_only and spec.kind != 'continuous':
                continue

            n = len(data.get('pred_real', data.get('pred_index', [])))
            confidence = np.asarray(data.get('confidence', [np.nan] * n), dtype=np.float64)
            uncertainty = np.asarray(data.get('uncertainty_real', [np.nan] * n), dtype=np.float64)
            aleatoric = np.asarray(data.get('aleatoric', [np.nan] * n), dtype=np.float64)
            epistemic = np.asarray(data.get('epistemic', [np.nan] * n), dtype=np.float64)

            if spec.kind == 'continuous' and 'true_real' in data:
                pred = np.asarray(data['pred_real'], dtype=np.float64)
                true = np.asarray(data['true_real'], dtype=np.float64)
                entry = np.asarray(data['entry_price'], dtype=np.float64)
                abs_error = np.asarray(data['error_abs'], dtype=np.float64)
                pct_error = np.asarray(data['error_pct'], dtype=np.float64)
                direction_correct = np.asarray(data['direction_correct'], dtype=np.int32)
                predicted_change = pred - entry
                price_change = true - entry
                has_truth = np.asarray(data.get('has_truth', np.isfinite(true)), dtype=bool)
                ts_arr = np.asarray(data.get('timestamp', np.full(n, np.nan)), dtype=np.float64)

                for i in range(n):
                    if not has_truth[i]:
                        continue    # عينة بلا هدف فعلي (مثل آخر شمعة في التداول الحي) — لا تدخل التحليل
                    rows.append({
                        'asset': asset, 'target': spec.name, 'kind': 'continuous',
                        'pred': pred[i], 'true': true[i], 'entry': entry[i],
                        'abs_error': abs_error[i], 'pct_error': pct_error[i],
                        'confidence': confidence[i] if i < len(confidence) else np.nan,
                        'uncertainty': uncertainty[i] if i < len(uncertainty) else np.nan,
                        'aleatoric': aleatoric[i] if i < len(aleatoric) else np.nan,
                        'epistemic': epistemic[i] if i < len(epistemic) else np.nan,
                        'direction_correct': int(direction_correct[i]),
                        'correct': int(direction_correct[i]),
                        'predicted_change': predicted_change[i],
                        'price_change': price_change[i],
                        'timestamp': ts_arr[i] if i < len(ts_arr) else np.nan,
                        'predicted_change_pct': (predicted_change[i] / (abs(entry[i]) + 1e-7)) * 100,
                    })

            elif spec.kind == 'categorical' and 'true_label' in data:
                pred_label = data['pred_label']
                true_label = data['true_label']
                correct = data['correct']
                pred_proba = np.asarray(data.get('pred_proba', [np.nan] * n), dtype=np.float64)

                for i in range(n):
                    rows.append({
                        'asset': asset, 'target': spec.name, 'kind': 'categorical',
                        'pred_label': pred_label[i], 'true_label': true_label[i],
                        'confidence': confidence[i] if i < len(confidence) else pred_proba[i],
                        'uncertainty': uncertainty[i] if i < len(uncertainty) else np.nan,
                        'aleatoric': aleatoric[i] if i < len(aleatoric) else np.nan,
                        'epistemic': epistemic[i] if i < len(epistemic) else np.nan,
                        'correct': int(correct[i]),
                        'direction_correct': int(correct[i]),
                        'pred_proba': pred_proba[i],
                    })

    df = pd.DataFrame(rows)

    # ✅ ضمان float64 (يمنع مشاكل float16 التي واجهها الكود الأصلي)
    numeric_cols = [c for c in df.columns if df[c].dtype.kind in 'fi' and c not in ('correct', 'direction_correct')]
    for c in numeric_cols:
        df[c] = df[c].astype(np.float64)

    return df


## 🔟 🧠 تقرير: الثقة والفهم (Trust & Calibration Report)

يجيب على: **"إلى أي مدى يفهم النموذج توقعاته، وهل يمكن الوثوق بدرجة ثقته؟"** — الآن مع مخطط معايرة (Calibration Curve) لكل هدف.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🧠 تقرير الثقة والفهم (Trust & Calibration Report)
# ═══════════════════════════════════════════════════════════════════════════
#
# الهدف: تقييم "مدى فهم النموذج لتوقعاته" — أي هل ثقة النموذج (confidence)
# وعدم يقينه (uncertainty) يعكسان فعلاً احتمال صحة التوقع؟ نموذج "يفهم نفسه"
# جيداً هو نموذج تكون ثقته عالية عندما يكون صحيحاً ومنخفضة عندما يخطئ.
# ═══════════════════════════════════════════════════════════════════════════

def generate_trust_report(
    df: pd.DataFrame,
    group_by: Optional[str] = 'target',
    n_bins: int = 10,
    verbose: bool = True,
) -> Dict:
    """
    يبني تقرير ثقة شامل من DataFrame مسطّح (خرج build_flat_dataframe).

    المقاييس المحسوبة لكل مجموعة (كل target أو كل asset حسب group_by):
      • ECE  (Expected Calibration Error): متوسط الفرق |ثقة - دقة فعلية| مرجّحاً بالعدد
      • MCE  (Maximum Calibration Error): أسوأ فجوة في أي bin
      • Brier Score: متوسط (confidence - correct)^2 — كلما قلّ كان أفضل
      • Sharpness: متوسط عدم اليقين (uncertainty) — نموذج "حاد" يعطي عدم يقين منخفض
                   عند الصحة وعالٍ عند الخطأ، وليس ثابتاً دوماً
      • Confidence-Accuracy Correlation: هل الثقة الأعلى تعني دقة أعلى فعلاً؟
      • Coverage@1σ / @2σ: نسبة الأخطاء الواقعة ضمن 1 أو 2 انحراف معياري متوقَّع
      • Trust Score (0-100): درجة مُركّبة = 100 × (1 - ECE) × (0.5 + 0.5×correlation_scaled)
        (تُعرض المعادلة بوضوح حتى يسهل تعديل الأوزان حسب الحاجة)

    Returns:
        dict: {'overall': {...}, 'by_group': DataFrame, 'reliability_curve': DataFrame}
    """
    def _metrics_for(sub: pd.DataFrame) -> Dict:
        sub = sub.dropna(subset=['confidence'])
        if len(sub) == 0:
            return {}

        correct = sub['correct'].astype(float).values
        conf = sub['confidence'].astype(float).values

        # --- Calibration bins ---
        try:
            bins = pd.qcut(conf, q=min(n_bins, len(np.unique(conf))), duplicates='drop')
        except Exception:
            bins = pd.cut(conf, bins=n_bins)

        bin_df = pd.DataFrame({'conf': conf, 'correct': correct, 'bin': bins})
        grouped = bin_df.groupby('bin', observed=True)

        ece, mce = 0.0, 0.0
        reliability_rows = []
        for bname, g in grouped:
            if len(g) == 0:
                continue
            avg_conf = g['conf'].mean()
            avg_acc = g['correct'].mean()
            weight = len(g) / len(bin_df)
            gap = abs(avg_conf - avg_acc)
            ece += weight * gap
            mce = max(mce, gap)
            reliability_rows.append({'bin': str(bname), 'avg_confidence': avg_conf,
                                      'avg_accuracy': avg_acc, 'gap': gap, 'count': len(g)})

        brier = float(np.mean((conf - correct) ** 2))

        # --- correlation confidence vs correctness ---
        if np.std(conf) > 1e-9 and np.std(correct) > 1e-9:
            conf_acc_corr = float(np.corrcoef(conf, correct)[0, 1])
        else:
            conf_acc_corr = 0.0

        # --- uncertainty-based metrics (إن وُجدت) ---
        sharpness = float(sub['uncertainty'].mean()) if 'uncertainty' in sub and sub['uncertainty'].notna().any() else None
        unc_error_corr = None
        if 'abs_error' in sub.columns and 'uncertainty' in sub.columns:
            u, e = sub['uncertainty'].dropna(), sub['abs_error']
            common = sub.dropna(subset=['uncertainty', 'abs_error'])
            if len(common) > 2 and common['uncertainty'].std() > 1e-9:
                unc_error_corr = float(np.corrcoef(common['uncertainty'], common['abs_error'])[0, 1])

        # --- Trust score مُركّب (0-100) ---
        corr_scaled = (conf_acc_corr + 1) / 2  # من [-1,1] إلى [0,1]
        trust_score = 100.0 * max(0.0, (1 - ece)) * (0.5 + 0.5 * corr_scaled)

        return {
            'n_samples': len(sub),
            'ece': ece,
            'mce': mce,
            'brier_score': brier,
            'confidence_accuracy_corr': conf_acc_corr,
            'uncertainty_error_corr': unc_error_corr,
            'sharpness_avg_uncertainty': sharpness,
            'mean_confidence': float(conf.mean()),
            'mean_accuracy': float(correct.mean()),
            'trust_score_0_100': trust_score,
            '_reliability_rows': reliability_rows,
        }

    overall = _metrics_for(df)

    by_group_rows = []
    reliability_all = []
    if group_by and group_by in df.columns:
        for gval, sub in df.groupby(group_by):
            m = _metrics_for(sub)
            if not m:
                continue
            rel_rows = m.pop('_reliability_rows')
            for rr in rel_rows:
                rr[group_by] = gval
                reliability_all.append(rr)
            by_group_rows.append({group_by: gval, **m})

    by_group_df = pd.DataFrame(by_group_rows)
    reliability_df = pd.DataFrame(reliability_all)
    overall.pop('_reliability_rows', None)

    if verbose:
        print("=" * 100)
        print("🧠 تقرير الثقة والفهم (Trust & Calibration Report)")
        print("=" * 100)
        print(f"\n📊 التقييم العام (كل البيانات):")
        for k, v in overall.items():
            print(f"   • {k}: {v:.4f}" if isinstance(v, float) else f"   • {k}: {v}")

        _trust_verdict(overall.get('trust_score_0_100'), overall.get('ece'), overall.get('confidence_accuracy_corr'))

        if not by_group_df.empty:
            print(f"\n📋 التفصيل حسب '{group_by}':")
            cols = [group_by, 'n_samples', 'trust_score_0_100', 'ece', 'mce',
                    'confidence_accuracy_corr', 'mean_confidence', 'mean_accuracy']
            cols = [c for c in cols if c in by_group_df.columns]
            save_or_print(by_group_df[cols].round(4), 'trust_report_by_group', out_dir=DEFAULT_OUTPUT_DIR)

    return {'overall': overall, 'by_group': by_group_df, 'reliability_curve': reliability_df}


def _trust_verdict(trust_score, ece, corr):
    print(f"\n💡 الحكم العام:")
    if trust_score is None:
        return
    if trust_score >= 75:
        print(f"   ✅ النموذج 'يفهم' توقعاته جيداً (Trust Score: {trust_score:.1f}/100) — يمكن الاعتماد على الثقة كمرشّح للجودة")
    elif trust_score >= 50:
        print(f"   ⚠️  فهم متوسط (Trust Score: {trust_score:.1f}/100) — الثقة مفيدة جزئياً لكنها تحتاج معايرة")
    else:
        print(f"   ❌ فهم ضعيف (Trust Score: {trust_score:.1f}/100) — لا يُنصح بالاعتماد على درجة الثقة كما هي")

    if ece is not None:
        if ece < 0.05:
            print(f"      • المعايرة (ECE={ece:.3f}) ممتازة")
        elif ece < 0.15:
            print(f"      • المعايرة (ECE={ece:.3f}) مقبولة")
        else:
            print(f"      • المعايرة (ECE={ece:.3f}) ضعيفة — النموذج مبالغ/متحفّظ في ثقته")


## 1️⃣1️⃣ 🎯 تقرير: تحليل موجَّه للمخرجات (Output-Conditioned Analysis)

نسبة النجاح حسب نوع/حجم الحركة المتوقعة، حسب فئة الثقة، وحسب **دفعات متتالية** من العينات (لاكتشاف انحدار الأداء في دفعة معيّنة) — كل جدول كبير يُحفظ تلقائياً في ملف.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🎯 تحليل موجَّه للمخرجات (Output-Conditioned Analysis)
# ═══════════════════════════════════════════════════════════════════════════
#
# يجيب على أسئلة من نوع:
#   "ما نسبة نجاح التوقع عندما يتوقع النموذج حركة صعود بأكثر من 2%؟"
#   "هل هناك دفعة (batch) معيّنة من العينات حدث فيها انحدار واضح في الأداء
#    مقارنة بغيرها؟"
# ═══════════════════════════════════════════════════════════════════════════

def analyze_by_predicted_movement(
    df: pd.DataFrame,
    magnitude_col: str = 'predicted_change_pct',
    bins: Optional[List[float]] = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    نسبة نجاح التوقع مقسّمة حسب "نوع الحركة المتوقعة" (اتجاه × حجم الحركة).

    مثال على القراءة: "عندما توقع النموذج صعوداً بين 0.5% و 2%، نجح في
    الاتجاه 61% من المرات (على 340 عينة)".
    """
    if magnitude_col not in df.columns:
        raise ValueError(f"العمود '{magnitude_col}' غير موجود — هذا التحليل يتطلب أهدافاً مستمرة (أسعار).")

    if bins is None:
        bins = [-np.inf, -2, -0.5, 0, 0.5, 2, np.inf]
    labels = [f"({bins[i]:.1f}%, {bins[i+1]:.1f}%]" for i in range(len(bins) - 1)]

    work = df.copy()
    work['direction'] = np.where(work[magnitude_col] > 0, 'صعود', np.where(work[magnitude_col] < 0, 'هبوط', 'ثابت'))
    work['movement_bucket'] = pd.cut(work[magnitude_col], bins=bins, labels=labels)

    grouped = work.groupby(['target', 'direction', 'movement_bucket'], observed=True).agg(
        عدد=('correct', 'size'),
        نسبة_نجاح=('correct', 'mean'),
        متوسط_الثقة=('confidence', 'mean'),
    ).reset_index()
    grouped['نسبة_نجاح'] = (grouped['نسبة_نجاح'] * 100).round(2)
    grouped['متوسط_الثقة'] = grouped['متوسط_الثقة'].round(3)
    grouped = grouped[grouped['عدد'] > 0].sort_values(['target', 'direction'])

    if verbose:
        print("=" * 100)
        print("🎯 نسبة النجاح حسب نوع/حجم الحركة المتوقعة")
        print("=" * 100)
        save_or_print(grouped, 'movement_analysis', out_dir=DEFAULT_OUTPUT_DIR)

        best = grouped.loc[grouped['نسبة_نجاح'].idxmax()]
        worst = grouped.loc[grouped['نسبة_نجاح'].idxmin()]
        print(f"\n✅ أفضل نطاق حركة: {best['target']} / {best['direction']} {best['movement_bucket']} "
              f"→ نجاح {best['نسبة_نجاح']}% ({int(best['عدد'])} عينة)")
        print(f"❌ أسوأ نطاق حركة: {worst['target']} / {worst['direction']} {worst['movement_bucket']} "
              f"→ نجاح {worst['نسبة_نجاح']}% ({int(worst['عدد'])} عينة)")

    return grouped


def analyze_by_confidence_bucket(df: pd.DataFrame, n_buckets: int = 5, verbose: bool = True) -> pd.DataFrame:
    """نسبة النجاح حسب فئة الثقة (buckets متساوية العدد) — لكل target."""
    work = df.dropna(subset=['confidence']).copy()
    work['conf_bucket'] = pd.qcut(work['confidence'], q=n_buckets, duplicates='drop')

    grouped = work.groupby(['target', 'conf_bucket'], observed=True).agg(
        عدد=('correct', 'size'),
        نسبة_نجاح=('correct', 'mean'),
        متوسط_الثقة=('confidence', 'mean'),
    ).reset_index()
    grouped['نسبة_نجاح'] = (grouped['نسبة_نجاح'] * 100).round(2)

    if verbose:
        print("\n" + "=" * 100)
        print("📊 نسبة النجاح حسب فئة الثقة")
        print("=" * 100)
        save_or_print(grouped, 'confidence_bucket_analysis', out_dir=DEFAULT_OUTPUT_DIR)

    return grouped


def analyze_by_batch(
    df: pd.DataFrame,
    batch_size: int = 100,
    target: Optional[str] = None,
    z_threshold: float = 1.5,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    تحليل الأداء عبر دفعات متتالية من العينات (بترتيبها كما وردت) لاكتشاف
    "انحدار" (regression) في دفعة معيّنة مقارنة ببقية الدفعات.

    الطريقة: تقسيم العينات إلى دفعات بحجم batch_size، حساب نسبة النجاح
    ومتوسط الخطأ لكل دفعة، ثم حساب Z-score لكل دفعة بالنسبة لمتوسط/انحراف
    كل الدفعات — أي دفعة بـ |Z| >= z_threshold تُعتبر "شاذة" (انحدار أو تحسّن
    غير معتاد) وتتم المقارنة صراحة بينها وبين أفضل دفعة.
    """
    work = df if target is None else df[df['target'] == target]
    work = work.reset_index(drop=True)
    work['batch_id'] = work.index // batch_size

    agg_dict = {'correct': 'mean', 'confidence': 'mean'}
    if 'abs_error' in work.columns:
        agg_dict['abs_error'] = 'mean'

    batch_stats = work.groupby('batch_id').agg(**{
        'عدد': ('correct', 'size'),
        'نسبة_نجاح': ('correct', 'mean'),
        'متوسط_الثقة': ('confidence', 'mean'),
        **({'متوسط_الخطأ': ('abs_error', 'mean')} if 'abs_error' in work.columns else {}),
    }).reset_index()

    mean_wr, std_wr = batch_stats['نسبة_نجاح'].mean(), batch_stats['نسبة_نجاح'].std()
    batch_stats['z_score'] = (batch_stats['نسبة_نجاح'] - mean_wr) / (std_wr + 1e-9)
    batch_stats['حالة'] = np.where(
        batch_stats['z_score'] <= -z_threshold, '⚠️ انحدار',
        np.where(batch_stats['z_score'] >= z_threshold, '✅ تحسّن', 'طبيعي')
    )
    batch_stats['نسبة_نجاح'] = (batch_stats['نسبة_نجاح'] * 100).round(2)

    if verbose:
        print("\n" + "=" * 100)
        print(f"📦 تحليل الأداء عبر الدفعات (batch_size={batch_size}"
              + (f", target={target}" if target else "") + ")")
        print("=" * 100)
        save_or_print(batch_stats.round(4), f"batch_performance{'_' + target if target else ''}", out_dir=DEFAULT_OUTPUT_DIR)

        anomalies = batch_stats[batch_stats['حالة'] != 'طبيعي']
        best_batch = batch_stats.loc[batch_stats['نسبة_نجاح'].idxmax()]
        if not anomalies.empty:
            print(f"\n🔎 دفعات شاذة مقارنة بأفضل دفعة (#{int(best_batch['batch_id'])}, "
                  f"{best_batch['نسبة_نجاح']}% نجاح):")
            for _, row in anomalies.iterrows():
                diff = best_batch['نسبة_نجاح'] - row['نسبة_نجاح']
                print(f"   • الدفعة #{int(row['batch_id'])}: {row['نسبة_نجاح']}% نجاح "
                      f"({row['حالة']}) — أقل من الأفضل بـ {diff:.2f} نقطة مئوية")
        else:
            print("\n✅ لا توجد دفعات شاذة — الأداء مستقر عبر كل الدفعات")

    return batch_stats


## 1️⃣2️⃣ 🔬 تقرير: اكتشاف الأنماط آلياً (Pattern Discovery)

شجرة قرار مفسِّرة، أهمية الخصائص (Random Forest) + رسم بياني، تجميع سلوكي (K-Means) + إسقاط PCA، واكتشاف شذوذ (Isolation Forest) — لفهم متى ولماذا تكون توقعات النموذج متميزة أو خاطئة.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🔬 اكتشاف الأنماط آلياً في مخرجات النموذج (Pattern Discovery)
# ═══════════════════════════════════════════════════════════════════════════
#
# يستخدم خوارزميات تعلّم آلي بسيطة وقابلة للتفسير لاكتشاف الأنماط التي
# تجعل توقعات النموذج "متميزة" (صحيحة بثقة) أو "خاطئة" (نمط فشل متكرر):
#
#   1) شجرة قرار ضحلة  → قواعد "إن-فإن" مقروءة بشرياً تفسّر متى ينجح/يفشل
#   2) Random Forest    → أهمية كل خاصية (Feature Importance) في التمييز
#   3) K-Means Clustering → تجميع التوقعات في "أنماط سلوك" ووصف كل نمط
#   4) Isolation Forest → اكتشاف التوقعات الشاذة (Outliers) ومقارنة أدائها
# ═══════════════════════════════════════════════════════════════════════════

def detect_success_failure_patterns(
    df: pd.DataFrame,
    feature_cols: Optional[List[str]] = None,
    max_tree_depth: int = 4,
    n_clusters: int = 4,
    top_n_assets_onehot: int = 8,
    verbose: bool = True,
) -> Dict:
    """
    يبني نموذجاً تفسيرياً صغيراً لاكتشاف الأنماط وراء نجاح/فشل التوقعات.

    Returns:
        dict: {
            'tree_rules': str,                  # قواعد شجرة القرار كنص مقروء
            'feature_importance': DataFrame,     # ترتيب الخصائص حسب الأهمية
            'clusters': DataFrame,               # وصف كل عنقود سلوكي
            'clustered_df': DataFrame,           # البيانات الأصلية + عمود cluster
            'anomalies': DataFrame,              # التوقعات الشاذة وأداؤها
        }
    """
    from sklearn.tree import DecisionTreeClassifier, export_text
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.ensemble import RandomForestClassifier, IsolationForest
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler

    work = df.copy()
    if 'predicted_change_pct' not in work.columns and 'pct_error' in work.columns:
        work['predicted_change_pct'] = 0.0  # عمود بديل إن غاب (بيانات فئوية بحتة)

    if feature_cols is None:
        candidate_cols = ['confidence', 'uncertainty', 'aleatoric', 'epistemic', 'predicted_change_pct']
        feature_cols = [c for c in candidate_cols if c in work.columns]

    work = work.dropna(subset=feature_cols + ['correct'])
    if len(work) < 20:
        raise ValueError("عدد العينات الصالحة قليل جداً (<20) لاكتشاف أنماط موثوقة.")

    # ترميز أهم N أصول كـ one-hot (يبقي الباقي كفئة 'أخرى' لتفادي انفجار الأبعاد)
    if 'asset' in work.columns:
        top_assets = work['asset'].value_counts().nlargest(top_n_assets_onehot).index
        work['asset_grouped'] = np.where(work['asset'].isin(top_assets), work['asset'], 'أخرى')
        asset_dummies = pd.get_dummies(work['asset_grouped'], prefix='asset')
        X = pd.concat([work[feature_cols].reset_index(drop=True), asset_dummies.reset_index(drop=True)], axis=1)
    else:
        X = work[feature_cols].reset_index(drop=True)

    y = work['correct'].astype(int).reset_index(drop=True)

    results = {}

    # ── 1) شجرة قرار مفسِّرة ────────────────────────────────────────────────
    tree = DecisionTreeClassifier(max_depth=max_tree_depth, min_samples_leaf=max(10, len(X) // 50),
                                   class_weight='balanced', random_state=42)
    tree.fit(X, y)
    tree_rules = export_text(tree, feature_names=list(X.columns))
    results['tree_rules'] = tree_rules
    results['tree_accuracy'] = float(tree.score(X, y))
    # ✅ class_weight='balanced' يُحسّن عمد ترجيح الفئة الأقلّ على حساب دقة خامة
    # ممكنة على الفئة الأكثر — فقد تقلّ tree_accuracy (دقة خام غير موزونة)
    # عن خط أساس "توقّع الفئة الأكثر دائماً" بلا أن يعني ذلك فشل الشجرة —
    # الدقة الموزونة (balanced_accuracy_score) هي المقياس المطابق فعلياً لما دُرّبت
    # عليه الشجرة.
    results['tree_naive_baseline_accuracy'] = float(max(y.mean(), 1 - y.mean()))
    results['tree_balanced_accuracy'] = float(balanced_accuracy_score(y, tree.predict(X)))

    # ── 2) أهمية الخصائص عبر Random Forest ─────────────────────────────────
    rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
    rf.fit(X, y)
    importance_df = pd.DataFrame({
        'الخاصية': X.columns,
        'الأهمية': rf.feature_importances_,
    }).sort_values('الأهمية', ascending=False).reset_index(drop=True)
    results['feature_importance'] = importance_df

    # ── 3) تجميع سلوكي (Clustering) ─────────────────────────────────────────
    cluster_features = [c for c in feature_cols if c in work.columns]
    scaler = StandardScaler()
    Xc = scaler.fit_transform(work[cluster_features])
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    cluster_labels = km.fit_predict(Xc)

    clustered = work.copy()
    clustered['cluster'] = cluster_labels

    cluster_summary = clustered.groupby('cluster').agg(
        عدد=('correct', 'size'),
        نسبة_نجاح=('correct', 'mean'),
        **{f'متوسط_{c}': (c, 'mean') for c in cluster_features},
    ).reset_index()
    cluster_summary['نسبة_نجاح'] = (cluster_summary['نسبة_نجاح'] * 100).round(2)
    cluster_summary = cluster_summary.sort_values('نسبة_نجاح', ascending=False)
    results['clusters'] = cluster_summary
    results['clustered_df'] = clustered

    # ── 4) اكتشاف الشذوذ (نتائج "متميزة" بمعنى غير اعتيادية) ────────────────
    iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
    anomaly_flag = iso.fit_predict(work[cluster_features])  # -1 = شاذ
    work_anom = work.copy()
    work_anom['is_anomaly'] = anomaly_flag == -1
    anomalies = work_anom[work_anom['is_anomaly']]
    results['anomalies'] = anomalies

    if verbose:
        print("=" * 100)
        print("🔬 اكتشاف الأنماط آلياً في مخرجات النموذج")
        print("=" * 100)

        print(f"\n🌳 1) قواعد شجرة القرار (دقة خام: {results['tree_accuracy']*100:.1f}%، خط أساس ساذج: {results['tree_naive_baseline_accuracy']*100:.1f}%، دقة موزونة: {results['tree_balanced_accuracy']*100:.1f}%):")
        print("-" * 60)
        save_or_print(tree_rules, 'decision_tree_rules', out_dir=DEFAULT_OUTPUT_DIR)

        print(f"\n📊 2) أهمية الخصائص في التمييز بين النجاح والفشل:")
        print("-" * 60)
        save_or_print(importance_df, 'feature_importance', out_dir=DEFAULT_OUTPUT_DIR)
        print(f"   → أهم خاصية تفسّر نجاح/فشل التوقع: '{importance_df.iloc[0]['الخاصية']}'")
        plot_feature_importance(importance_df, save_dir=DEFAULT_OUTPUT_DIR)

        print(f"\n🧩 3) الأنماط السلوكية المكتشفة (K-Means, k={n_clusters}):")
        print("-" * 60)
        save_or_print(cluster_summary.round(4), 'cluster_summary', out_dir=DEFAULT_OUTPUT_DIR)
        best_c = cluster_summary.iloc[0]
        worst_c = cluster_summary.iloc[-1]
        print(f"   ✅ أفضل نمط: العنقود #{int(best_c['cluster'])} بنجاح {best_c['نسبة_نجاح']}%")
        print(f"   ❌ أسوأ نمط: العنقود #{int(worst_c['cluster'])} بنجاح {worst_c['نسبة_نجاح']}%")
        try:
            plot_clusters_2d(clustered, cluster_features, save_dir=DEFAULT_OUTPUT_DIR)
        except Exception as e:
            print(f"   ⚠️ تعذر رسم العناقيد: {e}")

        print(f"\n🚨 4) التوقعات الشاذة (Isolation Forest): {len(anomalies)} من {len(work)} "
              f"({len(anomalies)/len(work)*100:.1f}%)")
        print("-" * 60)
        if len(anomalies) > 0:
            normal_acc = work_anom[~work_anom['is_anomaly']]['correct'].mean() * 100
            anom_acc = anomalies['correct'].mean() * 100
            print(f"   • دقة التوقعات العادية: {normal_acc:.1f}%")
            print(f"   • دقة التوقعات الشاذة:  {anom_acc:.1f}%")
            save_or_print(anomalies, 'anomalous_predictions', out_dir=DEFAULT_OUTPUT_DIR)
            if anom_acc < normal_acc - 10:
                print(f"   ⚠️  التوقعات الشاذة أقل دقة بوضوح — قد تحتاج فلترة أو مراجعة يدوية")
            elif anom_acc > normal_acc + 10:
                print(f"   💡 التوقعات الشاذة أعلى دقة رغم ندرتها — تستحق دراسة (فرصة محتملة)")
            else:
                print(f"   ℹ️  لا فرق واضح في الدقة بين الشاذ والعادي")

    return results


## 1️⃣3️⃣ 📈 مقاييس أداء تداول إضافية (Tearsheet-style Metrics)

Sharpe, Sortino, Omega, Profit Factor, SQN, Max Drawdown + محاكاة منحنى رأس المال — مبنية على ممارسات تقارير الـ backtest المالية القياسية (Sharpe/Drawdown/Profit Factor هي المقاييس الأساسية في أي "tearsheet" تداول احترافي).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 📈 مقاييس أداء تداول إضافية (Tearsheet-style metrics)
# ═══════════════════════════════════════════════════════════════════════════
#
# مبنية على الممارسات القياسية في تقارير الـ backtest المالية (tearsheets):
# Sharpe, Sortino, Omega, Profit Factor, SQN, Max Drawdown — بالإضافة لمنحنى
# رأس المال (Equity Curve) نفسه لكل استراتيجية.
# ═══════════════════════════════════════════════════════════════════════════

def compute_trading_performance_metrics(returns_pct: np.ndarray, risk_free_pct: float = 0.0,
                                        periods_per_year: Optional[float] = None) -> Dict[str, float]:
    """
    يحسب مجموعة مقاييس أداء قياسية من سلسلة عوائد الصفقات (بالنسبة المئوية).

    • Sharpe Ratio    : العائد المعدَّل بالمخاطرة (متوسط العائد / انحرافه المعياري)
    • Sortino Ratio    : مثل Sharpe لكن يُعاقب فقط التذبذب السلبي (الخسائر)
    • Max Drawdown %   : أكبر تراجع من قمة إلى قاع في منحنى رأس المال
    • Profit Factor    : إجمالي الأرباح / إجمالي الخسائر (>1 مربح، >2 قوي)
    • Win Rate %       : نسبة الصفقات الرابحة
    • Omega Ratio      : نسبة الأرباح المرجَّحة باحتمالها إلى الخسائر المرجَّحة (>1 جيد)
    • SQN              : System Quality Number لـ Van Tharp (>2 قابل للتداول، >3 ممتاز)

    periods_per_year: إن كانت returns_pct عوائد فترات متتالية (مثلاً عائد المحفظة اليومي) يُحسب
    Sharpe/Sortino سنوياً بـ √periods_per_year. بلاها يُعاد Sharpe **لكل صفقة** (متوسط/انحراف بلا
    تضخيم). ⚠️ سابقاً كان Sharpe يُضرب بـ √(عدد الصفقات) — أي إحصاء t لا نسبة Sharpe، يكبر مع عدد
    الصفقات بلا حدّ (39,545 صفقة بميزة ضئيلة → "Sharpe 4.4 ✅"). SQN بسقف N=100 (SQN100) لنفس السبب.
    """
    r = np.asarray(returns_pct, dtype=np.float64)
    r = r[~np.isnan(r)]
    if len(r) == 0:
        return {}

    # ✅ قص عند -100% (لا يمكن خسارة أكثر من 100% من المركز المراهن عليه في
    # محاكاة بلا رافعة مالية). بلا هذا القص، صفقة واحدة بعائد أقل من -100%
    # (مثلاً: النموذج توقّع هبوطاً لأصل قفز أكثر من 100% في نفس الفترة — وارد
    # فعلياً في عملات ميمية متطرفة التقلّب) تقلب حاصل `equity` في `np.cumprod` إلى
    # قيمة سالبة، فيُفسد كل قيمة لاحقة (الضرب تراكمي) ويُنتج `max_drawdown_pct`
    # تتجاوز -100% رياضياً بلا معنى اقتصادي (لا يوجد "تراجع" أكثر من خسارة كل
    # رأس المال فعلياً).
    r = np.maximum(r, -100.0)

    excess = r - risk_free_pct
    ann = np.sqrt(periods_per_year) if periods_per_year else 1.0
    sharpe = float(np.mean(excess) / (np.std(excess) + 1e-9)) * ann

    downside = excess[excess < 0]
    sortino = float(np.mean(excess) / (np.std(downside) + 1e-9)) * ann if len(downside) > 0 else float('inf')

    equity = 100.0 * np.cumprod(1 + r / 100.0)
    running_max = np.maximum.accumulate(equity)
    drawdown = (equity - running_max) / running_max * 100
    max_drawdown = float(drawdown.min())

    gains = r[r > 0].sum()
    losses = -r[r < 0].sum()
    profit_factor = float(gains / losses) if losses > 0 else float('inf')

    win_rate = float(np.mean(r > 0)) * 100

    threshold = 0.0
    gains_omega = np.sum(np.maximum(r - threshold, 0))
    losses_omega = np.sum(np.maximum(threshold - r, 0))
    omega = float(gains_omega / losses_omega) if losses_omega > 0 else float('inf')

    sqn = float(np.mean(r) / (np.std(r) + 1e-9)) * np.sqrt(min(len(r), 100))

    return {
        'n_trades': len(r),
        'sharpe_basis': 'annualized' if periods_per_year else 'per_trade',
        'sharpe_ratio': sharpe,
        'sortino_ratio': sortino,
        'max_drawdown_pct': max_drawdown,
        'profit_factor': profit_factor,
        'win_rate_pct': win_rate,
        'omega_ratio': omega,
        'sqn': sqn,
        'total_return_pct': float(equity[-1] - 100.0),
        'avg_return_per_trade_pct': float(np.mean(r)),
    }


def print_performance_verdict(metrics: Dict[str, float]) -> None:
    """طباعة حكم نوعي سريع على مقاييس الأداء (يشبه تفسير tearsheet مالي قياسي)."""
    if not metrics:
        print("⚠️ لا توجد بيانات كافية لحساب مقاييس الأداء")
        return
    if metrics.get('sharpe_basis') == 'annualized':
        print(f"   • Sharpe Ratio (سنوي): {metrics['sharpe_ratio']:.3f}  "
              f"({'✅ جيد' if metrics['sharpe_ratio'] > 1 else '⚠️ ضعيف' if metrics['sharpe_ratio'] > 0 else '❌ سلبي'})")
    else:
        print(f"   • Sharpe (لكل صفقة، غير سنوي): {metrics['sharpe_ratio']:.4f}")
    print(f"   • Sortino Ratio:      {metrics['sortino_ratio']:.3f}")
    print(f"   • Max Drawdown:       {metrics['max_drawdown_pct']:.2f}%  "
          f"({'✅ محتمل' if metrics['max_drawdown_pct'] > -20 else '⚠️ مرتفع'})")
    print(f"   • Profit Factor:      {metrics['profit_factor']:.3f}  "
          f"({'✅ قوي' if metrics['profit_factor'] > 2 else '⚠️ مقبول' if metrics['profit_factor'] > 1 else '❌ خاسر'})")
    print(f"   • Omega Ratio:        {metrics['omega_ratio']:.3f}")
    print(f"   • SQN (Van Tharp):    {metrics['sqn']:.3f}  "
          f"({'✅ ممتاز' if metrics['sqn'] > 3 else '⚠️ قابل للتداول' if metrics['sqn'] > 2 else '❌ ضعيف'})")
    unit = "فترة (محفظة)" if metrics.get('sharpe_basis') == 'annualized' else "صفقة"
    print(f"   • إجمالي العائد:      {metrics['total_return_pct']:+.2f}% على {metrics['n_trades']} {unit}")


def simulate_equity_curve(returns_pct: np.ndarray, initial_capital: float = 10000.0,
                           risk_per_trade: float = 0.1) -> Dict[str, np.ndarray]:
    """
    يبني منحنى رأس المال (equity curve) خطوة بخطوة من سلسلة عوائد الصفقات،
    مع منحنى الـ drawdown المقابل — أساس رسم `plot_equity_curve`.
    """
    r = np.asarray(returns_pct, dtype=np.float64)
    # ✅ نفس قص -100% المطبّق أعلاه (compute_trading_performance_metrics) — عائد واحد
    # لا يمكن أن يقل عن -100% اقتصادياً (السعر لا يهبط تحت الصفر)، فلا معنى
    # لإطلاق قيمة أقل تُمرّر دون قص إلى دالة المحاكاة (وإن كانت `risk_per_trade`
    # الافتراضي يجعلها مأمونة رياضياً هنا دائماً لوحدها إلا في حالات `risk_per_trade`
    # مرتفعة جداً).
    r = np.maximum(r, -100.0)
    equity = np.empty(len(r) + 1)
    equity[0] = initial_capital
    for i, ret in enumerate(r):
        trade_amount = equity[i] * risk_per_trade
        equity[i + 1] = equity[i] + trade_amount * (ret / 100.0)

    running_max = np.maximum.accumulate(equity)
    drawdown_pct = (equity - running_max) / running_max * 100
    return {'equity': equity, 'drawdown_pct': drawdown_pct, 'running_max': running_max}


def _test_drawdown_never_exceeds_100pct():
    # صفقة واحدة بعائد أقل من -100% (رهان خاطئ في الاتجاه على أصل تحرّك
    # أكبر من 100%) يجب ألاّ تنتج 'max_drawdown'/`equity` منطقيين أبداً —
    # قبل الإصلاح كان `compute_trading_performance_metrics` يُنتج -138% وequity
    # سالبة من صفقة واحدة فقط بهذا السيناريو.
    rng = np.random.default_rng(1)
    r = rng.normal(0, 5, 200)
    r[50] = -150.0
    metrics = compute_trading_performance_metrics(r)
    assert metrics['max_drawdown_pct'] >= -100.0, (
        f"max_drawdown_pct تجاوز -100% ({metrics['max_drawdown_pct']}) — صفقة <-100% لم تُقصَّ.")
    eq = simulate_equity_curve(r)
    assert eq['drawdown_pct'].min() >= -100.0, (
        f"drawdown_pct تجاوز -100% ({eq['drawdown_pct'].min()}) في simulate_equity_curve.")
    assert np.all(eq['equity'] > 0), "equity أصبحت غير موجبة بسبب صفقة أقل من -100% غير مقصوصة."
    print("✅ compute_trading_performance_metrics/simulate_equity_curve: drawdown/equity مُقيّدان بحدود منطقية (≥-100%) حتى مع صفقة بعائد أقل من -100%.")
    return True


_test_drawdown_never_exceeds_100pct()


## 1️⃣4️⃣ 📊 الرسوم البيانية (Visual Reports)

كل دالة هنا تعرض الرسم inline **وتحفظه** تلقائياً كـ PNG في `analysis_outputs/figures/`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 📊 الرسوم البيانية (Visual Reports)
# ═══════════════════════════════════════════════════════════════════════════
#
# كل دالة رسم هنا: تعرض الرسم داخل الـ notebook مباشرة (plt.show) + تحفظه
# تلقائياً كملف PNG في analysis_outputs/figures/ (يفيد عند إنتاج عدد كبير
# من الرسوم دفعة واحدة عبر عدة عملات/أهداف، بدل تكديسها كلها في المخرجات).
#
# ملاحظة: عناوين/تسميات المحاور بالإنجليزية عمداً — matplotlib لا يدعم تشكيل
# الحروف العربية (RTL shaping) افتراضياً بدون مكتبات إضافية (arabic_reshaper).
# ═══════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


def plot_calibration_curve(reliability_df: pd.DataFrame, group_val=None, group_col: str = 'target',
                            title: Optional[str] = None, save_dir: str = DEFAULT_OUTPUT_DIR) -> Optional[str]:
    """مخطط المعايرة (Reliability Diagram): الثقة المتوقعة مقابل الدقة الفعلية."""
    df = reliability_df if group_val is None else reliability_df[reliability_df[group_col] == group_val]
    if df.empty:
        return None
    df = df.sort_values('avg_confidence')

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Calibration')
    ax.plot(df['avg_confidence'], df['avg_accuracy'], 'o-', color='steelblue', label='Model')
    for _, row in df.iterrows():
        ax.annotate(f"n={int(row['count'])}", (row['avg_confidence'], row['avg_accuracy']),
                    fontsize=7, alpha=0.6, xytext=(3, 3), textcoords='offset points')
    ax.set_xlabel('Predicted Confidence')
    ax.set_ylabel('Actual Accuracy')
    ax.set_title(title or f"Calibration Curve{f' — {group_val}' if group_val else ''}")
    ax.legend()
    ax.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    path = save_figure(fig, f"calibration_curve_{group_val or 'overall'}", save_dir)
    plt.show()
    plt.close(fig)
    return path


def plot_confidence_vs_error(df: pd.DataFrame, target: Optional[str] = None,
                              error_col: str = 'abs_error', save_dir: str = DEFAULT_OUTPUT_DIR) -> Optional[str]:
    """علاقة الثقة بالخطأ — يجب أن تظهر علاقة عكسية إن كان النموذج معايَراً جيداً."""
    work = df if target is None else df[df['target'] == target]
    work = work.dropna(subset=['confidence', error_col])
    if work.empty:
        return None

    fig, ax = plt.subplots(figsize=(7, 5))
    correct_mask = work['correct'].astype(bool) if 'correct' in work.columns else pd.Series(True, index=work.index)
    ax.scatter(work.loc[correct_mask, 'confidence'], work.loc[correct_mask, error_col],
               s=10, alpha=0.4, color='seagreen', label='Correct')
    ax.scatter(work.loc[~correct_mask, 'confidence'], work.loc[~correct_mask, error_col],
               s=10, alpha=0.4, color='crimson', label='Wrong')
    if work['confidence'].std() > 1e-9:
        z = np.polyfit(work['confidence'], work[error_col], 1)
        xs = np.linspace(work['confidence'].min(), work['confidence'].max(), 50)
        ax.plot(xs, np.polyval(z, xs), 'k--', alpha=0.7, label='Trend')
    ax.set_xlabel('Confidence')
    ax.set_ylabel(error_col)
    ax.set_title(f"Confidence vs Error{f' — {target}' if target else ''}")
    ax.legend()
    ax.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    path = save_figure(fig, f"confidence_vs_error_{target or 'all'}", save_dir)
    plt.show()
    plt.close(fig)
    return path


def plot_error_distribution(df: pd.DataFrame, target: Optional[str] = None,
                             error_col: str = 'pct_error', save_dir: str = DEFAULT_OUTPUT_DIR) -> Optional[str]:
    """توزيع الأخطاء — يكشف عن ذيول ثقيلة (Fat tails) أو انحياز في التوزيع."""
    work = df if target is None else df[df['target'] == target]
    vals = work[error_col].dropna()
    if vals.empty:
        return None

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(vals, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
    ax.axvline(vals.mean(), color='crimson', ls='--', label=f'Mean={vals.mean():.3f}')
    ax.axvline(vals.median(), color='orange', ls='--', label=f'Median={vals.median():.3f}')
    ax.set_xlabel(error_col)
    ax.set_ylabel('Count')
    ax.set_title(f"Error Distribution{f' — {target}' if target else ''}")
    ax.legend()
    ax.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    path = save_figure(fig, f"error_distribution_{target or 'all'}", save_dir)
    plt.show()
    plt.close(fig)
    return path


def plot_actual_vs_predicted(y_actual: np.ndarray, y_pred: np.ndarray, y_naive: Optional[np.ndarray] = None,
                              title: str = "Actual vs Predicted", save_dir: str = DEFAULT_OUTPUT_DIR,
                              filename: Optional[str] = None) -> str:
    """منحنى القيم الفعلية مقابل المتوقعة عبر الزمن (+ naive baseline اختياري)."""
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(y_actual, c="limegreen", lw=2.2, alpha=0.85, label="Actual", zorder=5)
    ax.plot(y_pred, c="darkorange", lw=1.8, label="Predicted", alpha=0.9)
    if y_naive is not None:
        ax.plot(y_naive, c="gray", lw=1.3, ls="--", label="Naive Baseline (t-1)", alpha=0.7)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Time Steps")
    ax.set_ylabel("Value")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3, linestyle="--")
    plt.tight_layout()
    path = save_figure(fig, filename or f"actual_vs_predicted_{title}".replace(' ', '_'), save_dir)
    plt.show()
    plt.close(fig)
    return path


def plot_lag_correlation(lags: List[int], corrs: List[float], best_lag: int,
                          title: str = "Cross-Correlation vs Lag", save_dir: str = DEFAULT_OUTPUT_DIR,
                          filename: Optional[str] = None) -> str:
    """أعلى ارتباط عند أي إزاحة زمنية؟ (0 = صحي، غير 0 = نسخ متأخر مشبوه)."""
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ["crimson" if l == best_lag else "steelblue" for l in lags]
    ax.bar(lags, corrs, color=colors)
    ax.axvline(0, color="black", lw=1, ls=":")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Lag (steps)")
    ax.set_ylabel("Correlation")
    ax.grid(True, alpha=0.3, linestyle="--")
    plt.tight_layout()
    path = save_figure(fig, filename or f"lag_correlation_{title}".replace(' ', '_'), save_dir)
    plt.show()
    plt.close(fig)
    return path


def plot_confusion_matrix_heatmap(cm: np.ndarray, class_names: List[str], title: str = "Confusion Matrix",
                                   save_dir: str = DEFAULT_OUTPUT_DIR, filename: Optional[str] = None) -> str:
    """خريطة حرارية لمصفوفة الارتباك — للأهداف الفئوية."""
    fig, ax = plt.subplots(figsize=(1.2 * len(class_names) + 2, 1.2 * len(class_names) + 2))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title, fontweight="bold")
    thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > thresh else 'black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    path = save_figure(fig, filename or f"confusion_matrix_{title}".replace(' ', '_'), save_dir)
    plt.show()
    plt.close(fig)
    return path


def plot_feature_importance(importance_df: pd.DataFrame, top_n: int = 15,
                             save_dir: str = DEFAULT_OUTPUT_DIR) -> str:
    """أهمية الخصائص في التمييز بين النجاح والفشل (من detect_success_failure_patterns)."""
    top = importance_df.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(top))))
    ax.barh(top['الخاصية'], top['الأهمية'], color='teal')
    ax.set_xlabel('Importance')
    ax.set_title('Feature Importance (Random Forest)', fontweight="bold")
    ax.grid(True, alpha=0.3, linestyle='--', axis='x')
    plt.tight_layout()
    path = save_figure(fig, "feature_importance", save_dir)
    plt.show()
    plt.close(fig)
    return path


def plot_clusters_2d(clustered_df: pd.DataFrame, feature_cols: List[str],
                      save_dir: str = DEFAULT_OUTPUT_DIR) -> str:
    """يُسقط الأنماط السلوكية المكتشفة (K-Means) على مستوى ثنائي الأبعاد عبر PCA."""
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    X = StandardScaler().fit_transform(clustered_df[feature_cols])
    coords = PCA(n_components=2, random_state=42).fit_transform(X)

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(coords[:, 0], coords[:, 1], c=clustered_df['cluster'],
                          cmap='tab10', s=15, alpha=0.6)
    ax.set_xlabel('PCA Component 1')
    ax.set_ylabel('PCA Component 2')
    ax.set_title('Behavioral Clusters (PCA Projection)', fontweight="bold")
    legend1 = ax.legend(*scatter.legend_elements(), title="Cluster", loc="best", fontsize=8)
    ax.add_artist(legend1)
    ax.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    path = save_figure(fig, "clusters_pca", save_dir)
    plt.show()
    plt.close(fig)
    return path


def plot_equity_curve(equity: np.ndarray, drawdown_pct: np.ndarray, title: str = "Equity Curve",
                       save_dir: str = DEFAULT_OUTPUT_DIR, filename: Optional[str] = None) -> str:
    """منحنى رأس المال + منحنى الـ Drawdown المقابل (نمط tearsheet قياسي)."""
    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
    axes[0].plot(equity, color='steelblue', lw=1.8)
    axes[0].set_title(title, fontweight="bold")
    axes[0].set_ylabel('Equity ($)')
    axes[0].grid(True, alpha=0.3, linestyle='--')

    axes[1].fill_between(range(len(drawdown_pct)), drawdown_pct, 0, color='crimson', alpha=0.5)
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].set_xlabel('Trade #')
    axes[1].grid(True, alpha=0.3, linestyle='--')

    plt.tight_layout()
    path = save_figure(fig, filename or f"equity_curve_{title}".replace(' ', '_'), save_dir)
    plt.show()
    plt.close(fig)
    return path


def plot_batch_performance(batch_stats: pd.DataFrame, z_threshold: float = 1.5,
                            title: str = "Performance Across Batches", save_dir: str = DEFAULT_OUTPUT_DIR) -> str:
    """نسبة النجاح عبر الدفعات المتتالية مع تمييز الدفعات الشاذة (اكتشاف الانحدار)."""
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = np.where(batch_stats['z_score'] <= -z_threshold, 'crimson',
              np.where(batch_stats['z_score'] >= z_threshold, 'seagreen', 'steelblue'))
    ax.bar(batch_stats['batch_id'], batch_stats['نسبة_نجاح'], color=colors)
    ax.axhline(batch_stats['نسبة_نجاح'].mean(), color='black', ls='--', alpha=0.6, label='Mean')
    ax.set_xlabel('Batch #')
    ax.set_ylabel('Win Rate (%)')
    ax.set_title(title, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3, linestyle='--', axis='y')
    plt.tight_layout()
    path = save_figure(fig, "batch_performance", save_dir)
    plt.show()
    plt.close(fig)
    return path


## 1️⃣5️⃣ 🩺 تشخيص نزاهة النموذج (Naive Baseline × Lag Scan × الانحياز)

مُدمَج من سكربتي التشخيص المرفقين: هل النموذج "يتنبأ" فعلاً أم يستفيد فقط من الترابط الذاتي للأسعار المتتالية (persistence)؟ لتفادي إغراق المخرجات، تُرسم المنحنيات فقط لأسوأ/أفضل التركيبات (الجدول الكامل محفوظ دوماً في ملف).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🩺 تشخيص نزاهة النموذج: Naive Baseline × Lag Scan × الانحياز الموجّه
# ═══════════════════════════════════════════════════════════════════════════
#
# مُدمَج من سكربتي التشخيص المرفقين، ومُكيَّف للعمل مباشرة فوق نتائج
# test_all_assets_v4 (بدل إعادة بناء test_dict يدوياً). يجيب على: هل النموذج
# "يتنبأ" فعلاً أم يعيد إنتاج آخر سعر معروف (persistence) مستفيداً من
# الترابط الذاتي الطبيعي في الأسعار المتتالية؟
# ═══════════════════════════════════════════════════════════════════════════

def compute_point_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if np.std(y_true) > 1e-9 and np.std(y_pred) > 1e-9 else 0.0
    return {"mae": mae, "rmse": rmse, "corr": corr}


def cross_correlation_lag_scan(y_actual: np.ndarray, y_pred: np.ndarray, max_lag: int = 5):
    """عند أي إزاحة زمنية يكون الارتباط بين actual و predicted أعلى؟ (0=صحي, ≠0=نسخ متأخر مشبوه)"""
    lags = list(range(-max_lag, max_lag + 1))
    correlations = []
    for lag in lags:
        if lag < 0:
            a, p = y_actual[:lag], y_pred[-lag:]
        elif lag > 0:
            a, p = y_actual[lag:], y_pred[:-lag]
        else:
            a, p = y_actual, y_pred
        correlations.append(float(np.corrcoef(a, p)[0, 1]) if len(a) > 1 and np.std(a) > 1e-9 and np.std(p) > 1e-9 else np.nan)
    valid = [c if not np.isnan(c) else -np.inf for c in correlations]
    best_lag = lags[int(np.argmax(valid))]
    return lags, correlations, best_lag


def momentum_direction_accuracy(y_actual: np.ndarray, entry_price: np.ndarray) -> float:
    """
    مرجع اتجاهي عادل (momentum): يتنبأ بأن الحركة القادمة تكرر آخر حركة فعلية.
    أعدل من persistence baseline الذي يعطي 0% اتجاهية بشكل مصطنع رياضياً
    (لأن حركته المتوقعة = صفر دائماً).
    """
    actual_dir = np.sign(y_actual - entry_price)
    if len(actual_dir) < 2:
        return float("nan")
    momentum_pred_dir = actual_dir[:-1]
    actual_dir_current = actual_dir[1:]
    mask = actual_dir_current != 0
    if mask.sum() == 0:
        return float("nan")
    return float(np.mean(momentum_pred_dir[mask] == actual_dir_current[mask]))


def directional_bias(y_actual: np.ndarray, y_pred: np.ndarray) -> Dict:
    """هل أخطاء النموذج منحازة بثبات في اتجاه واحد (متفائل/متشائم دائماً) أم عشوائية؟"""
    errors = y_pred - y_actual
    bias_mean = float(np.mean(errors))
    bias_pct = float(np.mean(errors / (np.abs(y_actual) + 1e-9)) * 100)
    label = "متفائل (يبالغ بالارتفاع)" if bias_mean > 0 else "متشائم (يبالغ بالانخفاض)" if bias_mean < 0 else "متوازن"
    ref_sign = np.sign(bias_mean) if bias_mean != 0 else 0
    same_sign_pct = float(np.mean(np.sign(errors) == ref_sign)) * 100 if ref_sign != 0 else float("nan")
    return {"bias_mean": bias_mean, "bias_pct": bias_pct, "same_sign_pct": same_sign_pct, "direction_label": label}


def run_integrity_diagnostics(
    per_asset_results: pd.DataFrame,
    target_specs: List['TargetSpec'],
    max_lag: int = 5,
    plot_worst_n: int = 2,
    plot_best_n: int = 1,
    out_dir: str = DEFAULT_OUTPUT_DIR,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    يشغّل تشخيص Naive-Baseline/Lag/Bias على كل (عملة × هدف مستمر)، ويُرجع
    جدول ملخص واحد. لتفادي إغراق المخرجات، تُرسم المنحنيات (actual vs
    predicted + lag correlation) فقط لأسوأ `plot_worst_n` وأفضل `plot_best_n`
    تركيبة (كل الرسوم تُحفظ في analysis_outputs/figures/ مع ذلك).
    """
    target_specs = resolve_targets(target_specs)
    rows = []

    for _, row in per_asset_results.iterrows():
        asset = row['asset']
        for spec in target_specs:
            if spec.kind != 'continuous' or spec.name not in row or not isinstance(row[spec.name], dict):
                continue
            d = row[spec.name]
            if 'true_real' not in d:
                continue

            y_actual = np.asarray(d['true_real'], dtype=np.float64)
            y_pred = np.asarray(d['pred_real'], dtype=np.float64)
            entry = np.asarray(d['entry_price'], dtype=np.float64)
            n = len(y_actual)
            if n < 5:
                continue

            y_naive = entry  # الـ naive baseline = "لا تغيير عن آخر سعر معروف"
            model_m = compute_point_metrics(y_actual, y_pred)
            naive_m = compute_point_metrics(y_actual, y_naive)
            mae_improve = (naive_m['mae'] - model_m['mae']) / (naive_m['mae'] + 1e-12) * 100

            dir_acc_model = float(np.mean(np.sign(y_pred - entry)[np.sign(y_actual - entry) != 0] ==
                                           np.sign(y_actual - entry)[np.sign(y_actual - entry) != 0])) \
                if np.any(np.sign(y_actual - entry) != 0) else float('nan')
            dir_acc_momentum = momentum_direction_accuracy(y_actual, entry)
            bias = directional_bias(y_actual, y_pred)

            safe_max_lag = max(1, min(max_lag, n // 3))
            lags, lag_corrs, best_lag = cross_correlation_lag_scan(y_actual, y_pred, safe_max_lag)

            rows.append({
                'asset': asset, 'target': spec.name, 'n': n,
                'model_mae': model_m['mae'], 'naive_mae': naive_m['mae'], 'mae_improvement_%': mae_improve,
                'model_corr': model_m['corr'], 'naive_corr': naive_m['corr'],
                'dir_acc_model_%': dir_acc_model * 100 if not np.isnan(dir_acc_model) else np.nan,
                'dir_acc_momentum_%': dir_acc_momentum * 100 if not np.isnan(dir_acc_momentum) else np.nan,
                'best_lag': best_lag, 'bias_pct': bias['bias_pct'],
                'bias_same_sign_%': bias['same_sign_pct'], 'bias_label': bias['direction_label'],
                '_y_actual': y_actual, '_y_pred': y_pred, '_y_naive': y_naive,
                '_lags': lags, '_lag_corrs': lag_corrs,
            })

    if not rows:
        if verbose:
            print("⚠️ لا توجد أهداف مستمرة كافية لتشخيص النزاهة")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    healthy = df[
        (df['best_lag'] == 0)
        & (df['mae_improvement_%'] > 10)
        & (df['dir_acc_model_%'] > np.maximum(55, df['dir_acc_momentum_%'] + 3))
    ]

    display_df = df.drop(columns=['_y_actual', '_y_pred', '_y_naive', '_lags', '_lag_corrs'])

    if verbose:
        print("=" * 100)
        print("🩺 تشخيص نزاهة النموذج (Naive Baseline × Lag Scan × الانحياز الموجّه)")
        print("=" * 100)
        save_or_print(display_df.round(3), 'integrity_diagnostics_summary', out_dir=out_dir)

        print(f"\n✅ عدد تركيبات (عملة/هدف) اجتازت كل الفحوصات: {len(healthy)} / {len(df)}")
        if len(healthy) < len(df):
            flagged = display_df[~display_df.index.isin(healthy.index)]
            print("⚠️ تركيبات تحتاج مراجعة (قد يكون النموذج يعتمد على الترابط الذاتي بدل تنبؤ حقيقي):")
            save_or_print(
                flagged[['asset', 'target', 'mae_improvement_%', 'dir_acc_model_%', 'dir_acc_momentum_%', 'best_lag']].round(2),
                'integrity_flagged', out_dir=out_dir,
            )

        # رسم أسوأ/أفضل التركيبات فقط لتفادي إغراق المخرجات
        ranked = df.sort_values('mae_improvement_%')
        # ⚠️ لا نستخدم drop_duplicates() هنا: الإطار يحوي أعمدة مصفوفات (_y_actual/_y_pred/...)
        #    وهي غير قابلة للتجزئة (TypeError: unhashable type: 'numpy.ndarray').
        #    نُزيل التكرار بالفهرس فقط (يحافظ على الترتيب: الأسوأ ثم الأفضل).
        _plot_idx = list(dict.fromkeys(list(ranked.head(plot_worst_n).index) + list(ranked.tail(plot_best_n).index)))
        to_plot = ranked.loc[_plot_idx]
        if len(to_plot) > 0:
            print(f"\n📊 رسوم بيانية لأسوأ {plot_worst_n} وأفضل {plot_best_n} تركيبة (الباقي محفوظ في الجدول أعلاه فقط):")
        for _, r in to_plot.iterrows():
            label = f"{r['asset']} — {r['target']}"
            plot_actual_vs_predicted(r['_y_actual'], r['_y_pred'], r['_y_naive'],
                                      title=f"{label} (MAE improve: {r['mae_improvement_%']:.1f}%)",
                                      save_dir=out_dir, filename=f"integrity_{r['asset']}_{r['target']}_actual_vs_pred")
            plot_lag_correlation(r['_lags'], r['_lag_corrs'], r['best_lag'],
                                  title=f"Lag Scan — {label}", save_dir=out_dir,
                                  filename=f"integrity_{r['asset']}_{r['target']}_lag_scan")

    return display_df


## 1️⃣6️⃣ 🚀 الدالة الرئيسية الموحّدة v2: كل التحليلات + الرسوم + إدارة المخرجات

`run_full_analysis` تُشغّل خط الأنابيب كاملاً ثم كل التقارير والرسوم أعلاه في استدعاء واحد، وتحفظ كل مخرج كبير في `analysis_outputs/`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🚀 الدالة الرئيسية الموحّدة v2: كل التحليلات + الرسوم + إدارة المخرجات
# ═══════════════════════════════════════════════════════════════════════════

def run_full_analysis(
    model,
    test_dict: Dict,
    timeframes: List[str],
    target_specs: List['TargetSpec'],
    batch_size: int = 256,
    range_frac: float = 0.2,
    batch_analysis_size: int = 100,
    out_dir: str = DEFAULT_OUTPUT_DIR,
    make_plots: bool = True,
    run_integrity_check: bool = True,
    verbose: bool = True,
    n_display: int = 1,
    timestamp_col: Optional[int] = None,
    timestamp_key: Optional[str] = None,
    run_trade_selection: bool = True,
    trade_min_edge_ratio: float = 1.0,
    trade_sort_by: str = 'both',
    trade_top_n: Optional[int] = 20,
    trade_cost_pct: float = 0.08,
) -> Dict:
    """
    خط الأنابيب الكامل: تنبؤ + فك تشفير مرن + تحقق + تقييم، ثم كل التقارير
    (ثقة/فهم، مخرجات موجّهة، اكتشاف أنماط، تشخيص نزاهة) + الرسوم البيانية،
    مع حفظ أي مخرج كبير في `out_dir` بدل إغراق الـ notebook.

    استخدام سريع:
        full = run_full_analysis(model, test_dict, ['1h','4h','1D'], 'close')   # أو ['high','low'] أو specs
    """
    ensure_output_dir(out_dir)

    test_results = test_all_assets_v4(
        model, test_dict, timeframes, target_specs,
        verbose=verbose, batch_size=batch_size, range_frac=range_frac,
        n_display=n_display, timestamp_col=timestamp_col, timestamp_key=timestamp_key,
    )
    target_specs = test_results.get('target_specs', target_specs)   # الأهداف بعد الحسم (نصوص → TargetSpec)
    per_asset = test_results['per_asset_results']
    if per_asset.empty:
        print("⚠️ لا توجد نتائج — تحقق من test_dict")
        return test_results

    flat_df = build_flat_dataframe(per_asset, target_specs)
    save_or_print(flat_df, 'flat_predictions_all', out_dir=out_dir, verbose=False)  # يُحفظ دوماً كأرشيف كامل

    # ── تقرير الثقة والفهم + مخطط المعايرة ──────────────────────────────────
    trust = generate_trust_report(flat_df, group_by='target', verbose=verbose)
    if make_plots and not trust['reliability_curve'].empty:
        for tgt in flat_df['target'].unique():
            plot_calibration_curve(trust['reliability_curve'], group_val=tgt, save_dir=out_dir)
        plot_confidence_vs_error(flat_df, save_dir=out_dir)

    # ── تحليل موجَّه للمخرجات ────────────────────────────────────────────────
    continuous_df = flat_df[flat_df['kind'] == 'continuous']
    movement_analysis = batch_analysis = None
    if not continuous_df.empty:
        movement_analysis = analyze_by_predicted_movement(continuous_df, verbose=verbose)
        batch_analysis = analyze_by_batch(continuous_df, batch_size=batch_analysis_size, verbose=verbose)
        if make_plots:
            plot_error_distribution(continuous_df, save_dir=out_dir)
            if batch_analysis is not None and not batch_analysis.empty:
                plot_batch_performance(batch_analysis, save_dir=out_dir)

    confidence_analysis = analyze_by_confidence_bucket(flat_df, verbose=verbose)

    # ── اختيار الصفقات: تحرك متوقع > عدم يقين + مقارنة معدل النجاح ────────────
    trade_candidates = None   # جميع الصفقات التي اجتازت Edge Filter قبل Top-N
    trade_selection = None    # الصفوف المعروضة/المحفوظة بعد Top-N والترتيب
    if run_trade_selection and not continuous_df.empty:
        trade_candidates = select_and_rank_trades(
            continuous_df,
            min_edge_ratio=trade_min_edge_ratio,
            sort_by=trade_sort_by,
            top_n=None,
        )
        trade_selection = (
            trade_candidates.head(int(trade_top_n)).copy()
            if trade_top_n is not None else trade_candidates.copy()
        )
        if trade_selection is not None and not trade_selection.empty:
            save_or_print(trade_selection, 'trade_selection', out_dir=out_dir, verbose=False)
        if verbose:
            print_trade_selection(
                trade_selection,
                original=continuous_df,
                title=(
                    f"\n🎯 أفضل الصفقات — |التحرك المتوقع| > "
                    f"{float(trade_min_edge_ratio):.2f}× عدم اليقين، "
                    f"ترتيب: {trade_sort_by}"
                ),
                max_rows=trade_top_n,
            )

            ok_col = next((c for c in ('direction_correct', 'correct', 'ok')
                           if c in continuous_df.columns and c in trade_candidates.columns), None)
            if ok_col and not trade_candidates.empty:
                filtered_acc, filtered_n = _trade_accuracy_pct(trade_candidates[ok_col])
                base_acc, base_n = _trade_accuracy_pct(continuous_df[ok_col])
                if base_n and filtered_n:
                    print(
                        f"  📊 Win Rate بعد Edge Filter (كل المؤهلين): {filtered_acc:.1f}% "
                        f"(n={filtered_n}) | قبل الفلترة: {base_acc:.1f}% (n={base_n})"
                    )

    # ── اكتشاف الأنماط آلياً ─────────────────────────────────────────────────
    patterns = None
    try:
        patterns = detect_success_failure_patterns(flat_df, verbose=verbose)
    except ValueError as e:
        if verbose:
            print(f"\n⚠️ تخطي اكتشاف الأنماط: {e}")

    # ── تشخيص النزاهة (Naive Baseline × Lag × الانحياز) ─────────────────────
    integrity_df = None
    if run_integrity_check:
        integrity_df = run_integrity_diagnostics(per_asset, target_specs, out_dir=out_dir, verbose=verbose)

    # ── مقاييس أداء تداول إضافية (Sharpe/Sortino/Profit Factor/SQN) ─────────
    trading_metrics = None
    if 'close' in continuous_df['target'].unique() if not continuous_df.empty else False:
        close_df = continuous_df[continuous_df['target'] == 'close'].copy()
        # العائد الفعلي للاستراتيجية: إن كان التوقع "شراء" (pred>entry) نربح true_change،
        # وإن كان "بيع" (pred<entry) نربح عكس true_change
        true_change_pct = ((close_df['true'] - close_df['entry']) / (np.abs(close_df['entry']) + 1e-7)) * 100
        direction = np.sign(close_df['predicted_change'])
        strategy_returns = direction * true_change_pct - trade_cost_pct   # بعد تكلفة الذهاب والإياب

        ts = close_df['timestamp'] if 'timestamp' in close_df else pd.Series(np.nan, index=close_df.index)
        if ts.notna().all() and ts.nunique() > 2:
            # ✅ محفظة متساوية الأوزان لكل فترة: صفقات كل العملات في نفس اللحظة تُنفَّذ معاً برأس مال
            # مقسوم بالتساوي، فعائد الفترة = متوسط عوائدها. سابقاً كانت كل صفقة تُركَّب على رأس المال
            # بعد سابقتها حتى لو كانت في نفس اليوم (ومرتَّبة حسب العملة لا الزمن) — منحنى مستحيل التنفيذ.
            period_returns = strategy_returns.groupby(ts.values).mean().sort_index()
            steps = np.diff(period_returns.index.values.astype(np.float64))
            step_ns = float(np.median(steps)) if len(steps) else 0.0
            # طوابع بالنانوثانية (last_candles). فاصل < دقيقة = ليست أوقاتاً حقيقية (مثلاً أرقام تسلسلية) — بلا تسنين.
            periods_per_year = (365.25 * 86400e9 / step_ns) if step_ns >= 60e9 else None
            trading_metrics = compute_trading_performance_metrics(period_returns.values,
                                                                  periods_per_year=periods_per_year)
            plot_returns, risk = period_returns.values, 1.0
            title = f"Equity Curve — Close (equal-weight portfolio per period, cost {trade_cost_pct}%)"
            basis = f"محفظة متساوية الأوزان: {len(period_returns)} فترة، {len(close_df)} صفقة، بعد تكلفة {trade_cost_pct}%"
        else:
            trading_metrics = compute_trading_performance_metrics(strategy_returns.values)
            plot_returns, risk = strategy_returns.values, 0.1
            title = "Equity Curve — Close (per trade, NO timestamps: sequential, not executable)"
            basis = "⚠️ بلا طوابع زمنية: صفقات متتالية — ليس منحنى قابلاً للتنفيذ"

        if verbose and trading_metrics:
            print("\n" + "=" * 100)
            print("📈 مقاييس أداء تداول إضافية (Tearsheet-style) — هدف Close، كل العملات مجمّعة")
            print(f"   ({basis})")
            print("=" * 100)
            print_performance_verdict(trading_metrics)

        if make_plots and trading_metrics:
            eq = simulate_equity_curve(plot_returns, risk_per_trade=risk)
            plot_equity_curve(eq['equity'], eq['drawdown_pct'], title=title, save_dir=out_dir)

    if verbose:
        print("\n" + "=" * 100)
        print("✅ اكتمل التحليل الشامل — الملخص التنفيذي")
        print("=" * 100)
        overall_trust = trust['overall'].get('trust_score_0_100')
        overall_acc = flat_df['correct'].mean() * 100
        print(f"   • إجمالي العينات المحللة: {len(flat_df):,}")
        print(f"   • الدقة/نسبة النجاح العامة: {overall_acc:.2f}%")
        if trade_selection is not None:
            print(f"   • الصفقات المؤهلة بعد Edge Filter: {len(trade_selection):,} صفقة")
        if overall_trust is not None:
            print(f"   • درجة الثقة والفهم (Trust Score): {overall_trust:.1f}/100")
        if not test_results['verification_summary'].empty:
            failed = (test_results['verification_summary']['status'] == 'FAILED').sum()
            print(f"   • حالة التحقق من فك التشفير: {'✅ سليم بالكامل' if failed == 0 else f'❌ {failed} فشل'}")
        if integrity_df is not None and not integrity_df.empty:
            healthy_pct = (integrity_df['mae_improvement_%'] > 10).mean() * 100
            print(f"   • نسبة التركيبات (عملة/هدف) التي تتفوق فعلياً على Naive Baseline: {healthy_pct:.0f}%")
        if trading_metrics:
            print(f"   • Sharpe Ratio (Close، {'سنوي' if trading_metrics.get('sharpe_basis') == 'annualized' else 'لكل صفقة'}): {trading_metrics['sharpe_ratio']:.2f} | "
                  f"Max Drawdown: {trading_metrics['max_drawdown_pct']:.1f}%")
        print(f"\n📁 كل الجداول/الرسوم الكبيرة محفوظة في: {os.path.abspath(out_dir)}")

    return {
        **test_results,
        'flat_df': flat_df,
        'trust_report': trust,
        'movement_analysis': movement_analysis,
        'confidence_analysis': confidence_analysis,
        'trade_candidates': trade_candidates,
        'trade_selection': trade_selection,
        'batch_analysis': batch_analysis,
        'pattern_discovery': patterns,
        'integrity_diagnostics': integrity_df,
        'trading_metrics': trading_metrics,
    }


# ══════════════════════════════════════════════════════════════════════════
# 🚀 مثال استخدام
# ══════════════════════════════════════════════════════════════════════════
#
# specs = DEFAULT_PRICE_TARGETS  # أو + [make_categorical_spec(...)] لأهداف فئوية
# full_results = run_full_analysis(
#     model=model,
#     test_dict=test_dict,
#     timeframes=['1h', '4h', '1D'],
#     target_specs=specs,
#     out_dir='analysis_outputs',   # كل الجداول/الرسوم الكبيرة تُحفظ هنا
# )
#
# # الوصول للتقارير الفردية:
# full_results['trust_report']['by_group']         # جدول الثقة حسب كل هدف
# full_results['movement_analysis']                 # نجاح التوقع حسب نوع/حجم الحركة
# full_results['batch_analysis']                    # اكتشاف الانحدار عبر الدفعات
# full_results['integrity_diagnostics']              # Naive baseline / lag / bias
# full_results['trading_metrics']                    # Sharpe/Sortino/Profit Factor/SQN
# full_results['trade_selection']                    # الصفقات بعد Edge Filter + الترتيب
# full_results['pattern_discovery']['tree_rules']    # قواعد نجاح/فشل مقروءة
# full_results['pattern_discovery']['clusters']       # الأنماط السلوكية المكتشفة


## 1️⃣7️⃣ 📚 التقارير القديمة (محفوظة ومحدَّثة لاستخدام save_or_print)

`comprehensive_uncertainty_analysis` من الدفتر الأصلي — نفس المنطق تماماً، فقط الجداول الكبيرة (تحليل الاتجاه، أداء كل عملة، عتبات الثقة، إحصائيات المعايرة) تُحفظ الآن في ملف بدل طباعتها كاملة.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import confusion_matrix, classification_report

def comprehensive_uncertainty_analysis(per_asset_results):
    """
    ✅ تحليل شامل مع إصلاح مشكلة float16 وإضافة تحليل مفصل لاتجاه الإغلاق
    """
    print("="*100)
    print("🔬 تحليل شامل: الثقة × عدم اليقين × الأداء × اتجاه الإغلاق")
    print("="*100)

    all_data = []

    # ═══════════════════════════════════════════════════════════════════════
    # 1️⃣ جمع البيانات
    # ═══════════════════════════════════════════════════════════════════════

    for idx, row in per_asset_results.iterrows():
        asset = row['asset']

        for target in ['high', 'low', 'close']:
            data = row.get(target)
            if not isinstance(data, dict) or 'true_real' not in data:
                continue   # هدف غير مطلوب/غير متاح لهذا الأصل

            # تحويل لـ float32 فوراً
            pred = np.array(data['pred_real'], dtype=np.float32)
            true = np.array(data['true_real'], dtype=np.float32)
            entry = np.array(data['entry_price'], dtype=np.float32)
            uncertainty = np.array(data['uncertainty_real'], dtype=np.float32)
            confidence = np.array(data['confidence'], dtype=np.float32)

            # Aleatoric & Epistemic
            aleatoric = np.array(data.get('aleatoric_real', [0] * len(pred)), dtype=np.float32)
            epistemic = np.array(data.get('epistemic_real', [0] * len(pred)), dtype=np.float32)

            # حساب الأخطاء
            abs_error = np.abs(pred - true).astype(np.float32)
            rel_error = (abs_error / (np.abs(true) + 1e-10)).astype(np.float32)
            pct_error = (rel_error * 100).astype(np.float32)

            # الاتجاه - مقارنة مع سعر الدخول
            pred_direction = np.sign(pred - entry)
            true_direction = np.sign(true - entry)
            direction_correct = (pred_direction == true_direction).astype(np.int32)

            # نوع الخطأ في الاتجاه
            direction_error_type = np.where(
                (pred_direction == 1) & (true_direction == -1), 'صعود خاطئ',
                np.where(
                    (pred_direction == -1) & (true_direction == 1), 'هبوط خاطئ',
                    'صحيح'
                )
            )

            # تجميع
            for i in range(len(pred)):
                all_data.append({
                    'asset': asset,
                    'target': target,
                    'pred': float(pred[i]),
                    'true': float(true[i]),
                    'entry': float(entry[i]),
                    'abs_error': float(abs_error[i]),
                    'rel_error': float(rel_error[i]),
                    'pct_error': float(pct_error[i]),
                    'uncertainty': float(uncertainty[i]),
                    'confidence': float(confidence[i]),
                    'aleatoric': float(aleatoric[i]) if len(aleatoric) > 0 else 0.0,
                    'epistemic': float(epistemic[i]) if len(epistemic) > 0 else 0.0,
                    'pred_direction': 'صعود' if pred_direction[i] > 0 else 'هبوط' if pred_direction[i] < 0 else 'ثابت',
                    'true_direction': 'صعود' if true_direction[i] > 0 else 'هبوط' if true_direction[i] < 0 else 'ثابت',
                    'direction_correct': int(direction_correct[i]),
                    'direction_error_type': direction_error_type[i],
                    'price_change': float(true[i] - entry[i]),
                    'predicted_change': float(pred[i] - entry[i])
                })

    df = pd.DataFrame(all_data)

    # ✅ تأكيد أن كل الأعمدة float64 (ليس float16)
    numeric_cols = ['pred', 'true', 'entry', 'abs_error', 'rel_error', 'pct_error',
                    'uncertainty', 'confidence', 'aleatoric', 'epistemic',
                    'price_change', 'predicted_change']
    for col in numeric_cols:
        df[col] = df[col].astype(np.float64)

    # ═══════════════════════════════════════════════════════════════════════
    # تحليل مفصل لاتجاه الإغلاق (CLOSE فقط)
    # ═══════════════════════════════════════════════════════════════════════

    print("\n" + "="*100)
    print("🎯 تحليل مفصل لاتجاه الإغلاق (CLOSE)")
    print("="*100)

    # فلترة البيانات للهدف close فقط
    df_close = df[df['target'] == 'close'].copy()

    if len(df_close) > 0:
        print(f"\n📊 إجمالي عدد توقعات الإغلاق: {len(df_close)}")

        # 1. تحليل حسب نوع التوقع
        print("\n📈 1. تحليل أداء التوقعات حسب الاتجاه المتوقع:")
        print("-"*60)

        direction_analysis = []
        for pred_dir in ['صعود', 'هبوط']:
            df_dir = df_close[df_close['pred_direction'] == pred_dir]
            if len(df_dir) > 0:
                correct = df_dir['direction_correct'].sum()
                total = len(df_dir)
                accuracy = (correct / total * 100) if total > 0 else 0

                # متوسط التغير الفعلي
                avg_actual_change = df_dir['price_change'].mean()
                avg_pred_change = df_dir['predicted_change'].mean()

                direction_analysis.append({
                    'الاتجاه المتوقع': pred_dir,
                    'عدد التوقعات': total,
                    'توقعات صحيحة': correct,
                    'دقة التوقع %': f"{accuracy:.2f}%",
                    'متوسط التغير المتوقع': f"{avg_pred_change:.4f}",
                    'متوسط التغير الفعلي': f"{avg_actual_change:.4f}",
                    'الفرق': f"{avg_pred_change - avg_actual_change:.4f}"
                })

        dir_df = pd.DataFrame(direction_analysis)
        save_or_print(dir_df, 'legacy_close_direction_analysis')

        # 2. مصفوفة الارتباك (Confusion Matrix)
        print("\n📊 2. مصفوفة الارتباك (Confusion Matrix):")
        print("-"*60)

        # إنشاء مصفوفة الارتباك
        confusion_data = []
        for pred_dir in ['صعود', 'هبوط']:
            for true_dir in ['صعود', 'هبوط']:
                count = len(df_close[(df_close['pred_direction'] == pred_dir) &
                                    (df_close['true_direction'] == true_dir)])
                confusion_data.append({
                    'المتوقع': pred_dir,
                    'الفعلي': true_dir,
                    'العدد': count,
                    'النسبة %': f"{(count/len(df_close)*100):.1f}%"
                })

        confusion_df = pd.DataFrame(confusion_data)

        # عرض بصيغة جدول
        pivot_table = confusion_df.pivot_table(
            values='العدد',
            index='المتوقع',
            columns='الفعلي',
            aggfunc='first'
        ).fillna(0)

        print("مصفوفة الارتباك:")
        print(pivot_table)

        # 3. تحليل حسب العملة
        print("\n📊 3. أداء توقعات الإغلاق حسب العملة:")
        print("-"*60)

        asset_performance = []
        for asset in df_close['asset'].unique():
            df_asset = df_close[df_close['asset'] == asset]

            for pred_dir in ['صعود', 'هبوط']:
                df_asset_dir = df_asset[df_asset['pred_direction'] == pred_dir]
                if len(df_asset_dir) > 0:
                    correct = df_asset_dir['direction_correct'].sum()
                    total = len(df_asset_dir)
                    accuracy = (correct / total * 100) if total > 0 else 0

                    asset_performance.append({
                        'العملة': asset,
                        'الاتجاه': pred_dir,
                        'عدد': total,
                        'صحيح': correct,
                        'دقة %': f"{accuracy:.2f}%",
                        'متوسط تغير فعلي': f"{df_asset_dir['price_change'].mean():.4f}"
                    })

        perf_df = pd.DataFrame(asset_performance)
        if len(perf_df) > 0:
            save_or_print(perf_df, 'legacy_asset_close_performance')

        # 4. تحليل التوقعات الخاطئة
        print("\n📊 4. تحليل التوقعات الخاطئة:")
        print("-"*60)

        wrong_predictions = df_close[df_close['direction_correct'] == 0]
        if len(wrong_predictions) > 0:
            error_analysis = wrong_predictions.groupby('direction_error_type').agg({
                'asset': 'count',
                'confidence': 'mean',
                'uncertainty': 'mean',
                'abs_error': 'mean',
                'price_change': ['min', 'max', 'mean']
            }).round(4)

            error_analysis.columns = ['عدد', 'متوسط ثقة', 'متوسط عدم يقين',
                                     'متوسط خطأ', 'أقل تغير', 'أعلى تغير', 'متوسط تغير']
            print(error_analysis)

            print(f"\n📌 ملاحظات على الأخطاء:")
            print(f"   • إجمالي التوقعات الخاطئة: {len(wrong_predictions)}")
            print(f"   • نسبة الأخطاء: {(len(wrong_predictions)/len(df_close)*100):.1f}%")

            # تحليل الثقة في التوقعات الخاطئة vs الصحيحة
            correct_conf = df_close[df_close['direction_correct'] == 1]['confidence'].mean()
            wrong_conf = df_close[df_close['direction_correct'] == 0]['confidence'].mean()
            print(f"   • متوسط الثقة في التوقعات الصحيحة: {correct_conf:.3f}")
            print(f"   • متوسط الثقة في التوقعات الخاطئة: {wrong_conf:.3f}")
            print(f"   • الفرق في الثقة: {correct_conf - wrong_conf:.3f}")

        # 5. تحسين الأداء
        print("\n📊 5. اقتراحات لتحسين الأداء:")
        print("-"*60)

        # حساب عتبة الثقة المثلى
        thresholds = [ 0.0,0.1,0.2,0.3,0.4,0.5, 0.6, 0.7, 0.8, 0.9]
        improvement_data = []

        for threshold in thresholds:
            high_conf = df_close[(df_close['confidence'] >= threshold)&(df_close['confidence'] <= (threshold+0.1))]
            if len(high_conf) > 0:
                high_conf_accuracy = high_conf['direction_correct'].mean() * 100
                coverage = len(high_conf) / len(df_close) * 100
                improvement_data.append({
                    'عتبة الثقة': threshold,
                    'دقة عالية الثقة %': f"{high_conf_accuracy:.1f}%",
                    'تغطية %': f"{coverage:.1f}%",
                    'عدد': len(high_conf)
                })

        if improvement_data:
            improve_df = pd.DataFrame(improvement_data)
            print("أداء التوقعات عالية الثقة:")
            save_or_print(improve_df, 'legacy_confidence_threshold_performance')

            # اقتراح أفضل عتبة
            best_threshold = max(improvement_data,
                               key=lambda x: float(x['دقة عالية الثقة %'].replace('%', '')))
            print(f"\n✅ أفضل عتبة: {best_threshold['عتبة الثقة']} → "
                  f"دقة: {best_threshold['دقة عالية الثقة %']} "
                  f"(تغطية: {best_threshold['تغطية %']})")

    else:
        print("⚠️ لا توجد بيانات للإغلاق (close) للتحليل")

    # ═══════════════════════════════════════════════════════════════════════
    # 2️⃣ Confidence-Error Correlation (نفس الكود السابق)
    # ═══════════════════════════════════════════════════════════════════════

    print("\n" + "─"*100)
    print("📈 6. Confidence-Error Correlation (CEC)")
    print("─"*100)

    corr_conf_abs, p_abs = pearsonr(df['confidence'], df['abs_error'])
    corr_conf_rel, p_rel = pearsonr(df['confidence'], df['rel_error'])

    print(f"  📊 الارتباط الكلي:")
    print(f"     Confidence ↔ Absolute Error: {corr_conf_abs:+.4f} (p={p_abs:.4e})")
    print(f"     Confidence ↔ Relative Error: {corr_conf_rel:+.4f} (p={p_rel:.4e})")

    if corr_conf_abs < -0.3:
        print(f"     ✅ ممتاز: الثقة العالية = خطأ منخفض")
    elif corr_conf_abs < 0:
        print(f"     ⚠️  ضعيف: ارتباط سلبي لكن ضعيف")
    else:
        print(f"     ❌ مشكلة: الثقة لا ترتبط بالدقة!")

    # ═══════════════════════════════════════════════════════════════════════
    # 3️⃣ Uncertainty-Error Correlation (نفس الكود السابق)
    # ═══════════════════════════════════════════════════════════════════════

    print("\n" + "─"*100)
    print("📈 7. Uncertainty-Error Correlation")
    print("─"*100)

    corr_unc_abs, p_abs = pearsonr(df['uncertainty'], df['abs_error'])
    corr_unc_rel, p_rel = pearsonr(df['uncertainty'], df['rel_error'])

    print(f"  📊 الارتباط الكلي:")
    print(f"     Uncertainty ↔ Absolute Error: {corr_unc_abs:+.4f} (p={p_abs:.4e})")
    print(f"     Uncertainty ↔ Relative Error: {corr_unc_rel:+.4f} (p={p_rel:.4e})")

    if corr_unc_abs > 0.5:
        print(f"     ✅ ممتاز: عدم اليقين يعكس الخطأ بدقة")
    elif corr_unc_abs > 0.2:
        print(f"     ⚠️  متوسط: ارتباط إيجابي لكن ضعيف")
    else:
        print(f"     ❌ مشكلة: عدم اليقين لا يعكس الخطأ!")

    # ═══════════════════════════════════════════════════════════════════════
    # 4️⃣ Calibration Analysis (نفس الكود السابق)
    # ═══════════════════════════════════════════════════════════════════════

    print("\n" + "─"*100)
    print("📈 8. Calibration Quality (Binned Analysis)")
    print("─"*100)

    try:
        df['conf_bin'] = pd.qcut(
            df['confidence'].astype(np.float64),
            q=5,
            labels=['Very_Low', 'Low', 'Med', 'High', 'Very_High'],
            duplicates='drop'
        )

        calibration_stats = df.groupby('conf_bin', observed=True).agg({
            'abs_error': ['mean', 'std', 'median'],
            'pct_error': ['mean', 'std'],
            'direction_correct': 'mean',
            'confidence': ['mean', 'count']
        }).round(4)

        save_or_print(calibration_stats.reset_index(), 'legacy_calibration_stats')

        ece = 0
        for bin_name in df['conf_bin'].unique():
            if pd.isna(bin_name):
                continue
            df_bin = df[df['conf_bin'] == bin_name]
            avg_conf = float(df_bin['confidence'].mean())
            avg_acc = float(df_bin['direction_correct'].mean())
            weight = len(df_bin) / len(df)
            ece += weight * abs(avg_conf - avg_acc)

        print(f"\n  📊 Expected Calibration Error (ECE): {ece:.4f}")
        if ece < 0.05:
            print(f"     ✅ ECE < 0.05: ممتاز")
        elif ece < 0.15:
            print(f"     ⚠️  0.05 < ECE < 0.15: مقبول")
        else:
            print(f"     ❌ ECE > 0.15: سيء - النموذج غير مُعاير")

    except Exception as e:
        print(f"  ⚠️ تعذر إنشاء bins: {str(e)}")
        ece = None

    # ═══════════════════════════════════════════════════════════════════════
    # 9️⃣ Summary
    # ═══════════════════════════════════════════════════════════════════════

    print("\n" + "="*100)
    print("📋 الملخص والتوصيات النهائية")
    print("="*100)

    recommendations = []

    # تحليل اتجاه الإغلاق
    if len(df_close) > 0:
        close_accuracy = df_close['direction_correct'].mean() * 100
        recommendations.append(f"📊 دقة توقعات الإغلاق: {close_accuracy:.1f}%")

        if close_accuracy > 60:
            recommendations.append("✅ أداء توقعات الإغلاق ممتاز")
        elif close_accuracy > 55:
            recommendations.append("⚠️  أداء توقعات الإغلاق مقبول")
        else:
            recommendations.append("❌ أداء توقعات الإغلاق ضعيف - يحتاج تحسين")

    # Confidence-Error
    if corr_conf_abs > -0.2:
        recommendations.append("❌ الثقة لا ترتبط بالدقة → أعد معايرة النموذج")
    elif corr_conf_abs > -0.4:
        recommendations.append("⚠️  الارتباط ضعيف → استخدم Calibration Loss")
    else:
        recommendations.append("✅ الثقة تعكس الدقة بشكل جيد")

    # Uncertainty-Error
    if corr_unc_abs < 0.3:
        recommendations.append("❌ عدم اليقين لا يعكس الخطأ → راجع lambda_reg")
    elif corr_unc_abs < 0.5:
        recommendations.append("⚠️  عدم اليقين متوسط → زد penalty_weight")
    else:
        recommendations.append("✅ عدم اليقين مُعاير ممتاز")

    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")

    # ═══════════════════════════════════════════════════════════════════════
    # Return
    # ═══════════════════════════════════════════════════════════════════════

    return {
        'full_data': df,
        'close_data': df_close if 'df_close' in locals() else pd.DataFrame(),
        'correlations': {
            'conf_abs_error': corr_conf_abs,
            'conf_rel_error': corr_conf_rel,
            'unc_abs_error': corr_unc_abs,
            'unc_rel_error': corr_unc_rel
        },
        'ece': ece,
        'close_direction_analysis': dir_df if 'dir_df' in locals() else pd.DataFrame(),
        'confusion_matrix': pivot_table if 'pivot_table' in locals() else pd.DataFrame()
    }


# # ══════════════════════════════════════════════════════════════════════════════
# # 🚀 الاستخدام
# # ══════════════════════════════════════════════════════════════════════════════

# per_asset_df = results2['per_asset_results']
# analysis = comprehensive_uncertainty_analysis(per_asset_df)


## 1️⃣8️⃣ 📚 تقارير محاكاة التداول القديمة (محفوظة ومحدَّثة)

`comprehensive_asset_analysis`, `generate_detailed_report`, `compare_multiple_models` — نفس المنطق، مع حفظ الجداول الكبيرة (ملخص كل العملات، مقارنة النماذج) في ملف بدل طباعتها كاملة.

> 💡 **توصية**: للتحليلات الجديدة استخدم `run_full_analysis` أعلاه — تدعم الفئوي، تحقق فك التشفير، إدارة المخرجات، والرسوم البيانية تلقائياً.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

def comprehensive_asset_analysis(per_asset_results, model_name="النموذج"):
    """
    ✅ تحليل شامل لكل عملة على حدة مع التركيز على تداول الإغلاق
    """
    print("="*100)
    print(f"📊 تحليل شامل لكل عملة - {model_name} (الإغلاق فقط)")
    print("="*100)

    all_assets_analysis = []
    detailed_results = []

    # لكل عملة في النتائج
    for idx, row in per_asset_results.iterrows():
        asset = row['asset']
        print(f"\n{'='*60}")
        print(f"🔍 تحليل مفصل لـ {asset}")
        print(f"{'='*60}")

        # تحليل بيانات الإغلاق فقط
        if 'close' in row:
            data = row['close']

            # تحويل البيانات
            pred = np.array(data['pred_real'], dtype=np.float32)
            true = np.array(data['true_real'], dtype=np.float32)
            entry = np.array(data['entry_price'], dtype=np.float32)
            confidence = np.array(data['confidence'], dtype=np.float32) if 'confidence' in data else np.ones_like(pred) * 0.5

            # إحصائيات العملة
            asset_stats = {
                'العملة': asset,
                'عدد_التوقعات': len(pred),
                'التوقعات_الصحيحة': 0,
                'التوقعات_الخاطئة': 0,
                'صفقات_الشراء': 0,
                'صفقات_الشراء_الصحيحة': 0,
                'صفقات_البيع': 0,
                'صفقات_البيع_الصحيحة': 0,
                'إجمالي_الربح_%': 0,
                'أعلى_ربح_%': float('-inf'),
                'أعلى_خسارة_%': float('inf'),
                'متوسط_الربح_%': 0,
                'متوسط_الثقة': np.mean(confidence) if len(confidence) > 0 else 0
            }

            # تحليل كل توقع
            for i in range(len(pred)):
                # سعر الدخول: سعر إغلاق الشمعة السابقة
                # سعر الخروج: سعر إغلاق الشمعة الحالية

                # حساب الربح/الخسارة بالنسبة المئوية
                if entry[i] != 0:
                    actual_return_pct = ((true[i] - entry[i]) / entry[i]) * 100
                else:
                    actual_return_pct = 0

                # تحديد اتجاه التوقع
                predicted_direction = 'شراء' if pred[i] > entry[i] else 'بيع' if pred[i] < entry[i] else 'حياد'
                actual_direction = 'صعود' if true[i] > entry[i] else 'هبوط' if true[i] < entry[i] else 'ثابت'

                # حساب الربح بناءً على التوقع
                if predicted_direction == 'شراء':
                    profit_pct = actual_return_pct
                    asset_stats['صفقات_الشراء'] += 1
                    if actual_direction == 'صعود':  # شراء صحيح
                        asset_stats['التوقعات_الصحيحة'] += 1
                        asset_stats['صفقات_الشراء_الصحيحة'] += 1
                        asset_stats['إجمالي_الربح_%'] += profit_pct
                    else:  # شراء خاطئ
                        asset_stats['التوقعات_الخاطئة'] += 1
                        asset_stats['إجمالي_الربح_%'] += profit_pct  # profit_pct سيكون سالباً

                elif predicted_direction == 'بيع':
                    profit_pct = -actual_return_pct  # الربح عند الهبوط الصحيح
                    asset_stats['صفقات_البيع'] += 1
                    if actual_direction == 'هبوط':  # بيع صحيح
                        asset_stats['التوقعات_الصحيحة'] += 1
                        asset_stats['صفقات_البيع_الصحيحة'] += 1
                        asset_stats['إجمالي_الربح_%'] += profit_pct
                    else:  # بيع خاطئ
                        asset_stats['التوقعات_الخاطئة'] += 1
                        asset_stats['إجمالي_الربح_%'] += profit_pct  # profit_pct سيكون سالباً

                # تحديث أعلى ربح/خسارة
                if profit_pct > asset_stats['أعلى_ربح_%']:
                    asset_stats['أعلى_ربح_%'] = profit_pct
                if profit_pct < asset_stats['أعلى_خسارة_%']:
                    asset_stats['أعلى_خسارة_%'] = profit_pct

                # حفظ تفاصيل الصفقة
                detailed_results.append({
                    'النموذج': model_name,
                    'العملة': asset,
                    'التوقع': predicted_direction,
                    'الواقع': actual_direction,
                    'الربح_%': profit_pct,
                    'الربح_مطلق': true[i] - entry[i],
                    'الثقة': confidence[i] if i < len(confidence) else 0.5,
                    'سعر_الدخول': entry[i],
                    'سعر_الخروج': true[i],
                    'السعر_المتوقع': pred[i]
                })

            # حساب الإحصائيات النهائية للعملة
            if asset_stats['عدد_التوقعات'] > 0:
                asset_stats['دقة_التوقع_%'] = (asset_stats['التوقعات_الصحيحة'] / asset_stats['عدد_التوقعات']) * 100
                asset_stats['متوسط_الربح_%'] = asset_stats['إجمالي_الربح_%'] / asset_stats['عدد_التوقعات']

                if asset_stats['صفقات_الشراء'] > 0:
                    asset_stats['دقة_الشراء_%'] = (asset_stats['صفقات_الشراء_الصحيحة'] / asset_stats['صفقات_الشراء']) * 100
                else:
                    asset_stats['دقة_الشراء_%'] = 0

                if asset_stats['صفقات_البيع'] > 0:
                    asset_stats['دقة_البيع_%'] = (asset_stats['صفقات_البيع_الصحيحة'] / asset_stats['صفقات_البيع']) * 100
                else:
                    asset_stats['دقة_البيع_%'] = 0

            # عرض نتائج العملة
            print(f"📈 إحصائيات {asset}:")
            print(f"   • عدد التوقعات: {asset_stats['عدد_التوقعات']}")
            print(f"   • الدقة الإجمالية: {asset_stats.get('دقة_التوقع_%', 0):.1f}%")
            print(f"   • دقة صفقات الشراء: {asset_stats.get('دقة_الشراء_%', 0):.1f}% ({asset_stats['صفقات_الشراء_الصحيحة']}/{asset_stats['صفقات_الشراء']})")
            print(f"   • دقة صفقات البيع: {asset_stats.get('دقة_البيع_%', 0):.1f}% ({asset_stats['صفقات_البيع_الصحيحة']}/{asset_stats['صفقات_البيع']})")
            print(f"   • متوسط الربح لكل صفقة: {asset_stats.get('متوسط_الربح_%', 0):.4f}%")
            print(f"   • أعلى ربح: {asset_stats['أعلى_ربح_%']:.2f}%")
            print(f"   • أعلى خسارة: {asset_stats['أعلى_خسارة_%']:.2f}%")
            print(f"   • متوسط الثقة: {asset_stats['متوسط_الثقة']:.3f}")

            # محاكاة التداول للعملة الواحدة
            if asset_stats['عدد_التوقعات'] > 0:
                initial_capital = 10000
                capital = initial_capital

                print(f"\n💰 محاكاة التداول لـ {asset} برأس مال ${initial_capital:,}:")

                # إستراتيجية 1: جميع الإشارات
                capital_all = initial_capital
                # إستراتيجية 2: إشارات عالية الثقة (>0.7)
                capital_high_conf = initial_capital
                # إستراتيجية 3: صفقات الشراء فقط
                capital_buy_only = initial_capital

                for trade in detailed_results:
                    if trade['العملة'] == asset:
                        trade_amount = capital_all * 0.1  # 10% من رأس المال
                        profit_usd = trade_amount * (trade['الربح_%'] / 100)

                        # جميع الإشارات
                        capital_all += profit_usd

                        # إشارات عالية الثقة
                        if trade['الثقة'] > 0.7:
                            capital_high_conf += profit_usd

                        # صفقات الشراء فقط
                        if trade['التوقع'] == 'شراء':
                            capital_buy_only += profit_usd

                print(f"   • جميع الإشارات: ${capital_all:,.2f} (عائد: {((capital_all/initial_capital-1)*100):.1f}%)")
                print(f"   • عالية الثقة (>0.7): ${capital_high_conf:,.2f} (عائد: {((capital_high_conf/initial_capital-1)*100):.1f}%)")
                print(f"   • صفقات الشراء فقط: ${capital_buy_only:,.2f} (عائد: {((capital_buy_only/initial_capital-1)*100):.1f}%)")

                # إضافة نتائج المحاكاة للعملة
                asset_stats['رأس_المال_النهائي_جميع'] = capital_all
                asset_stats['رأس_المال_النهائي_عالية_ثقة'] = capital_high_conf
                asset_stats['رأس_المال_النهائي_شراء_فقط'] = capital_buy_only
                asset_stats['العائد_%_جميع'] = ((capital_all/initial_capital-1)*100)

            all_assets_analysis.append(asset_stats)

    # عرض جدول مقارنة العملات
    if all_assets_analysis:
        df_summary = pd.DataFrame(all_assets_analysis)

        # إنشاء جدول عرض بسيط
        display_cols = [
            'العملة', 'عدد_التوقعات', 'دقة_التوقع_%', 'دقة_الشراء_%',
            'دقة_البيع_%', 'متوسط_الربح_%', 'العائد_%_جميع'
        ]

        df_display = df_summary[display_cols].copy()
        df_display.columns = ['العملة', 'عدد التوقعات', 'الدقة %', 'دقة الشراء %',
                             'دقة البيع %', 'متوسط الربح %', 'العائد %']

        print(f"\n{'='*100}")
        print(f"📋 ملخص أداء جميع العملات - {model_name}")
        print(f"{'='*100}")
        save_or_print(df_display.round(2), f'legacy_assets_summary_{model_name}')

        # تحليل إجمالي
        total_trades = df_summary['عدد_التوقعات'].sum()
        weighted_accuracy = (df_summary['عدد_التوقعات'] * df_summary['دقة_التوقع_%']).sum() / total_trades
        weighted_profit = (df_summary['عدد_التوقعات'] * df_summary['متوسط_الربح_%']).sum() / total_trades

        print(f"\n📊 الإحصائيات الإجمالية - {model_name}:")
        print(f"   • إجمالي التوقعات: {total_trades:,}")
        print(f"   • متوسط الدقة المرجح: {weighted_accuracy:.2f}%")
        print(f"   • متوسط الربح المرجح: {weighted_profit:.4f}%")

        # أفضل وأسوأ عملة
        best_asset = df_summary.loc[df_summary['دقة_التوقع_%'].idxmax()]
        worst_asset = df_summary.loc[df_summary['دقة_التوقع_%'].idxmin()]
        most_profitable = df_summary.loc[df_summary['متوسط_الربح_%'].idxmax()]

        print(f"\n🏆 أفضل العملات - {model_name}:")
        print(f"   • أعلى دقة: {best_asset['العملة']} ({best_asset['دقة_التوقع_%']:.1f}%)")
        print(f"   • أعلى ربحية: {most_profitable['العملة']} ({most_profitable['متوسط_الربح_%']:.4f}%)")
        print(f"   • أقل دقة: {worst_asset['العملة']} ({worst_asset['دقة_التوقع_%']:.1f}%)")

    return {
        'تفاصيل_النتائج': detailed_results,
        'ملخص_العملات': all_assets_analysis,
        'إسم_النموذج': model_name
    }


def compare_multiple_models(models_data, model_names):
    """
    ✅ مقارنة بين عدة نماذج
    """
    print("="*100)
    print("🤝 مقارنة شاملة بين النماذج")
    print("="*100)

    all_models_results = []

    # تحليل كل نموذج
    for i, (model_data, model_name) in enumerate(zip(models_data, model_names)):
        print(f"\n📊 تحليل {model_name}...")
        model_result = comprehensive_asset_analysis(model_data, model_name)
        all_models_results.append(model_result)

    # إنشاء جدول مقارنة
    comparison_table = []

    for model_result in all_models_results:
        model_name = model_result['إسم_النموذج']
        df_summary = pd.DataFrame(model_result['ملخص_العملات'])

        if not df_summary.empty:
            total_trades = df_summary['عدد_التوقعات'].sum()
            weighted_accuracy = (df_summary['عدد_التوقعات'] * df_summary['دقة_التوقع_%']).sum() / total_trades
            weighted_profit = (df_summary['عدد_التوقعات'] * df_summary['متوسط_الربح_%']).sum() / total_trades

            # محاكاة رأس المال الإجمالي
            initial_capital = 10000
            total_capital = initial_capital

            # تجميع جميع الصفقات من كل العملات
            all_trades = [trade for trade in model_result['تفاصيل_النتائج']]

            for trade in all_trades:
                trade_amount = total_capital * 0.1
                profit_usd = trade_amount * (trade['الربح_%'] / 100)
                total_capital += profit_usd

            final_return = ((total_capital / initial_capital - 1) * 100)

            comparison_table.append({
                'النموذج': model_name,
                'عدد_الصفقات': total_trades,
                'متوسط_الدقة_%': f"{weighted_accuracy:.2f}",
                'متوسط_الربح_%': f"{weighted_profit:.4f}",
                'رأس_المال_النهائي': f"${total_capital:,.2f}",
                'العائد_%': f"{final_return:.1f}"
            })

    # عرض جدول المقارنة
    if comparison_table:
        df_comparison = pd.DataFrame(comparison_table)
        df_comparison.columns = ['النموذج', 'عدد الصفقات', 'متوسط الدقة %', 'متوسط الربح %',
                                'رأس المال النهائي', 'العائد %']

        print(f"\n{'='*100}")
        print("📊 جدول مقارنة النماذج")
        print(f"{'='*100}")
        save_or_print(df_comparison, 'legacy_models_comparison')

        # تحديد النموذج الأفضل
        if len(comparison_table) > 1:
            print(f"\n{'='*100}")
            print("🏆 التقييم النهائي")
            print(f"{'='*100}")

            # أفضل نموذج من حيث الدقة
            best_accuracy = max(comparison_table, key=lambda x: float(x['متوسط_الدقة_%']))
            # أفضل نموذج من حيث الربحية
            best_profit = max(comparison_table, key=lambda x: float(x['متوسط_الربح_%']))
            # أفضل نموذج من حيث العائد
            best_return = max(comparison_table, key=lambda x: float(x['العائد_%']))

            print(f"   • أعلى دقة: {best_accuracy['النموذج']} ({best_accuracy['متوسط_الدقة_%']}%)")
            print(f"   • أعلى ربحية: {best_profit['النموذج']} ({best_profit['متوسط_الربح_%']}%)")
            print(f"   • أعلى عائد: {best_return['النموذج']} ({best_return['العائد_%']}%)")

            # التوصية النهائية
            print(f"\n💡 التوصية:")
            if best_accuracy['النموذج'] == best_profit['النموذج'] == best_return['النموذج']:
                print(f"   ✅ {best_accuracy['النموذج']} هو النموذج الأفضل بشكل شامل")
            else:
                print(f"   📊 اختر النموذج بناءً على أولويتك:")
                print(f"      - للدقة: {best_accuracy['النموذج']}")
                print(f"      - للربحية: {best_profit['النموذج']}")
                print(f"      - للعائد المالي: {best_return['النموذج']}")

    return all_models_results


def generate_detailed_report(analysis_results):
    """
    ✅ إنشاء تقرير مفصل عن النموذج
    """
    model_name = analysis_results['إسم_النموذج']
    df_summary = pd.DataFrame(analysis_results['ملخص_العملات'])

    print("="*100)
    print(f"📄 تقرير مفصل - {model_name}")
    print("="*100)

    if not df_summary.empty:
        # 1. إحصائيات عامة
        total_trades = df_summary['عدد_التوقعات'].sum()
        total_correct = df_summary['التوقعات_الصحيحة'].sum()
        total_wrong = df_summary['التوقعات_الخاطئة'].sum()
        total_buy = df_summary['صفقات_الشراء'].sum()
        total_sell = df_summary['صفقات_البيع'].sum()

        print(f"\n📊 الإحصائيات العامة:")
        print(f"   • إجمالي التوقعات: {total_trades:,}")
        print(f"   • التوقعات الصحيحة: {total_correct:,} ({total_correct/total_trades*100:.1f}%)")
        print(f"   • التوقعات الخاطئة: {total_wrong:,} ({total_wrong/total_trades*100:.1f}%)")
        print(f"   • صفقات الشراء: {total_buy:,} ({total_buy/total_trades*100:.1f}%)")
        print(f"   • صفقات البيع: {total_sell:,} ({total_sell/total_trades*100:.1f}%)")

        # 2. تحليل الربحية
        weighted_profit = (df_summary['عدد_التوقعات'] * df_summary['متوسط_الربح_%']).sum() / total_trades
        total_profit_pct = df_summary['إجمالي_الربح_%'].sum()

        print(f"\n💰 تحليل الربحية:")
        print(f"   • متوسط الربح لكل صفقة: {weighted_profit:.4f}%")
        print(f"   • إجمالي الربح المتراكم: {total_profit_pct:.2f}%")

        # 3. أفضل 3 عملات
        print(f"\n🏅 أفضل 3 عملات في {model_name}:")
        for i, (idx, row) in enumerate(df_summary.nlargest(3, 'دقة_التوقع_%').iterrows(), 1):
            print(f"   {i}. {row['العملة']}: {row['دقة_التوقع_%']:.1f}% دقة، {row['متوسط_الربح_%']:.4f}% ربح")

        # 4. التوصيات
        print(f"\n🎯 التوصيات العملية لـ {model_name}:")

        # توصية عامة
        overall_accuracy = total_correct / total_trades * 100
        if overall_accuracy > 70:
            print(f"   ✅ النموذج ممتاز للتداول (دقة {overall_accuracy:.1f}%)")
        elif overall_accuracy > 60:
            print(f"   ⚠️  النموذج جيد للتداول مع إدارة مخاطر (دقة {overall_accuracy:.1f}%)")
        elif overall_accuracy > 50:
            print(f"   ⚠️  النموذج مقبول للتداول الورقي فقط (دقة {overall_accuracy:.1f}%)")
        else:
            print(f"   ❌ النموذج يحتاج تحسين (دقة {overall_accuracy:.1f}%)")

        # توصية حسب العملات
        best_asset = df_summary.loc[df_summary['دقة_التوقع_%'].idxmax()]
        print(f"\n   🎯 أفضل عملة: {best_asset['العملة']}")
        print(f"      • الدقة: {best_asset['دقة_التوقع_%']:.1f}%")
        print(f"      • متوسط الربح: {best_asset['متوسط_الربح_%']:.4f}%")
        print(f"      • عدد الصفقات: {best_asset['عدد_التوقعات']}")

        # 5. إدارة المخاطر
        print(f"\n⚠️  إدارة المخاطر الموصى بها:")
        print(f"   • حجم الصفقة: 2-5% من رأس المال")
        print(f"   • وقف الخسارة: 1-2% لكل صفقة")
        print(f"   • جني الأرباح: 2-3% لكل صفقة")
        print(f"   • الحد الأقصى للخسارة اليومية: 5% من رأس المال")

        # 6. الإستراتيجية المثلى
        print(f"\n📈 الإستراتيجية المثلى لـ {model_name}:")

        # حساب أفضل إستراتيجية بناءً على المحاكاة
        avg_return_all = df_summary['العائد_%_جميع'].mean() if 'العائد_%_جميع' in df_summary.columns else 0

        if avg_return_all > 100:
            print(f"   • استخدم جميع الإشارات (عائد متوقع: {avg_return_all:.1f}%)")
        else:
            # تحليل الثقة
            if 'متوسط_الثقة' in df_summary.columns:
                avg_confidence = df_summary['متوسط_الثقة'].mean()
                if avg_confidence > 0.7:
                    print(f"   • استخدم إشارات عالية الثقة (>0.7)")
                else:
                    print(f"   • استخدم عتبة ثقة 0.5 للحصول على تغطية جيدة")

            # تحليل الاتجاهات
            avg_buy_accuracy = df_summary['دقة_الشراء_%'].mean()
            avg_sell_accuracy = df_summary['دقة_البيع_%'].mean()

            if avg_buy_accuracy > avg_sell_accuracy + 5:
                print(f"   • ركز على صفقات الشراء (دقة أعلى: {avg_buy_accuracy:.1f}% vs {avg_sell_accuracy:.1f}%)")
            elif avg_sell_accuracy > avg_buy_accuracy + 5:
                print(f"   • ركز على صفقات البيع (دقة أعلى: {avg_sell_accuracy:.1f}% vs {avg_buy_accuracy:.1f}%)")
            else:
                print(f"   • استخدم كلا نوعي الصفقات (دقة متقاربة: شراء {avg_buy_accuracy:.1f}%، بيع {avg_sell_accuracy:.1f}%)")


# ══════════════════════════════════════════════════════════════════════════════
# 🚀 استخدام الكود مع بياناتك
# ══════════════════════════════════════════════════════════════════════════════

print("="*100)
print("🤖 نظام تحليل النماذج المتقدم")
print("="*100)

# الحالة 1: إذا لديك نموذج واحد
# if 'results2' in locals():
#     print("\n📊 تحليل نموذج واحد...")

#     # تحليل شامل للنموذج الواحد
#     analysis_result = comprehensive_asset_analysis(
#         per_asset_results=results2['per_asset_results'],
#         model_name="النموذج الأساسي"
#     )

#     # إنشاء تقرير مفصل
#     generate_detailed_report(analysis_result)

# # الحالة 2: إذا لديك نموذجان للمقارنة
# elif all(key in locals() for key in ['results_model1', 'results_model2']):
# # if True:
#     print("\n📊 مقارنة بين نموذجين...")

#     models_data = [
#         results['per_asset_results'],
#         results2['per_asset_results']
#     ]

#     model_names = ["النموذج الأول", "النموذج الثاني"]

#     # مقارنة النماذج
#     all_results = compare_multiple_models(models_data, model_names)

#     # إنشاء تقارير مفصلة لكل نموذج
#     for result in all_results:
#         generate_detailed_report(result)

# # الحالة 3: إذا لديك بيانات مباشرة
# else:
#     print("⚠️  لم يتم العثور على بيانات النماذج.")
#     print("يرجى التأكد من وجود متغيرات:")
#     print("   - results (لنموذج واحد)")
#     print("   - results_model1 و results_model2 (لمقارنة نموذجين)")

# print("\n" + "="*100)
# print("✅ تم الانتهاء من التحليل بنجاح")
# print("="*100)

# # ══════════════════════════════════════════════════════════════════════════════
# # 💾 حفظ النتائج
# # ══════════════════════════════════════════════════════════════════════════════

# def save_analysis_results(analysis_results, filename_prefix):
#     """
#     حفظ نتائج التحليل في ملفات CSV
#     """
#     import os

#     # إنشاء مجلد للنتائج إذا لم يكن موجوداً
#     os.makedirs('analysis_results', exist_ok=True)

#     # حفظ ملخص العملات
#     if 'ملخص_العملات' in analysis_results and analysis_results['ملخص_العملات']:
#         df_summary = pd.DataFrame(analysis_results['ملخص_العملات'])
#         summary_filename = f'analysis_results/{filename_prefix}_summary.csv'
#         df_summary.to_csv(summary_filename, index=False, encoding='utf-8-sig')
#         print(f"✅ تم حفظ ملخص العملات في: {summary_filename}")

#     # حفظ التفاصيل
#     if 'تفاصيل_النتائج' in analysis_results and analysis_results['تفاصيل_النتائج']:
#         df_details = pd.DataFrame(analysis_results['تفاصيل_النتائج'])
#         details_filename = f'analysis_results/{filename_prefix}_details.csv'
#         df_details.to_csv(details_filename, index=False, encoding='utf-8-sig')
#         print(f"✅ تم حفظ تفاصيل الصفقات في: {details_filename}")

#     return True

# # حفظ النتائج إذا كان هناك تحليل
# if 'analysis_result' in locals():
#     save_analysis_results(analysis_result, 'single_model_analysis')
# elif 'all_results' in locals():
#     for i, result in enumerate(all_results):
#         save_analysis_results(result, f'model_{i+1}_analysis')

## 1️⃣9️⃣ 🎯 اختيار وتصنيف الصفقات: التحرك المتوقع مقابل عدم اليقين (Edge vs Uncertainty)

**الفكرة**: صفقة "تستحق الثقة" فقط إن كان **التحرك المتوقع** (المسافة بين
السعر المتوقع وسعر الدخول) **أكبر من عدم اليقين** المصاحب للتنبؤ — أي أن
الإشارة تتجاوز الضجيج الإحصائي. أي صفقة يكون فيها التحرك المتوقع أصغر من (أو
يساوي) عدم اليقين تُستبعد تلقائياً بالكامل لأنها غير موثوقة (النموذج نفسه غير
متأكد أن الاتجاه المتوقع حقيقي وليس ضوضاء).

`select_and_rank_trades` تعمل تلقائياً على:
- ناتج `build_flat_dataframe` (بيانات اختبار/باكتست تاريخية بوحدات حقيقية)
- ناتج `build_latest_table` / `predict_latest_v4` / `predict_latest_all_assets`
  (تقرير التداول الحي بالنسبة المئوية)

وتكتشف الأعمدة المناسبة تلقائياً حسب مصدر البيانات (أو تقبلها صراحةً)، ثم
تُرتّب الصفقات الناجية من الفلتر حسب: **الثقة** فقط، أو **حجم التحرك** فقط،
أو **الاثنين معاً** (نقاط مركّبة تمنع صفقة قوية في معيار واحد وضعيفة جداً في
الآخر من الصعود للقمة ظلماً).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🎯 select_and_rank_trades: فلترة الصفقات (تحرك متوقع > عدم يقين) + ترتيبها
# ═══════════════════════════════════════════════════════════════════════════

# كل عنصر: (عمود التحرك, عمود عدم اليقين, عمود الثقة) — يجب أن تكون الثلاثة
# بنفس "الفضاء" (وحدات حقيقية معاً، أو نسبة مئوية معاً) حتى تُقارَن بصحة.
_COMPATIBLE_TRADE_COLUMNS = [
    ('predicted_change', 'uncertainty', 'confidence'),   # build_flat_dataframe (وحدات حقيقية)
    ('move_%', 'uncertainty_%', 'confidence_%'),         # build_latest_table / predict_latest_v4 (نسبة مئوية)
]


def _detect_trade_columns(df: pd.DataFrame, move_col=None, uncertainty_col=None, confidence_col=None):
    '''يختار ثلاثية (move, uncertainty, confidence) متوافقة: إمّا الثلاثة صراحةً معاً، أو اكتشاف تلقائي لزوج معروف.'''
    any_explicit = any(c is not None for c in (move_col, uncertainty_col, confidence_col))
    if any_explicit:
        missing = [n for n, c in (('move_col', move_col), ('uncertainty_col', uncertainty_col),
                                   ('confidence_col', confidence_col)) if c is None]
        if missing:
            raise ValueError(f"عند تحديد أي عمود يدوياً يجب تحديد الثلاثة معاً: {missing} مفقودة.")
        for c in (move_col, uncertainty_col, confidence_col):
            if c not in df.columns:
                raise KeyError(f"العمود '{c}' غير موجود. الأعمدة المتاحة: {list(df.columns)}")
        return move_col, uncertainty_col, confidence_col

    for m, u, c in _COMPATIBLE_TRADE_COLUMNS:
        if m in df.columns and u in df.columns and c in df.columns:
            return m, u, c

    raise KeyError(
        "تعذّر اكتشاف أعمدة (تحرك / عدم يقين / ثقة) متوافقة تلقائياً ضمن الصيغ المعروفة "
        f"{_COMPATIBLE_TRADE_COLUMNS}. مرّرها صراحةً (الثلاثة معاً) عبر "
        "move_col / uncertainty_col / confidence_col.\n"
        f"الأعمدة المتاحة فعلياً: {list(df.columns)}"
    )


def select_and_rank_trades(
    df: pd.DataFrame,
    move_col: Optional[str] = None,
    uncertainty_col: Optional[str] = None,
    confidence_col: Optional[str] = None,
    min_edge_ratio: float = 1.0,
    sort_by: str = 'both',
    ascending: bool = False,
    top_n: Optional[int] = None,
) -> pd.DataFrame:
    '''
    يُبقي فقط الصفقات التي |التحرك المتوقع| > عدم اليقين × min_edge_ratio (الإشارة
    أكبر من الضجيج)، ويستبعد كل الباقي تماماً (لا يظهر إطلاقاً في الناتج). ثم
    يرتّب الصفقات الناجية حسب sort_by.

    تعمل تلقائياً على ناتج build_flat_dataframe (باكتست) أو build_latest_table/
    predict_latest_v4/predict_latest_all_assets (لايف) — تكتشف الأعمدة المناسبة
    تلقائياً، أو مرّرها صراحةً (الثلاثة معاً) إن كان لديك تسمية مختلفة.

    Args:
        min_edge_ratio: الحد الأدنى لنسبة (|move| / عدم اليقين) لقبول الصفقة.
            1.0 (افتراضي) = التحرك المتوقع أكبر من عدم اليقين بمقدار مرة واحدة
            على الأقل. أكبر من 1.0 أكثر تشدداً (هامش أمان إضافي، صفقات أقل
            وأنقى)، أصغر من 1.0 أكثر تساهلاً (صفقات أكثر).
        sort_by:
            'confidence' -> الأعلى ثقة أولاً.
            'move'       -> الأكبر تحركاً متوقعاً أولاً (بالقيمة المطلقة).
            'both'       -> (افتراضي) نقاط مركّبة = متوسط الرتبة المئوية للثقة
                            والرتبة المئوية لحجم التحرك؛ يمنع صفقة متفوقة في
                            معيار واحد فقط من التصدّر ظلماً.
            'edge'       -> الأعلى في نسبة (|move| / عدم اليقين) نفسها (أنقى صفقة
                            إحصائياً، بغض النظر عن حجم التحرك أو الثقة المطلقة).
        top_n: أعد فقط أفضل top_n صفقة بعد الترتيب (None = كل الصفقات الناجية).

    Returns:
        DataFrame (نسخة مُرشَّحة ومرتَّبة من df) + أعمدة إضافية:
        edge_ratio (|move|/عدم اليقين)، direction ('up'/'down')، rank_score.
        فارغ إن لم تنجُ أي صفقة من الفلتر.
    '''
    if df is None or df.empty:
        return df.copy() if df is not None else pd.DataFrame()

    work = df.copy()
    if 'kind' in work.columns:
        work = work[work['kind'] == 'continuous'].copy()
    if work.empty:
        return work

    mcol, ucol, ccol = _detect_trade_columns(work, move_col, uncertainty_col, confidence_col)

    move = pd.to_numeric(work[mcol], errors='coerce')
    unc = pd.to_numeric(work[ucol], errors='coerce')
    conf = pd.to_numeric(work[ccol], errors='coerce')

    valid = np.isfinite(move) & np.isfinite(unc) & (unc > 0)
    work, move, unc, conf = work.loc[valid].copy(), move[valid], unc[valid], conf[valid]

    edge_ratio = move.abs() / unc
    keep = edge_ratio > float(min_edge_ratio)
    work = work.loc[keep].copy()

    if work.empty:
        print(f"⚠️ لا توجد صفقات يتجاوز فيها |التحرك المتوقع| عدم اليقين × {min_edge_ratio} (0 من {int(valid.sum())}).")
        return work

    work['edge_ratio'] = edge_ratio.loc[keep].values
    work['direction'] = np.where(move.loc[keep].values > 0, '▲', '▼')

    conf_kept, move_kept = conf.loc[keep], move.loc[keep].abs()
    if sort_by == 'confidence':
        work['rank_score'] = conf_kept.values
    elif sort_by == 'move':
        work['rank_score'] = move_kept.values
    elif sort_by == 'edge':
        work['rank_score'] = work['edge_ratio'].values
    elif sort_by == 'both':
        # ✅ رتبة مئوية *داخل كل هدف على حدة* لا على المجموع المُجمَّع: الثقة/الحجم
        # لكل هدف (close/high/low) لهما رأس NIG/تصنيف منفصل تماماً، فمقياسهما غير
        # قابل للمقارنة المباشرة بين الأهداف (اكتُشف عملياً: هدف "high" أظهر ثقة
        # أعلى بكثير من "close"/"low" رغم ترابط ضعيف بين ثقته ودقّته الفعلية —
        # راجع تقرير المعايرة — فرتبة مئوية مُجمَّعة عبر كل الأهداف معاً كانت
        # تُصعّد صفقات "high" ظلماً بلا علاقة بجودتها الفعلية). التجميع لكل هدف
        # على حدة يُطبَّق فقط حين تتعدّد الأهداف فعلياً في work['target']؛ هدف
        # واحد (الاستخدام المُوصى به في أمثلة القسم) يُعيد نفس السلوك القديم تماماً.
        if 'target' in work.columns and work['target'].nunique() > 1:
            groups = work['target'].values
            conf_pct = conf_kept.groupby(groups).rank(pct=True, na_option='bottom')
            move_pct = move_kept.groupby(groups).rank(pct=True, na_option='bottom')
        else:
            conf_pct = conf_kept.rank(pct=True, na_option='bottom')
            move_pct = move_kept.rank(pct=True, na_option='bottom')
        work['rank_score'] = ((conf_pct.values + move_pct.values) / 2.0)
    else:
        raise ValueError("sort_by يجب أن تكون إحدى: 'confidence' | 'move' | 'both' | 'edge'")

    work = work.sort_values('rank_score', ascending=ascending).reset_index(drop=True)
    return work.head(int(top_n)) if top_n is not None else work


def _test_select_and_rank_trades_per_target_fairness():
    # يثبت أن رتبة مئوية مجمَّعة عبر أهداف مختلفة لا تُرقّي هدفاً
    # واحداً مجرّد ارتفاع نطاق ثقته عن بقية الأهداف (بلا تحسُّن حقيقي في الدقة)،
    # بخلاف السلوك القديم الذي كان يُصعّد ذلك الهدف 100% من أعلى N صفقة بلا استثناء.
    rng = np.random.default_rng(0)
    n = 200
    rows = []
    for target, conf_level in (("high", 0.85), ("close", 0.30), ("low", 0.25)):
        move = rng.normal(0, 1, n)
        unc = np.abs(rng.normal(1, 0.2, n)) + 0.1
        conf = np.clip(conf_level + rng.normal(0, 0.05, n), 0.01, 0.99)
        for m, u, c in zip(move, unc, conf):
            rows.append({"target": target, "predicted_change": m, "uncertainty": u,
                        "confidence": c, "kind": "continuous"})
    df = pd.DataFrame(rows)
    selected = select_and_rank_trades(df, min_edge_ratio=1.0, sort_by='both', top_n=30)
    counts = selected['target'].value_counts()
    assert len(counts) == 3, (
        f"الرتبة المئوية المجمَّعة ما زالت تُرقّي هدفاً واحداً بسبب اختلاف مقياس الثقة (الموجود: {dict(counts)})")
    assert counts.max() <= 15, (
        f"هدف واحد يستحوذ أكثر من نصف أفضل 30 صفقة رغم تساوي الجودة الفعلية بين الأهداف الثلاثة في البيانات التركيبية (الموجود: {dict(counts)})")
    print("✅ select_and_rank_trades: الرتبة المئوية لـ sort_by='both' عند تعدّد الأهداف محسوبة داخل كل هدف على حدة، لا تُرقّي هدفاً واحداً بسبب تضخم ثقته الخام.")
    return True


_test_select_and_rank_trades_per_target_fairness()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🖨️ print_trade_selection: عرض مضغوط لناتج select_and_rank_trades
# ═══════════════════════════════════════════════════════════════════════════

def _trade_accuracy_pct(series: pd.Series) -> Tuple[float, int]:
    '''معدّل الإصابة % من عمود ok/direction_correct/correct (يتجاهل None/NaN بأمان).'''
    s = series.dropna()
    if len(s) == 0:
        return float('nan'), 0
    return float(s.astype(bool).mean()) * 100.0, len(s)


def print_trade_selection(
    result: pd.DataFrame,
    original: Optional[pd.DataFrame] = None,
    title: Optional[str] = None,
    max_rows: Optional[int] = 20,
    indent: str = "  ",
) -> None:
    '''
    يطبع ناتج select_and_rank_trades بشكل مضغوط (رقم، تاريخ/عملة/هدف، اتجاه،
    ثقة، عدم يقين، تحرك، edge، سعر متوقع/دخول، صح/خطأ إن توفّر هدف فعلي).

    إن مُرِّر original (الـ DataFrame *قبل* الفلترة، بنفس عمود النتيجة
    direction_correct/correct/ok)، يُطبع أسفل الجدول مقارنة معدل الإصابة قبل
    وبعد الفلترة + Top-N المعروضة — لإظهار أداء الصفقات المعروضة مقارنةً بالمجموعة الأصلية.
    '''
    if result is None or result.empty:
        if title:
            print(title)
        print(indent + "(لا توجد صفقات تجتاز الفلتر)")
        return

    if title:
        print(title)

    show = result.head(max_rows) if max_rows else result
    mcol, ucol, ccol = _detect_trade_columns(show)

    out = pd.DataFrame(index=show.index)
    out['#'] = np.arange(1, len(show) + 1)
    if 'date' in show.columns and show['date'].notna().any():
        out['date'] = show['date'].map(lambda d: d.strftime('%Y-%m-%d %H:%M') if pd.notna(d) else '—')
    elif 'idx' in show.columns:
        out['#idx'] = show['idx']
    if 'asset' in show.columns:
        out['asset'] = show['asset']
    out['target'] = show['target']
    out['dir'] = show['direction']

    conf_vals = show[ccol].astype(float)
    conf_pct = conf_vals * 100.0 if conf_vals.max(skipna=True) <= 1.5 else conf_vals
    out['conf%'] = conf_pct.map(lambda v: _fmt_num(v, '.1f'))
    out['unc'] = show[ucol].map(lambda v: _fmt_num(v, '.4g'))
    out['move'] = show[mcol].map(lambda v: _fmt_num(v, '+.3f'))
    out['edge_x'] = show['edge_ratio'].map(lambda v: _fmt_num(v, '.2f'))
    if 'pred' in show.columns:
        out['pred'] = show['pred'].map(_fmt_price)
    if 'entry' in show.columns:
        out['entry'] = show['entry'].map(_fmt_price)

    ok_col = next((c for c in ('direction_correct', 'ok', 'correct') if c in show.columns), None)
    if ok_col:
        out['ok'] = show[ok_col].map(
            lambda v: '⏳' if (v is None or (isinstance(v, float) and np.isnan(v))) else ('✅' if bool(v) else '❌')
        )

    print("\n".join(indent + ln for ln in out.to_string(index=False).split("\n")))
    if max_rows and len(result) > max_rows:
        print(indent + f"… و{len(result) - max_rows} صفقة أخرى (max_rows=None لعرض الكل)")

    ok_col = next((c for c in ('direction_correct', 'correct', 'ok') if c in result.columns), None)
    if ok_col and original is not None and ok_col in original.columns:
        base_acc, base_n = _trade_accuracy_pct(original[ok_col])
        sel_acc, sel_n = _trade_accuracy_pct(result[ok_col])
        if base_n and sel_n:
            print(f"\n{indent}📈 معدل الإصابة: قبل الفلترة {base_acc:.1f}% (n={base_n})  ->  "
                  f"بعد الفلترة + Top-N {sel_acc:.1f}% (n={sel_n}, {sel_n / base_n * 100:.0f}% من الصفقات)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 🚀 أمثلة استخدام
# ══════════════════════════════════════════════════════════════════════════
#
# # 1) على نتائج باكتست (test_all_assets_v4 → build_flat_dataframe):
# flat_df = build_flat_dataframe(results_df, target_specs='close')
# selected = select_and_rank_trades(flat_df, min_edge_ratio=1.0, sort_by='both', top_n=20)
# print_trade_selection(selected, original=flat_df,
#                        title="\n🎯 أفضل 20 صفقة (تحرك متوقع > عدم يقين، مرتّبة بالثقة+الحجم):")
#
# # 2) بترتيب حسب الثقة فقط، أو حسب حجم التحرك فقط، أو حسب نقاء الإشارة (edge):
# by_conf = select_and_rank_trades(flat_df, sort_by='confidence')
# by_move = select_and_rank_trades(flat_df, sort_by='move')
# by_edge = select_and_rank_trades(flat_df, sort_by='edge')
#
# # 3) مباشرة على تقرير التداول الحي (نفس الدالة، بلا أي تغيير):
# live_table = predict_latest_all_assets(model, test_dict, ['1h', '4h', '1D'],
#                                        target_specs='close', n_display=10, verbose=False)
# live_selected = select_and_rank_trades(live_table, min_edge_ratio=1.2, sort_by='confidence')
# print_trade_selection(live_selected, title="\n🔴 صفقات لايف مؤهّلة (تحرك > 1.2x عدم اليقين):")


## 2️⃣0️⃣ 🧬 تحليل المدخلات ↔ المخرجات مع التحقق الإحصائي (Validated Input-Output Pattern Discovery)

يجيب على السؤال: **ما الأنماط في المدخلات الخام (نوافذ X) التي ترتبط فعلاً بنجاح
التوقع، وما الأنماط التي تبدو مقنعة لكنها عشوائية/حفظ زائد؟**

على عكس `detect_success_failure_patterns` (١٢) الذي يعمل على خصائص *مُشتَقّة من
المخرجات* (الثقة، عدم اليقين، حجم الحركة المتوقعة)، القسم هنا يعمل على
**المدخلات الخام** نفسها (نوافذ X التي تدخل النموذج) ويضيف خطوة **تحقق إحصائية
إلزامية** (تقسيم زمني train/validation + Binomial Test + Permutation Test) قبل
تصنيف أي نمط كـ"جيد" أو "سيء" — أي نمط لا يتكرر خارج بيانات اكتشافه يُصنَّف
صراحة "🎲 غير موثوق" ولا يظهر ضمن القوائم النهائية.

**الدوال:**
1. `extract_input_snapshot_features` — يحوّل نوافذ X الخام إلى جدول خصائص واحد (صف/عينة).
2. `analyze_input_output_validated_patterns` — القلب: شجرة قرار + Binomial Test لكل ورقة + Permutation Test لكل خاصية مفردة.
3. `run_input_output_pattern_discovery` — غلاف سريع يربط الخطوتين أعلاه مباشرة بأصل من `test_dict` (تنبؤ + تقييم + تحليل أنماط في استدعاء واحد).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🧬 بصمة المدخلات: تحويل نوافذ X الخام إلى جدول خصائص واحد (صف لكل عينة)
# ═══════════════════════════════════════════════════════════════════════════

def extract_input_snapshot_features(
    X_inputs: Tuple[np.ndarray, ...],
    timeframe_names: List[str],
    feature_names: Optional[List[Optional[List[str]]]] = None,
    agg: Tuple[str, ...] = ('last', 'mean', 'std'),
    max_features_per_tf: Optional[int] = None,
) -> pd.DataFrame:
    """
    يحوّل نوافذ مدخلات النموذج الخام (X_inputs — نفس ما يدخل predict_batch_v4)
    إلى جدول واحد "بصمة مدخلات" (Input Snapshot) بصف واحد لكل عينة، ليصلح
    كمدخل لأي تحليل إحصائي/تعلّم آلي لاحق (خاصة analyze_input_output_validated_patterns
    و run_input_output_pattern_discovery أدناه).

    لكل إطار زمني ولكل خاصية داخل النافذة تُحسب إحصاءات ملخِّصة عبر محور الزمن
    (آخر قيمة/متوسط/انحراف معياري...) بدل تمرير النافذة الكاملة — لأن الهدف هنا
    تفسير الأنماط، لا إعادة تدريب النموذج.

    Args:
        X_inputs: tuple بنفس طول/ترتيب timeframe_names؛ كل عنصر [N, window, n_features]
                  أو [N, n_features] (بلا نافذة زمنية).
        timeframe_names: أسماء الأطر الزمنية بنفس ترتيب X_inputs، مثل ['1h','4h','1D'].
        feature_names: أسماء خصائص كل إطار (قائمة قوائم بنفس طول timeframe_names)؛
                       إن غابت (أو غاب عنصر لإطار معيّن) تُولَّد أسماء عامة f0..fK.
        agg: الإحصاءات المُحسَبة عبر محور النافذة الزمنية — من
             {'last','mean','std','min','max'}.
        max_features_per_tf: تحديد عدد أول N خاصية من كل إطار (لتفادي انفجار الأبعاد
                              إن كانت الخصائص كثيرة جداً)؛ None = كل الخصائص.

    Returns:
        DataFrame بأسماء أعمدة مقروءة مثل '1D_rsi_last', '4h_close_mean', ...
        (N صف بنفس ترتيب X_inputs — أي بنفس ترتيب last_candles/base_params).
    """
    if len(X_inputs) != len(timeframe_names):
        raise ValueError("طول X_inputs يجب أن يطابق طول timeframe_names.")

    all_cols: Dict[str, np.ndarray] = {}

    for tf_idx, (tf_name, X_tf) in enumerate(zip(timeframe_names, X_inputs)):
        arr = np.asarray(X_tf)
        if arr.ndim == 2:          # [N, features] بلا نافذة زمنية → نافذة طولها 1
            arr = arr[:, None, :]
        elif arr.ndim != 3:
            raise ValueError(f"شكل غير مدعوم لـ X_inputs['{tf_name}']: {arr.shape}")

        n_feat = arr.shape[-1]
        if max_features_per_tf is not None:
            n_feat = min(n_feat, max_features_per_tf)
            arr = arr[..., :n_feat]

        names_tf = None
        if feature_names is not None and tf_idx < len(feature_names) and feature_names[tf_idx] is not None:
            names_tf = list(feature_names[tf_idx])[:n_feat]
        if not names_tf or len(names_tf) != n_feat:
            names_tf = [f"f{i}" for i in range(n_feat)]

        for fi, fname in enumerate(names_tf):
            series = arr[:, :, fi].astype(np.float64)  # [N, window]
            if 'last' in agg:
                all_cols[f"{tf_name}_{fname}_last"] = series[:, -1]
            if 'mean' in agg:
                all_cols[f"{tf_name}_{fname}_mean"] = np.nanmean(series, axis=1)
            if 'std' in agg:
                all_cols[f"{tf_name}_{fname}_std"] = np.nanstd(series, axis=1)
            if 'min' in agg:
                all_cols[f"{tf_name}_{fname}_min"] = np.nanmin(series, axis=1)
            if 'max' in agg:
                all_cols[f"{tf_name}_{fname}_max"] = np.nanmax(series, axis=1)

    return pd.DataFrame(all_cols)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🔬 ربط المدخلات بالمخرجات + التحقق الإحصائي (Validated Input-Output Patterns)
# ═══════════════════════════════════════════════════════════════════════════
#
# الفكرة: أي شجرة قرار أو ترابط إحصائي سيجد "أنماطاً" حتى في بيانات عشوائية
# تماماً إن قِسناها على نفس البيانات التي اكتُشفت منها (overfitting). لذلك كل
# نمط هنا يُختبر على بيانات لم يرَها الاكتشاف (تقسيم زمني train/validation)
# ويُخضَع لاختبار إحصائي (Binomial Test / Permutation Test) قبل أن يُصنَّف
# "جيداً" أو "سيئاً" — وإلا يُصنَّف صراحة كنمط "غير موثوق/يشبه العشوائية".
# ═══════════════════════════════════════════════════════════════════════════

def analyze_input_output_validated_patterns(
    input_features: pd.DataFrame,
    outcome: Union[pd.Series, np.ndarray],
    timestamps: Optional[Union[pd.Series, np.ndarray]] = None,
    validation_frac: float = 0.3,
    max_tree_depth: int = 3,
    min_leaf_samples: int = 30,
    n_permutations: int = 300,
    alpha: float = 0.05,
    consistency_gap_max: float = 0.15,
    verbose: bool = True,
) -> Dict:
    """
    يربط "بصمة المدخلات" (input_features — جدول خصائص خام، صف لكل عينة، مثل
    ناتج extract_input_snapshot_features) بنتيجة التوقع (outcome — 0/1: هل كان
    صحيحاً؟) ثم يتحقق إحصائياً من كل نمط مُكتشَف قبل تصنيفه، لتفادي أنماط
    عشوائية تبدو مقنعة على بيانات التدريب لكنها لا تتكرر خارجها.

    المنهجية:
      1) تقسيم **زمني** (لا عشوائي) train/validation — يحاكي واقع التداول
         (اكتشاف على الماضي، تحقق على المستقبل الأقرب؛ يفترض أن صفوف
         input_features/outcome مرتّبة زمنياً أصلاً، أو تُرتَّب عبر timestamps
         إن مُرِّرت).
      2) شجرة قرار ضحلة على train فقط → قواعد "إن-فإن" (كل ورقة = نمط مرشّح).
      3) لكل قاعدة: نسبة النجاح على train **و** على validation منفصلتين.
      4) اختبار ذو حدين (Binomial Test) يقارن نجاح القاعدة في validation
         بمعدل النجاح الأساسي (baseline) في نفس الفترة → p-value.
      5) القاعدة تُصنَّف "✅ نمط موثوق جيد" فقط إن كانت دلالتها الإحصائية
         موجودة في validation (p<alpha) **و** فجوة train/validation صغيرة
         (< consistency_gap_max) — أي لم "تحفظ" الشجرة صدفة من التدريب.
         نفس المنطق بالعكس لـ"❌ نمط موثوق سيء". أي قاعدة تفشل أحد الشرطين
         تُصنَّف "🎲 غير موثوق / يشبه العشوائية" ولا يُعتمَد عليها.
      6) بالتوازي: لكل خاصية مفردة (عمود عددي) اختبار تبديل (Permutation Test)
         مستقل عن الشجرة — يخلط outcome عشوائياً n_permutations مرة ويقارن
         الارتباط الحقيقي بتوزيع الارتباطات العشوائية، ثم يتحقق أن الارتباط
         **يتكرر بنفس الاتجاه** على validation قبل اعتباره حقيقياً.

    Args:
        input_features: DataFrame خصائص خام (عمودي رقمية)، صف لكل عينة.
        outcome: مصفوفة/عمود 0/1 (أو bool) — هل كان التوقع صحيحاً لكل عينة؟
        timestamps: (اختياري) لترتيب العينات زمنياً قبل التقسيم؛ إن غابت
                    يُفترض أن الترتيب الحالي زمني أصلاً (كما تُبنى بيانات هذا
                    المشروع عبر last_candles/base_params).
        validation_frac: نسبة عينات التحقق (من نهاية السلسلة الزمنية).
        max_tree_depth / min_leaf_samples: ضبط شجرة القرار الاستكشافية.
        n_permutations: عدد تكرارات اختبار التبديل لكل خاصية.
        alpha: مستوى الدلالة الإحصائية المطلوب (افتراضياً 5%).
        consistency_gap_max: أقصى فرق مقبول بين نسبة نجاح train وvalidation
                              لاعتبار القاعدة مستقرة (وليست حفظاً زائداً).

    Returns:
        dict: {
            'rules_df': DataFrame,             # كل ورقة/قاعدة + إحصاءات + الحكم
            'feature_stats_df': DataFrame,      # كل خاصية + ارتباط + p-value + الحكم
            'tree_rules_text': str,
            'baseline_rate': float, 'n_train': int, 'n_val': int,
            'best_patterns': DataFrame,         # أنماط موثوقة جيدة فقط (مرتّبة)
            'worst_patterns': DataFrame,        # أنماط موثوقة سيئة فقط
            'unreliable_patterns': DataFrame,   # أنماط فشلت التحقق (تُستبعَد من القرار)
        }
    """
    from sklearn.tree import DecisionTreeClassifier, export_text
    from scipy.stats import binomtest, pointbiserialr

    X_all = input_features.reset_index(drop=True).copy()
    y_all = pd.Series(np.asarray(outcome, dtype=np.float64), name='outcome').reset_index(drop=True)

    mask = y_all.notna() & X_all.notna().all(axis=1)
    X_all, y_all = X_all[mask].reset_index(drop=True), y_all[mask].reset_index(drop=True)

    if timestamps is not None:
        ts = pd.Series(np.asarray(timestamps)[np.asarray(mask)]).reset_index(drop=True)
        order = np.argsort(ts.values, kind='stable')
        X_all, y_all = X_all.iloc[order].reset_index(drop=True), y_all.iloc[order].reset_index(drop=True)

    n = len(X_all)
    if n < 50:
        raise ValueError(f"عدد العينات الصالحة قليل جداً ({n} < 50) لتحقق إحصائي موثوق.")

    n_val = max(int(round(n * validation_frac)), min_leaf_samples * 2)
    n_train = n - n_val
    if n_train < min_leaf_samples * 4:
        raise ValueError("عدد عينات التدريب غير كافٍ بعد الفصل الزمني — قلّل validation_frac أو min_leaf_samples.")

    X_train, X_val = X_all.iloc[:n_train].reset_index(drop=True), X_all.iloc[n_train:].reset_index(drop=True)
    y_train, y_val = y_all.iloc[:n_train].reset_index(drop=True), y_all.iloc[n_train:].reset_index(drop=True)
    baseline_rate = float(y_val.mean())  # معدل النجاح الأساسي في فترة التحقق نفسها (مقارنة عادلة)

    # ── 1) شجرة قرار على train فقط ──────────────────────────────────────────
    tree = DecisionTreeClassifier(max_depth=max_tree_depth, min_samples_leaf=min_leaf_samples,
                                   class_weight='balanced', random_state=42)
    tree.fit(X_train, y_train)
    tree_rules_text = export_text(tree, feature_names=list(X_all.columns))

    leaf_train = tree.apply(X_train)
    leaf_val = tree.apply(X_val)

    rule_rows = []
    for leaf_id in np.unique(leaf_train):
        tr_mask = leaf_train == leaf_id
        n_tr = int(tr_mask.sum())
        if n_tr < min_leaf_samples:
            continue
        rate_tr = float(y_train[tr_mask].mean())

        va_mask = leaf_val == leaf_id
        n_va = int(va_mask.sum())
        if n_va == 0:
            rule_rows.append({
                'leaf_id': int(leaf_id), 'n_train': n_tr, 'نسبة_نجاح_train': round(rate_tr * 100, 2),
                'n_val': 0, 'نسبة_نجاح_val': float('nan'), 'فجوة_train_val': float('nan'),
                'p_value': float('nan'), 'الحكم': "🎲 غير موثوق (لا عينات مطابقة في validation)",
            })
            continue

        rate_va = float(y_val[va_mask].mean())
        k_success = int(y_val[va_mask].sum())
        alt = 'greater' if rate_tr >= baseline_rate else 'less'
        p_val = float(binomtest(k_success, n_va, baseline_rate, alternative=alt).pvalue)
        gap = abs(rate_tr - rate_va)

        if p_val < alpha and gap <= consistency_gap_max and rate_va > baseline_rate:
            verdict = "✅ نمط موثوق جيد"
        elif p_val < alpha and gap <= consistency_gap_max and rate_va < baseline_rate:
            verdict = "❌ نمط موثوق سيء"
        else:
            verdict = "🎲 غير موثوق / يشبه العشوائية"

        rule_rows.append({
            'leaf_id': int(leaf_id), 'n_train': n_tr, 'نسبة_نجاح_train': round(rate_tr * 100, 2),
            'n_val': n_va, 'نسبة_نجاح_val': round(rate_va * 100, 2),
            'فجوة_train_val': round(gap * 100, 2), 'p_value': round(p_val, 4), 'الحكم': verdict,
        })

    rules_df = pd.DataFrame(rule_rows).sort_values('نسبة_نجاح_val', ascending=False, na_position='last').reset_index(drop=True)

    # ── 2) اختبار تبديل لكل خاصية مفردة (مستقل عن الشجرة) ───────────────────
    numeric_cols = X_all.select_dtypes(include=[np.number]).columns
    rng = np.random.default_rng(42)
    feat_rows = []
    for col in numeric_cols:
        v_train, v_val = X_train[col].values, X_val[col].values
        if np.std(v_train) < 1e-12:
            continue
        r_train, _ = pointbiserialr(y_train.values, v_train)

        null_r = np.empty(n_permutations)
        y_shuf = y_train.values.copy()
        for i in range(n_permutations):
            rng.shuffle(y_shuf)
            null_r[i], _ = pointbiserialr(y_shuf, v_train)
        p_perm = float(np.mean(np.abs(null_r) >= abs(r_train)))

        r_val = float(pointbiserialr(y_val.values, v_val)[0]) if np.std(v_val) > 1e-12 else float('nan')
        replicates = (not np.isnan(r_val)) and (np.sign(r_val) == np.sign(r_train)) and (abs(r_val) > 0.05)

        if p_perm < alpha and replicates:
            verdict = "✅ خاصية مؤثرة وموثوقة"
        elif p_perm < alpha and not replicates:
            verdict = "🎲 دالة على train فقط (لا تتكرر) — غالباً صدفة/overfitting"
        else:
            verdict = "◻️ غير مؤثرة إحصائياً"

        feat_rows.append({
            'الخاصية': col, 'ارتباط_train': round(float(r_train), 4),
            'ارتباط_val': round(r_val, 4) if not np.isnan(r_val) else float('nan'),
            'p_value_permutation': round(p_perm, 4), 'الحكم': verdict,
        })

    feature_stats_df = pd.DataFrame(feat_rows).sort_values('p_value_permutation').reset_index(drop=True)

    best_patterns = rules_df[rules_df['الحكم'] == "✅ نمط موثوق جيد"].reset_index(drop=True)
    worst_patterns = rules_df[rules_df['الحكم'] == "❌ نمط موثوق سيء"].reset_index(drop=True)
    unreliable_patterns = rules_df[rules_df['الحكم'].str.startswith("🎲")].reset_index(drop=True)

    results = {
        'rules_df': rules_df, 'feature_stats_df': feature_stats_df, 'tree_rules_text': tree_rules_text,
        'baseline_rate': baseline_rate, 'n_train': n_train, 'n_val': n_val,
        'best_patterns': best_patterns, 'worst_patterns': worst_patterns, 'unreliable_patterns': unreliable_patterns,
    }

    if verbose:
        print("=" * 100)
        print("🔬 تحليل المدخلات ↔ المخرجات مع التحقق الإحصائي (Validated Pattern Discovery)")
        print("=" * 100)
        print(f"عينات: train={n_train} | validation={n_val} | معدل النجاح الأساسي (val) = {baseline_rate*100:.1f}%")

        print(f"\n🌳 قواعد شجرة القرار (على train فقط):")
        print("-" * 60)
        save_or_print(tree_rules_text, 'input_output_tree_rules', out_dir=DEFAULT_OUTPUT_DIR)

        print(f"\n📋 كل الأنماط (أوراق الشجرة) مع نتيجة التحقق على validation:")
        print("-" * 60)
        save_or_print(rules_df, 'input_output_rules_validated', out_dir=DEFAULT_OUTPUT_DIR)

        print(f"\n✅ أفضل الأنماط الموثوقة ({len(best_patterns)}):")
        for _, r in best_patterns.head(5).iterrows():
            print(f"   • ورقة #{r['leaf_id']}: نجاح_val={r['نسبة_نجاح_val']}% (train={r['نسبة_نجاح_train']}%) "
                  f"| n_val={r['n_val']} | p={r['p_value']}")
        if best_patterns.empty:
            print("   (لا يوجد نمط اجتاز التحقق الإحصائي كـ'جيد موثوق')")

        print(f"\n❌ أسوأ الأنماط الموثوقة ({len(worst_patterns)}):")
        for _, r in worst_patterns.head(5).iterrows():
            print(f"   • ورقة #{r['leaf_id']}: نجاح_val={r['نسبة_نجاح_val']}% (train={r['نسبة_نجاح_train']}%) "
                  f"| n_val={r['n_val']} | p={r['p_value']}")
        if worst_patterns.empty:
            print("   (لا يوجد نمط اجتاز التحقق الإحصائي كـ'سيء موثوق')")

        print(f"\n🎲 {len(unreliable_patterns)} نمط غير موثوق — لا يُعتمَد عليه (يشبه الصدفة/الحفظ الزائد)")

        print(f"\n🧪 إحصاءات الخصائص الفردية (اختبار تبديل، {n_permutations} تكرار):")
        print("-" * 60)
        save_or_print(feature_stats_df, 'input_feature_permutation_stats', out_dir=DEFAULT_OUTPUT_DIR)
        reliable_feats = feature_stats_df[feature_stats_df['الحكم'] == "✅ خاصية مؤثرة وموثوقة"]
        if len(reliable_feats):
            top = reliable_feats.iloc[0]
            print(f"   → أهم خاصية مفردة موثوقة: '{top['الخاصية']}' "
                  f"(ارتباط train={top['ارتباط_train']}, val={top['ارتباط_val']})")
        else:
            print("   → لا توجد خاصية مفردة ذات دلالة إحصائية تتكرر في train وvalidation معاً.")

        try:
            plot_pattern_validation(rules_df, save_dir=DEFAULT_OUTPUT_DIR)
        except Exception as e:
            print(f"   ⚠️ تعذر رسم مخطط الأنماط: {e}")

    return results


def plot_pattern_validation(rules_df: pd.DataFrame, save_dir: str = DEFAULT_OUTPUT_DIR) -> Optional[str]:
    """رسم شريطي: نسبة نجاح كل نمط (ورقة) على train مقابل validation، بلون يعكس الحكم النهائي."""
    df = rules_df.dropna(subset=['نسبة_نجاح_val']).sort_values('نسبة_نجاح_val', ascending=False)
    if df.empty:
        return None

    color_map = {"✅ نمط موثوق جيد": "#2ca02c", "❌ نمط موثوق سيء": "#d62728",
                 "🎲 غير موثوق / يشبه العشوائية": "#7f7f7f"}
    colors = df['الحكم'].map(color_map).fillna("#7f7f7f")

    fig, ax = plt.subplots(figsize=(max(6, len(df) * 0.6), 5))
    x = np.arange(len(df))
    ax.bar(x - 0.18, df['نسبة_نجاح_train'], width=0.36, color='#9ecae1', label='Train success %')
    ax.bar(x + 0.18, df['نسبة_نجاح_val'], width=0.36, color=colors, label='Validation success %')
    ax.set_xticks(x)
    ax.set_xticklabels([f"#{int(l)}" for l in df['leaf_id']], rotation=0)
    ax.set_ylabel('Success rate (%)')
    ax.set_xlabel('Leaf / pattern id')
    ax.set_title('Input→Output Patterns: Train vs Validation success rate')
    ax.legend()
    fig.tight_layout()

    path = None
    if save_dir:
        fig_dir = os.path.join(save_dir, 'figures')
        os.makedirs(fig_dir, exist_ok=True)
        path = os.path.join(fig_dir, 'pattern_validation.png')
        fig.savefig(path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return path


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🚀 غلاف سريع: مدخلات أصل واحد → تقييم → تحليل أنماط مُتحقَّق منها
# ═══════════════════════════════════════════════════════════════════════════

def run_input_output_pattern_discovery(
    model,
    test_dict: Dict,
    asset: str,
    timeframes: List[str],
    target_specs,
    feature_names: Optional[List[Optional[List[str]]]] = None,
    agg: Tuple[str, ...] = ('last', 'mean', 'std'),
    validation_frac: float = 0.3,
    batch_size: int = 256,
    timestamp_key: Optional[str] = None,
    timestamp_col: Optional[int] = None,
    verbose: bool = True,
    **analyze_kwargs,
) -> Dict[str, Dict]:
    """
    غلاف سريع يربط مدخلات نموذج أصل واحد الخام (X_inputs) بنتائج تقييمه
    (عبر predict_with_evaluation_v4) ثم يمرّرها إلى
    analyze_input_output_validated_patterns لكل هدف على حدة — أي دالة واحدة
    تُجيب على: "ما الأنماط في المدخلات الخام التي تُميَّز فعلاً بين توقعات
    صحيحة وتوقعات خاطئة/عشوائية، بعد التحقق الإحصائي؟"

    ملاحظة: تعمل على أصل واحد لأن ترتيب X_inputs زمني ومستقل لكل أصل (لا يجوز
    خلط أصول مختلفة في نفس التقسيم الزمني train/validation). لتحليل عدة أصول
    استدعِ الدالة لكل أصل على حدة، ثم قارن جداول rules_df/feature_stats_df يدوياً
    (أو اجمعها إن أردتَ حكماً عاماً عبر الأصول).

    Args:
        test_dict / timeframes / target_specs: نفس اصطلاحات test_all_assets_v4.
        feature_names: أسماء خصائص كل إطار زمني (تُمرَّر كما هي إلى
                       extract_input_snapshot_features) — مهم لقراءة الأنماط
                       (مثل 'rsi', 'close', 'volume'...) بدل f0..fK.
        باقي المعاملات: انظر extract_input_snapshot_features و
                        analyze_input_output_validated_patterns.

    Returns:
        dict: {target_name: نتيجة analyze_input_output_validated_patterns} —
              هدف واحد فقط إن فشل التحليل عليه يُستبعَد مع طباعة تحذير.
    """
    if asset not in test_dict:
        raise KeyError(f"الأصل '{asset}' غير موجود في test_dict.")

    test_data = test_dict[asset]
    X_inputs = tuple(test_data[f'X_{tf}'] for tf in timeframes)
    base_params = test_data['base_params']
    last_candles = test_data.get('last_candles')

    specs = resolve_targets(target_specs)
    y_true = {f'y_{s.name}': test_data['y'][s.name] for s in specs if s.name in test_data.get('y', {})}

    ts = _align_timestamps(_find_timestamps(test_data, timestamp_key, timestamp_col),
                            len(base_params), len(base_params))

    evaluated = predict_with_evaluation_v4(
        model, X_inputs, specs, base_params=base_params, last_candles=last_candles,
        y_true=y_true, batch_size=batch_size, verbose=False, timestamps=ts, asset=asset,
    )

    input_features = extract_input_snapshot_features(X_inputs, timeframes, feature_names=feature_names, agg=agg)

    out: Dict[str, Dict] = {}
    for spec in specs:
        data = evaluated.get(spec.name)
        if not isinstance(data, dict):
            continue

        outcome_col = 'direction_correct' if 'direction_correct' in data else ('correct' if 'correct' in data else None)
        if outcome_col is None:
            if verbose:
                print(f"⚠️ لا يوجد عمود نتيجة (direction_correct/correct) للهدف '{spec.name}' — تخطّي (يحتاج y_true).")
            continue

        outcome = np.asarray(data[outcome_col], dtype=np.float64)
        n = min(len(input_features), len(outcome))
        has_truth = np.asarray(data.get('has_truth', np.ones(n, dtype=bool)))[:n]

        feats = input_features.iloc[:n][has_truth].reset_index(drop=True)
        oc = outcome[:n][has_truth]
        ts_valid = np.asarray(ts)[:n][has_truth] if ts is not None else None

        if verbose:
            print(f"\n{'#' * 100}\n🎯 الهدف: {spec.name}  |  الأصل: {asset}\n{'#' * 100}")
        try:
            out[spec.name] = analyze_input_output_validated_patterns(
                feats, oc, timestamps=ts_valid, validation_frac=validation_frac, verbose=verbose,
                **analyze_kwargs,
            )
        except ValueError as e:
            print(f"   ⚠️ تعذّر التحليل للهدف '{spec.name}': {e}")

    return out


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 🚀 أمثلة استخدام
# ══════════════════════════════════════════════════════════════════════════
#
# # 1) الطريقة السريعة: أصل واحد من test_dict مباشرة (تنبؤ + تقييم + تحليل أنماط):
# patterns = run_input_output_pattern_discovery(
#     model, test_dict, asset='ENSUSDT', timeframes=['1h', '4h', '1D'],
#     target_specs='close',
#     feature_names=[['rsi','close','volume','ema','atr', ...]] * 3,  # اختياري لكن مفيد جداً للقراءة
#     validation_frac=0.3,
# )
# best = patterns['close']['best_patterns']     # أفضل الأنماط الموثوقة
# worst = patterns['close']['worst_patterns']   # أسوأ الأنماط الموثوقة
# feats = patterns['close']['feature_stats_df'] # كل خاصية مفردة + دلالتها
#
# # 2) الطريقة اليدوية (تحكّم كامل بمصدر outcome/input_features):
# X_inputs = tuple(test_dict['ENSUSDT'][f'X_{tf}'] for tf in ['1h', '4h', '1D'])
# input_features = extract_input_snapshot_features(X_inputs, ['1h', '4h', '1D'])
# # outcome: أي عمود 0/1 من نتيجة evaluate_predictions_v4 (direction_correct / correct)
# result = analyze_input_output_validated_patterns(input_features, outcome, validation_frac=0.3)